In [ ]:
from pathlib import Path
import os
import sys
import hashlib
import subprocess
import torch

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
assert ROOT.is_dir(), f"Repositório não encontrado: {ROOT}"
os.chdir(ROOT)

print("=== AMBIENTE ===")
print("Diretório:", Path.cwd())
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CUDA indisponível")

def consultar(comando):
    try:
        r = subprocess.run(
            comando, capture_output=True, text=True, timeout=15
        )
        print((r.stdout or r.stderr).strip())
    except (OSError, subprocess.TimeoutExpired) as erro:
        print(f"Não foi possível consultar: {erro}")

consultar(["nvidia-smi",
           "--query-gpu=name,driver_version,memory.total",
           "--format=csv,noheader"])
consultar(["git", "branch", "--show-current"])
consultar(["git", "rev-parse", "HEAD"])
consultar(["git", "status", "--short"])

print("\n=== ARQUIVOS DE REFERÊNCIA ===")
referencias = [
    "exp/17_eft_tiny_512.ipynb",
    "models/vmamba.py",
    "pretrained_model/vmamba_tiny_v2.pth",
    "pretrained_model/sam_vit_h.pth",
    "reproducao/protocolo_polipos_20261011T143313814944Z/manifesto.json",
    "reproducao/eft_sp_piloto_dados_20261011T152315570066Z/config.json",
    "reproducao/eft_sp_piloto_dados_20261011T152315570066Z/divisao.json",
    "reproducao/eft_sp_piloto_dados_20261011T152315570066Z/planos.json",
]
for nome in referencias:
    p = ROOT / nome
    print(f"{'EXISTE' if p.is_file() else 'AUSENTE'} | {nome}")

vmamba = ROOT / "models/vmamba.py"
if vmamba.is_file():
    sha = hashlib.sha256(vmamba.read_bytes()).hexdigest()
    esperado = "572207a2b6553ff85d81b602eebda2041623073f01fc0c6d1280c5a2add040f6"
    print("SHA256 VMamba:", sha)
    print("Confere com o handoff:", sha == esperado)

print("\n=== SCRIPTS PARA INSPEÇÃO ===")
exp = ROOT / "exp"
scripts = sorted(
    p for p in exp.rglob("*.py")
    if any("eft" in parte.lower() for parte in p.relative_to(exp).parts)
)
for p in scripts:
    print(f"{p.relative_to(ROOT)} | {p.stat().st_size:,} bytes")

print("\n=== PLANOS NAS EXECUÇÕES 2000/5000 ===")
repro = ROOT / "reproducao"
for pasta in sorted(repro.glob("eft_tiny_*")):
    if pasta.is_dir() and any(x in pasta.name for x in ("2000", "5000")):
        for p in sorted(pasta.glob("*.json")):
            print(f"{p.relative_to(ROOT)} | {p.stat().st_size:,} bytes")

In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

scripts = [
    "exp/eft_tiny_2000_v2/eft_dados.py",
    "exp/eft_tiny_2000_v2/eft_modelo.py",
    "exp/eft_tiny_2000_v2/eft_treino.py",
    "exp/eft_tiny_2000_v2/eft_continuar.py",
    "exp/eft_tiny_2000_v2/eft_alinhamento_seguro.py",
    "exp/eft_tiny_2000_v2/eft_plano_2000.py",
]

for nome in scripts:
    p = ROOT / nome
    print(f"\n{'=' * 70}\nARQUIVO: {nome}\n{'=' * 70}")
    if not p.is_file():
        print("AUSENTE")
        continue
    print(p.read_text(encoding="utf-8"))

# Mostra a estrutura e uma amostra das listas, sem imprimir
# milhares de episódios ou registros.
def resumir(obj, nivel=0):
    if nivel >= 5:
        return repr(obj)[:500]
    if isinstance(obj, dict):
        return {k: resumir(v, nivel + 1) for k, v in obj.items()}
    if isinstance(obj, list):
        return {
            "quantidade": len(obj),
            "primeiro": resumir(obj[0], nivel + 1) if obj else None,
        }
    return obj

jsons = [
    "reproducao/protocolo_polipos_20261011T143313814944Z/manifesto.json",
    "reproducao/eft_sp_piloto_dados_20261011T152315570066Z/config.json",
    "reproducao/eft_sp_piloto_dados_20261011T152315570066Z/divisao.json",
    "reproducao/eft_tiny_2000_v2_20261011T204725195685Z/config.json",
    "reproducao/eft_tiny_2000_v2_20261011T204725195685Z/planos.json",
]

for nome in jsons:
    p = ROOT / nome
    print(f"\n{'=' * 70}\nJSON: {nome}\n{'=' * 70}")
    if not p.is_file():
        print("AUSENTE")
        continue
    dados = json.loads(p.read_text(encoding="utf-8"))
    print(json.dumps(resumir(dados), indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from PIL import Image

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
BASE = ROOT.parent
DADOS = ROOT / "reproducao/eft_sp_piloto_dados_20261011T152315570066Z"
HISTORICO = ROOT / "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha256(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def caminho_atual(nome):
    p = Path(nome)
    if p.is_relative_to("/workspace"):
        return BASE / p.relative_to("/workspace")
    return p

divisao = ler_json(DADOS / "divisao.json")
planos = ler_json(HISTORICO / "planos.json")
config = ler_json(HISTORICO / "config.json")

print("=== DIVISÃO E PLANO ===")
print("Imagens de treino:", sum(r["split"] == "treino" for r in divisao))
print("Imagens de validação:", sum(r["split"] == "validacao" for r in divisao))
print("Episódios de treino:", len(planos["treino"]))
print("Episódios de validação:", len(planos["validacao"]))
print("Weight decay registrado:", config["weight_decay"])

ausentes = []
for r in divisao:
    imagem = caminho_atual(r["imagem"])
    cache = DADOS / "cache" / f'{r["indice"]:04d}.npz'
    for p in (imagem, cache):
        if not p.is_file():
            ausentes.append(str(p))

print("\n=== CAMINHOS ===")
print("Exemplo antigo:", divisao[0]["imagem"])
print("Exemplo atual:", caminho_atual(divisao[0]["imagem"]))
print("Arquivos ausentes:", len(ausentes))
for p in ausentes[:10]:
    print(" AUSENTE:", p)

assert not ausentes, (
    "Os dados não estão todos nos caminhos previstos. "
    "Envie esta saída antes de continuar."
)

print("\n=== PESOS PRÉ-TREINADOS ===")
pesos = ROOT / "pretrained_model/vmamba_tiny_v2.pth"
digest = sha256(pesos)
print("SHA256:", digest)
assert digest == config["checkpoint_origem_sha256"], (
    "Os pesos iniciais diferem dos usados no histórico."
)
print("Pesos idênticos aos registrados.")

print("\n=== AMOSTRAS: ORIGINAL E CACHE ===")
for split in ("treino", "validacao"):
    r = planos[split][0]
    imagem = caminho_atual(r["imagem"])
    cache = DADOS / r["cache"]

    assert sha256(imagem) == r["imagem_sha256"], (
        f"Imagem original diferente: {imagem}"
    )
    assert sha256(cache) == r["cache_sha256"], (
        f"Cache diferente: {cache}"
    )

    with Image.open(imagem) as im:
        tamanho_original = im.size
        rgb512 = np.array(
            im.convert("RGB").resize((512, 512), Image.Resampling.BILINEAR)
        )

    with np.load(cache, allow_pickle=False) as d:
        rgb256 = d["rgb"].copy()
        mascara256 = (d["segmentos"] == r["regiao"]) & d["util"]

    assert rgb256.shape == (256, 256, 3)
    assert mascara256.shape == (256, 256) and mascara256.any()

    # Replicar cada pixel em um bloco 2×2 equivale a nearest-neighbor
    # nesta ampliação exata de 256 para 512.
    mascara512 = mascara256.repeat(2, axis=0).repeat(2, axis=1)
    assert mascara512.shape == (512, 512)
    assert np.array_equal(mascara512[::2, ::2], mascara256)

    print(f"\n{split}: {r['nome']} | região {r['regiao']}")
    print("Original (largura, altura):", tamanho_original)
    print("RGB reaberto em 512:", rgb512.shape)
    print("Máscara ampliada:", mascara512.shape)
    print("Pixels da região:", int(mascara256.sum()), "→",
          int(mascara512.sum()))
    print("Hashes da imagem e do cache: conferem")

print("\nConferência concluída. Nenhum arquivo alterado ou treino iniciado.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import hashlib
import json
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
BASE = ROOT.parent
DADOS256 = ROOT / "reproducao/eft_sp_piloto_dados_20261011T152315570066Z"
HISTORICO = ROOT / "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha256(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def gravar_json(p, obj):
    p = Path(p)
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    os.replace(tmp, p)

def caminho_atual(nome):
    p = Path(nome)
    return BASE / p.relative_to("/workspace") if p.is_relative_to("/workspace") else p

planos256 = ler_json(HISTORICO / "planos.json")
divisao256 = ler_json(DADOS256 / "divisao.json")

# Os 2.000 episódios devem abranger todas as 1.304 imagens de treino.
indices_treino = {r["indice"] for r in planos256["treino"]}
indices_val = {r["indice"] for r in planos256["validacao"]}
assert len(planos256["treino"]) == 2000
assert len(indices_treino) == 1304 and len(indices_val) == 145
assert not indices_treino & indices_val

registros = {}
for r in planos256["treino"] + planos256["validacao"]:
    registros.setdefault(r["indice"], r)

assert set(registros) == {r["indice"] for r in divisao256}
for r in divisao256:
    assert registros[r["indice"]]["split"] == r["split"]
    assert registros[r["indice"]]["imagem"] == r["imagem"]

# Pasta exclusiva: os dados de 256 permanecem preservados.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DADOS512 = ROOT / "reproducao" / f"eft_sp_dados512_regioes256_{stamp}"
DADOS512.mkdir(parents=True, exist_ok=False)
(DADOS512 / "cache").mkdir()

config512 = {
    "resolucao_rgb": [512, 512],
    "resolucao_origem_regioes": [256, 256],
    "rgb": "imagem original convertida para RGB; resize PIL BILINEAR",
    "rotulos": "replicacao 2x2, equivalente a nearest-neighbor",
    "superpixels_recalculados": False,
    "regiao_util_recalculada": False,
    "episodios": "mesmas imagens, regioes e ordem do plano historico",
    "mascaras_manuais_utilizadas": False,
    "dados256": str(DADOS256),
    "plano_origem": str(HISTORICO / "planos.json"),
    "plano_origem_sha256": sha256(HISTORICO / "planos.json"),
    "divisao_origem_sha256": sha256(DADOS256 / "divisao.json"),
    "mapeamento_caminhos": {"/workspace": str(BASE)},
}
gravar_json(DADOS512 / "config.json", config512)

print("Nova pasta:", DADOS512, flush=True)
print("Verificando originais e preparando 1.449 caches...", flush=True)

novos = {}
for numero, indice in enumerate(sorted(registros), start=1):
    r = registros[indice]
    original = caminho_atual(r["imagem"])
    cache256 = DADOS256 / r["cache"]

    assert sha256(original) == r["imagem_sha256"], f"Imagem alterada: {original}"
    assert sha256(cache256) == r["cache_sha256"], f"Cache alterado: {cache256}"

    with Image.open(original) as im:
        tamanho_original = list(im.size)
        rgb512 = np.array(
            im.convert("RGB").resize((512, 512), Image.Resampling.BILINEAR)
        )

    with np.load(cache256, allow_pickle=False) as d:
        segmentos256 = d["segmentos"].copy()
        util256 = d["util"].copy()
        candidatos = d["candidatos"].copy()

    assert segmentos256.shape == util256.shape == (256, 256)
    assert candidatos.tolist() == r["candidatos"]

    segmentos512 = segmentos256.repeat(2, axis=0).repeat(2, axis=1)
    util512 = util256.repeat(2, axis=0).repeat(2, axis=1)

    cache512 = DADOS512 / r["cache"]
    with cache512.open("xb") as f:
        np.savez_compressed(
            f, rgb=rgb512, segmentos=segmentos512,
            util=util512, candidatos=candidatos
        )

    novo = copy.deepcopy(r)
    novo.pop("regiao", None)  # A região será restaurada para cada episódio.
    novo.update({
        "imagem": str(original),
        "imagem_origem": r["imagem"],
        "cache256_sha256": r["cache_sha256"],
        "cache_sha256": sha256(cache512),
        "tamanho_original_wh": tamanho_original,
    })
    novos[indice] = novo
    gravar_json(cache512.with_suffix(".json"), novo)

    if numero % 100 == 0 or numero == len(registros):
        print(f"Preparados: {numero}/{len(registros)}", flush=True)

planos512 = {}
for split in ("treino", "validacao"):
    planos512[split] = [
        dict(novos[r["indice"]], regiao=r["regiao"])
        for r in planos256[split]
    ]

    # Confere a identidade e a ordem das tarefas.
    assert [
        (r["indice"], r["regiao"]) for r in planos512[split]
    ] == [
        (r["indice"], r["regiao"]) for r in planos256[split]
    ]

divisao512 = [
    dict(r, imagem=str(caminho_atual(r["imagem"])))
    for r in divisao256
]
gravar_json(DADOS512 / "divisao.json", divisao512)
gravar_json(DADOS512 / "planos.json", planos512)
gravar_json(DADOS512 / "preparacao_concluida.json", {
    "imagens": len(novos),
    "episodios_treino": len(planos512["treino"]),
    "episodios_validacao": len(planos512["validacao"]),
    "planos_sha256": sha256(DADOS512 / "planos.json"),
})

# Mostra o primeiro episódio de treino: imagem e pseudo-máscara.
r = planos512["treino"][0]
with np.load(DADOS512 / r["cache"], allow_pickle=False) as d:
    rgb = d["rgb"].copy()
    mascara = (d["segmentos"] == r["regiao"]) & d["util"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(rgb)
axes[0].set_title("Support: RGB original → 512")
axes[1].imshow(rgb)
axes[1].imshow(
    np.ma.masked_where(~mascara, mascara),
    cmap="autumn", alpha=0.55, vmin=0, vmax=1
)
axes[1].set_title(f"Pseudo-região {r['regiao']}")
axes[2].imshow(np.fliplr(rgb))
axes[2].imshow(
    np.ma.masked_where(~np.fliplr(mascara), np.fliplr(mascara)),
    cmap="autumn", alpha=0.55, vmin=0, vmax=1
)
axes[2].set_title("Query: imagem e máscara espelhadas")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(DADOS512 / "primeiro_episodio.png", dpi=150)
plt.show()
plt.close(fig)

print("\nPreparação concluída:", DADOS512)
print("Imagens, regiões e ordem preservadas.")
print("Nenhum treinamento iniciado.")

In [ ]:
from pathlib import Path
import json
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display, Image as DisplayImage

# Reutiliza exatamente a pasta criada pela célula anterior.
assert "DADOS512" in globals(), "A variável DADOS512 não está disponível."
pasta = Path(DADOS512)
assert (pasta / "preparacao_concluida.json").is_file(), (
    f"Preparação sem registro de conclusão: {pasta}"
)

planos = json.loads((pasta / "planos.json").read_text())
r = planos["treino"][0]

with np.load(pasta / r["cache"], allow_pickle=False) as d:
    rgb = d["rgb"].copy()
    mascara = (d["segmentos"] == r["regiao"]) & d["util"]

fig = Figure(figsize=(13, 4))
canvas = FigureCanvasAgg(fig)
axes = fig.subplots(1, 3)

axes[0].imshow(rgb)
axes[0].set_title("Support: RGB original → 512")

axes[1].imshow(rgb)
axes[1].imshow(
    np.ma.masked_where(~mascara, mascara),
    cmap="autumn", alpha=0.55, vmin=0, vmax=1
)
axes[1].set_title(f"Pseudo-região {r['regiao']}")

mascara_query = np.fliplr(mascara)
axes[2].imshow(np.fliplr(rgb))
axes[2].imshow(
    np.ma.masked_where(~mascara_query, mascara_query),
    cmap="autumn", alpha=0.55, vmin=0, vmax=1
)
axes[2].set_title("Query: imagem e máscara espelhadas")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
arquivo = pasta / "primeiro_episodio.png"
fig.savefig(arquivo, dpi=150)
display(DisplayImage(filename=str(arquivo)))

print("Dados preparados em:", pasta)
print("Episódios de treino:", len(planos["treino"]))
print("Episódios de validação:", len(planos["validacao"]))
print("Figura salva em:", arquivo)

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import importlib.util
import torch
import torch.nn.functional as F
import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS512 = ROOT / (
    "reproducao/eft_sp_dados512_regioes256_20261016T212513467126Z"
)
ORIGEM = ROOT / "exp/eft_tiny_2000_v2"
CODIGO512 = ROOT / "exp/eft_tiny_512_v1"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

assert (DADOS512 / "preparacao_concluida.json").is_file()

# Confere a versão que foi inspecionada antes de adaptá-la.
fonte = ORIGEM / "eft_modelo.py"
texto = fonte.read_text(encoding="utf-8")
assert hashlib.sha256(fonte.read_bytes()).hexdigest() == (
    "9b1760c94819447eedfc048cebc652431c084f1491d8eafb00aa40d112223dac"
), "O módulo do modelo mudou; interrompido para inspeção."

# Mudanças explícitas, sem modificar o módulo antigo.
substituicoes = {
    "size=(256, 256)": "size=(512, 512)",
    "(canais, 8, 8)": "(canais, 16, 16)",
    "image_size=256": "image_size=512",
}
for antigo, novo in substituicoes.items():
    assert texto.count(antigo) == 1, f"Trecho inesperado: {antigo}"
    texto = texto.replace(antigo, novo)

texto = texto.replace("/workspace/ProtoSAM-repro", str(ROOT))
compile(texto, "eft_modelo_512.py", "exec")

CODIGO512.mkdir(exist_ok=True)

def salvar_sem_sobrescrever(p, conteudo):
    if p.exists():
        assert p.read_text(encoding="utf-8") == conteudo, (
            f"Já existe uma versão diferente: {p}"
        )
    else:
        with p.open("x", encoding="utf-8") as f:
            f.write(conteudo)

salvar_sem_sobrescrever(CODIGO512 / "eft_modelo_512.py", texto)
salvar_sem_sobrescrever(
    CODIGO512 / "eft_alinhamento_seguro.py",
    (ORIGEM / "eft_alinhamento_seguro.py").read_text(encoding="utf-8")
)

def importar_arquivo(nome, caminho):
    spec = importlib.util.spec_from_file_location(nome, caminho)
    modulo = importlib.util.module_from_spec(spec)
    sys.modules[nome] = modulo
    spec.loader.exec_module(modulo)
    return modulo

mod512 = importar_arquivo(
    "experimento512_modelo", CODIGO512 / "eft_modelo_512.py"
)
mod_align512 = importar_arquivo(
    "experimento512_alinhamento", CODIGO512 / "eft_alinhamento_seguro.py"
)

# Mesmo conjunto de opções numéricas registrado no histórico.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(False)

planos512 = json.loads((DADOS512 / "planos.json").read_text())

def carregar_episodio512(registro, device="cuda"):
    with np.load(DADOS512 / registro["cache"], allow_pickle=False) as d:
        rgb = d["rgb"].copy()
        mascara = (d["segmentos"] == registro["regiao"]) & d["util"]

    assert rgb.shape == (512, 512, 3)
    assert mascara.shape == (512, 512) and mascara.any()

    def normalizar(a):
        t = torch.from_numpy(np.ascontiguousarray(a))
        t = t.permute(2, 0, 1)[None].to(device, dtype=torch.float32)
        media = t.new_tensor([123.675, 116.28, 103.53])[None, :, None, None]
        desvio = t.new_tensor([58.395, 57.12, 57.375])[None, :, None, None]
        return (t - media) / desvio

    s = normalizar(rgb)
    q = normalizar(np.fliplr(rgb))
    fg = torch.from_numpy(mascara.copy())[None].to(device, dtype=torch.float32)
    alvo = torch.from_numpy(np.fliplr(mascara).copy())[None].to(
        device, dtype=torch.long
    )
    return s, fg, q, alvo

# Se esta célula for repetida, libera apenas o modelo deste diagnóstico.
if "modelo512" in globals():
    del modelo512
if "otimizador512" in globals():
    del otimizador512
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

modelo512 = mod512.construir_modelo("tiny", repo=ROOT)
estado_align512 = mod_align512.instalar(modelo512)

otimizador512 = torch.optim.AdamW(
    modelo512.parameters(),
    lr=1e-5, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0
)

# Observa a saída nativa do encoder, antes da interpolação para o ALP.
shapes_nativos = []
def observar_encoder(modulo, entradas, saida):
    shapes_nativos.append(list(saida[-1].shape))

hook = modelo512.encoder.register_forward_hook(observar_encoder)

r = planos512["treino"][0]
s, fg, q, alvo = carregar_episodio512(r)

modelo512.train()
otimizador512.zero_grad(set_to_none=True)
estado_align512["ultimo"] = None

try:
    retorno = modelo512(
        [[s]], [[fg]], [[1 - fg]], [q],
        isval=False, val_wsize=2, show_viz=False
    )
finally:
    hook.remove()

logits, alinhamento = retorno[:2]
assert shapes_nativos and all(
    shape[1:] == [768, 16, 16] for shape in shapes_nativos
), shapes_nativos
assert logits.shape == (1, 2, 512, 512)
assert alvo.shape == (1, 512, 512)
assert torch.isfinite(logits).all().item()

ce = F.cross_entropy(logits, alvo)
loss = ce + alinhamento
assert torch.isfinite(loss).item(), "Loss não finita."
loss.backward()

com_grad = [
    (nome, p) for nome, p in modelo512.encoder.named_parameters()
    if p.grad is not None
]
assert com_grad, "Nenhum gradiente chegou ao encoder."
assert all(torch.isfinite(p.grad).all().item() for _, p in com_grad)

ativos = [
    (nome, p) for nome, p in com_grad
    if torch.count_nonzero(p.grad).item() > 0
]
assert ativos, "Todos os gradientes são zero."

# Confere uma alteração real após o passo do otimizador.
nome_verificado, parametro = max(
    ativos, key=lambda item: item[1].grad.detach().abs().max().item()
)
antes = parametro.detach().clone()
otimizador512.step()
delta = (parametro.detach() - antes).abs().max().item()
assert delta > 0, "O parâmetro verificado não mudou."
assert all(torch.isfinite(p).all().item() for p in modelo512.parameters())

relatorio512 = {
    "episodio": r["nome"],
    "regiao": r["regiao"],
    "entrada_support": list(s.shape),
    "features_nativas": shapes_nativos,
    "grade_alp_configurada": modelo512.config["feature_hw"],
    "logits": list(logits.shape),
    "alvo": list(alvo.shape),
    "ce": ce.item(),
    "alinhamento": float(alinhamento.detach())
        if torch.is_tensor(alinhamento) else float(alinhamento),
    "loss_total": loss.item(),
    "tensores_encoder_com_gradiente": len(com_grad),
    "tensores_encoder_com_gradiente_nao_zero": len(ativos),
    "excecao_alinhamento": estado_align512["ultimo"],
    "parametro_verificado": nome_verificado,
    "maior_alteracao_no_parametro": delta,
    "pico_memoria_alocada_GiB": torch.cuda.max_memory_allocated() / 1024**3,
}
print(json.dumps(relatorio512, indent=2, ensure_ascii=False))

# Mantém modelo e otimizador para o próximo teste de salvamento/retomada.
otimizador512.zero_grad(set_to_none=True)
del retorno, logits, alinhamento, ce, loss, antes
del com_grad, ativos, parametro, s, fg, q, alvo

print("\nUma atualização de diagnóstico concluída.")
print("Treinamento definitivo ainda não iniciado.")

In [ ]:
import sys
import subprocess
import importlib
import traceback

subprocess.run(
    [sys.executable, "-m", "pip", "install", "fvcore"],
    check=True
)
importlib.invalidate_caches()

for nome in ("models.vmamba", "models.grid_proto_fewshot"):
    print(f"\nVerificando {nome}...")
    try:
        importlib.import_module(nome)
        print("OK")
    except Exception:
        traceback.print_exc()
        break

In [ ]:
from datetime import datetime, timezone
import random
import gc
import os
import json
import numpy as np
import torch
import torch.nn.functional as F

assert "modelo512" in globals() and "otimizador512" in globals(), (
    "Execute primeiro o diagnóstico de uma atualização."
)

def capturar_rng512():
    n = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": [n[0], n[1].tolist(), n[2], n[3], n[4]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all(),
    }

def restaurar_rng512(r):
    random.setstate(r["python"])
    n = r["numpy"]
    np.random.set_state(
        (n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4])
    )
    torch.set_rng_state(r["torch"])
    torch.cuda.set_rng_state_all(r["cuda"])

def conferir_igual(a, b, caminho="estado"):
    if torch.is_tensor(a):
        assert torch.is_tensor(b)
        assert a.dtype == b.dtype and a.shape == b.shape, caminho
        assert torch.equal(a.cpu(), b.cpu()), caminho
    elif isinstance(a, dict):
        assert a.keys() == b.keys(), caminho
        for k in a:
            conferir_igual(a[k], b[k], f"{caminho}.{k}")
    elif isinstance(a, (list, tuple)):
        assert type(a) is type(b) and len(a) == len(b), caminho
        for i, (x, y) in enumerate(zip(a, b)):
            conferir_igual(x, y, f"{caminho}[{i}]")
    else:
        assert a == b, caminho

def executar_passo_diagnostico(modelo, otimizador, estado_align, registro):
    modelo.train()
    otimizador.zero_grad(set_to_none=True)
    estado_align["ultimo"] = None

    s, fg, q, alvo = carregar_episodio512(registro)
    retorno = modelo(
        [[s]], [[fg]], [[1 - fg]], [q],
        isval=False, val_wsize=2, show_viz=False
    )
    logits, alinhamento = retorno[:2]
    assert logits.shape == (1, 2, 512, 512)
    assert torch.isfinite(logits).all().item()

    ce = F.cross_entropy(logits, alvo)
    loss = ce + alinhamento
    assert torch.isfinite(loss).item()
    loss.backward()

    grads = [p.grad for p in modelo.encoder.parameters() if p.grad is not None]
    assert grads and all(torch.isfinite(g).all().item() for g in grads)
    assert any(torch.count_nonzero(g).item() > 0 for g in grads)

    otimizador.step()
    assert all(torch.isfinite(p).all().item() for p in modelo.parameters())

    resultado = {
        "ce": ce.item(),
        "alinhamento": float(alinhamento.detach())
            if torch.is_tensor(alinhamento) else float(alinhamento),
        "loss": loss.item(),
        "excecao_alinhamento": estado_align["ultimo"],
    }
    otimizador.zero_grad(set_to_none=True)
    return resultado

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIAG512 = ROOT / "reproducao" / f"eft512_diagnostico_retomada_{stamp}"
DIAG512.mkdir(parents=True, exist_ok=False)

(DIAG512 / "primeira_atualizacao.json").write_text(
    json.dumps(relatorio512, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# Grava o estado após a única atualização já realizada.
checkpoint_diag = DIAG512 / "checkpoint_passo1.pt"
temporario = checkpoint_diag.with_suffix(".tmp")
torch.save({
    "formato": 1,
    "uso": "diagnostico; nao usar como inicio do treinamento definitivo",
    "passo": 1,
    "modelo": modelo512.state_dict(),
    "otimizador": otimizador512.state_dict(),
    "rng": capturar_rng512(),
    "dados": str(DADOS512),
}, temporario)
os.replace(temporario, checkpoint_diag)

registro = planos512["treino"][1]
print("Checkpoint de diagnóstico:", checkpoint_diag, flush=True)
print("Executando passo 2 sem interrupção...", flush=True)

direto = executar_passo_diagnostico(
    modelo512, otimizador512, estado_align512, registro
)
pesos_diretos = {
    k: v.detach().cpu().clone()
    for k, v in modelo512.state_dict().items()
}

# Libera o modelo antes de construir outro.
del modelo512, otimizador512
gc.collect()
torch.cuda.empty_cache()

print("Construindo outro modelo e restaurando o passo 1...", flush=True)
modelo512 = mod512.construir_modelo("tiny", repo=ROOT)
estado_align512 = mod_align512.instalar(modelo512)
otimizador512 = torch.optim.AdamW(
    modelo512.parameters(),
    lr=1e-5, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0
)

salvo = torch.load(checkpoint_diag, map_location="cpu", weights_only=True)
assert salvo["formato"] == 1 and salvo["passo"] == 1
modelo512.load_state_dict(salvo["modelo"], strict=True)
otimizador512.load_state_dict(salvo["otimizador"])

conferir_igual(modelo512.state_dict(), salvo["modelo"], "pesos")
conferir_igual(otimizador512.state_dict(), salvo["otimizador"], "otimizador")
restaurar_rng512(salvo["rng"])
conferir_igual(capturar_rng512(), salvo["rng"], "aleatoriedade")
del salvo

print("Pesos, otimizador e aleatoriedade restaurados exatamente.", flush=True)
print("Executando o mesmo passo 2 após retomada...", flush=True)

retomado = executar_passo_diagnostico(
    modelo512, otimizador512, estado_align512, registro
)

diferencas = {}
for nome, tensor in modelo512.state_dict().items():
    atual = tensor.detach().cpu()
    referencia = pesos_diretos[nome]
    if not torch.equal(atual, referencia):
        diferencas[nome] = (
            atual.to(torch.float64) - referencia.to(torch.float64)
        ).abs().max().item()

resultado_retomada512 = {
    "episodio_passo2": registro["nome"],
    "restauracao_exata_pesos_otimizador_rng": True,
    "passo2_direto": direto,
    "passo2_retomado": retomado,
    "diferenca_absoluta_loss": abs(direto["loss"] - retomado["loss"]),
    "tensores_diferentes_apos_passo2": len(diferencas),
    "maior_diferenca_absoluta_pesos": max(diferencas.values(), default=0.0),
    "trajetoria_exatamente_igual": not diferencas and direto == retomado,
}

(DIAG512 / "resultado_retomada.json").write_text(
    json.dumps(resultado_retomada512, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(json.dumps(resultado_retomada512, indent=2, ensure_ascii=False))
print("\nDiagnóstico salvo em:", DIAG512)

# Descarta os modelos de diagnóstico para não iniciar o treino com eles.
del pesos_diretos, modelo512, otimizador512
gc.collect()
torch.cuda.empty_cache()
print("Modelos de diagnóstico descartados. Treino definitivo não iniciado.")

In [ ]:
from pathlib import Path
import gc
import json
import torch
import torch.nn.functional as F

CHECKPOINT_DIAG = ROOT / (
    "reproducao/eft512_diagnostico_retomada_20261016T213252385061Z/"
    "checkpoint_passo1.pt"
)

salvo = torch.load(CHECKPOINT_DIAG, map_location="cpu", weights_only=True)
assert salvo["passo"] == 1

modelo512 = mod512.construir_modelo("tiny", repo=ROOT)
estado_align512 = mod_align512.instalar(modelo512)
otimizador512 = torch.optim.AdamW(
    modelo512.parameters(),
    lr=1e-5, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0
)

registro = planos512["treino"][1]

def repetir_mesmo_passo():
    modelo512.load_state_dict(salvo["modelo"], strict=True)
    otimizador512.load_state_dict(salvo["otimizador"])
    modelo512.train()
    otimizador512.zero_grad(set_to_none=True)
    estado_align512["ultimo"] = None
    restaurar_rng512(salvo["rng"])

    conferir_igual(modelo512.state_dict(), salvo["modelo"], "pesos_iniciais")
    conferir_igual(
        otimizador512.state_dict(), salvo["otimizador"], "otimizador_inicial"
    )
    conferir_igual(capturar_rng512(), salvo["rng"], "rng_inicial")

    s, fg, q, alvo = carregar_episodio512(registro)
    retorno = modelo512(
        [[s]], [[fg]], [[1 - fg]], [q],
        isval=False, val_wsize=2, show_viz=False
    )
    logits, alinhamento = retorno[:2]
    loss = F.cross_entropy(logits, alvo) + alinhamento
    assert torch.isfinite(logits).all().item()
    assert torch.isfinite(loss).item()

    logits_cpu = logits.detach().cpu().clone()
    loss.backward()

    gradientes = {
        nome: p.grad.detach().cpu().clone()
        for nome, p in modelo512.named_parameters()
        if p.grad is not None
    }
    assert gradientes
    assert all(torch.isfinite(g).all().item() for g in gradientes.values())

    otimizador512.step()
    pesos = {
        nome: p.detach().cpu().clone()
        for nome, p in modelo512.state_dict().items()
    }
    assert all(torch.isfinite(p).all().item() for p in pesos.values())

    resultado = {
        "loss": loss.item(),
        "logits": logits_cpu,
        "gradientes": gradientes,
        "pesos": pesos,
    }
    otimizador512.zero_grad(set_to_none=True)
    return resultado

print("Tentativa A: mesmo checkpoint, mesmo episódio...", flush=True)
a = repetir_mesmo_passo()
print("Tentativa B: restaurando tudo na mesma instância...", flush=True)
b = repetir_mesmo_passo()

def comparar_tensores(dic_a, dic_b):
    assert dic_a.keys() == dic_b.keys()
    diferentes = []
    soma_diff2 = 0.0
    soma_ref2 = 0.0

    for nome in dic_a:
        x = dic_a[nome].to(torch.float64)
        y = dic_b[nome].to(torch.float64)
        diff = x - y
        soma_diff2 += diff.square().sum().item()
        soma_ref2 += x.square().sum().item()

        if not torch.equal(dic_a[nome], dic_b[nome]):
            diferentes.append({
                "nome": nome,
                "max_diff_absoluta": diff.abs().max().item(),
                "max_absoluto_referencia": x.abs().max().item(),
            })

    diferentes.sort(key=lambda r: r["max_diff_absoluta"], reverse=True)
    return {
        "tensores_diferentes": len(diferentes),
        "diferenca_L2_relativa": (
            (soma_diff2 / soma_ref2) ** 0.5 if soma_ref2 > 0 else None
        ),
        "cinco_maiores_diferencas": diferentes[:5],
    }

relatorio = {
    "experimento": "duas atualizacoes a partir do mesmo estado, mesma instancia",
    "loss_A": a["loss"],
    "loss_B": b["loss"],
    "logits_identicos": torch.equal(a["logits"], b["logits"]),
    "max_diff_logits": (a["logits"] - b["logits"]).abs().max().item(),
    "gradientes": comparar_tensores(a["gradientes"], b["gradientes"]),
    "pesos_apos_atualizacao": comparar_tensores(a["pesos"], b["pesos"]),
    "opcoes_numericas": {
        "deterministico": torch.are_deterministic_algorithms_enabled(),
        "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
        "tf32_cudnn": torch.backends.cudnn.allow_tf32,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    },
}

# Nome exclusivo para preservar diagnósticos anteriores.
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
arquivo = CHECKPOINT_DIAG.parent / f"repeticao_mesma_instancia_{stamp}.json"
arquivo.write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(relatorio, indent=2, ensure_ascii=False))
print("\nRelatório:", arquivo)

del a, b, salvo, modelo512, otimizador512
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import gc
import json
import torch
from datetime import datetime, timezone

assert callable(globals().get("repetir_mesmo_passo"))
assert callable(globals().get("comparar_tensores"))
assert CHECKPOINT_DIAG.is_file()

modelo512 = mod512.construir_modelo("tiny", repo=ROOT)
estado_align512 = mod_align512.instalar(modelo512)
otimizador512 = torch.optim.AdamW(
    modelo512.parameters(),
    lr=1e-5, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0
)
registro = planos512["treino"][1]

# A função repetir_mesmo_passo usa a variável global "salvo".
# Reabrir o arquivo garante um estado independente em cada tentativa.
print("Tentativa A: checkpoint recém-lido do disco...", flush=True)
salvo = torch.load(CHECKPOINT_DIAG, map_location="cpu", weights_only=True)
a = repetir_mesmo_passo()
del salvo

print("Tentativa B: checkpoint lido novamente do disco...", flush=True)
salvo = torch.load(CHECKPOINT_DIAG, map_location="cpu", weights_only=True)
b = repetir_mesmo_passo()
del salvo

relatorio_corrigido = {
    "experimento": "mesma instancia; checkpoint reaberto antes de cada tentativa",
    "loss_A": a["loss"],
    "loss_B": b["loss"],
    "logits_identicos": torch.equal(a["logits"], b["logits"]),
    "max_diff_logits": (a["logits"] - b["logits"]).abs().max().item(),
    "gradientes": comparar_tensores(a["gradientes"], b["gradientes"]),
    "pesos_apos_atualizacao": comparar_tensores(a["pesos"], b["pesos"]),
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
arquivo = CHECKPOINT_DIAG.parent / f"repeticao_estado_independente_{stamp}.json"
arquivo.write_text(
    json.dumps(relatorio_corrigido, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(relatorio_corrigido, indent=2, ensure_ascii=False))
print("\nRelatório:", arquivo)

del a, b, modelo512, otimizador512
gc.collect()
torch.cuda.empty_cache()

In [ ]:
%%writefile /home/1542521/cassio_pereira/ProtoSAM-repro/exp/eft_tiny_512_v1/treinar_5000.py
from pathlib import Path
import argparse
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
import traceback

import numpy as np
import torch
import torch.nn.functional as F

CODIGO = Path(__file__).resolve().parent
ROOT = CODIGO.parent.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from eft_modelo_512 import construir_modelo
from eft_alinhamento_seguro import instalar

DADOS = ROOT / "reproducao/eft_sp_dados512_regioes256_20261016T212513467126Z"
H2000 = ROOT / "reproducao/eft_tiny_2000_v2_20261011T204725195685Z"
H5000 = ROOT / "reproducao/eft_tiny_5000_20261011T224215171455Z"
PRESERVAR = {0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000}

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def json_atomico(p, obj):
    p = Path(p)
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, p)

def rng_atual():
    n = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": [n[0], n[1].tolist(), n[2], n[3], n[4]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all(),
    }

def restaurar_rng(r):
    random.setstate(r["python"])
    n = r["numpy"]
    np.random.set_state((n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(r["torch"])
    torch.cuda.set_rng_state_all(r["cuda"])

def episodio(registro):
    with np.load(DADOS / registro["cache"], allow_pickle=False) as d:
        rgb = d["rgb"].copy()
        mask = (d["segmentos"] == registro["regiao"]) & d["util"]
    assert rgb.shape == (512, 512, 3)
    assert mask.shape == (512, 512) and mask.any()

    def imagem(a):
        t = torch.from_numpy(np.ascontiguousarray(a))
        t = t.permute(2, 0, 1)[None].to("cuda", dtype=torch.float32)
        media = t.new_tensor([123.675, 116.28, 103.53])[None, :, None, None]
        desvio = t.new_tensor([58.395, 57.12, 57.375])[None, :, None, None]
        return (t - media) / desvio

    return (
        imagem(rgb),
        torch.from_numpy(mask.copy())[None].to("cuda", dtype=torch.float32),
        imagem(np.fliplr(rgb)),
        torch.from_numpy(np.fliplr(mask).copy())[None].to("cuda", dtype=torch.long),
    )

def preparar_plano():
    assert (DADOS / "preparacao_concluida.json").is_file()
    novos = ler(DADOS / "planos.json")
    antigo2000 = ler(H2000 / "planos.json")
    antigo5000 = ler(H5000 / "planos.json")

    assert len(antigo5000["treino"]) == 5000
    assert antigo5000["treino"][:2000] == antigo2000["treino"], (
        "O plano histórico de 5000 não preserva o de 2000."
    )

    por_indice = {}
    for r in novos["treino"] + novos["validacao"]:
        por_indice.setdefault(r["indice"], r)

    treino_ids = {r["indice"] for r in novos["treino"]}
    val_ids = {r["indice"] for r in novos["validacao"]}
    assert len(treino_ids) == 1304 and len(val_ids) == 145
    assert not treino_ids & val_ids

    plano = []
    for r in antigo5000["treino"]:
        assert r["indice"] in treino_ids
        base = por_indice[r["indice"]]
        assert base["imagem_origem"] == r["imagem"]
        assert base["imagem_sha256"] == r["imagem_sha256"]
        assert base["cache256_sha256"] == r["cache_sha256"]
        assert base["split"] == r["split"] == "treino"
        assert r["regiao"] in base["candidatos"]
        plano.append(dict(base, regiao=r["regiao"]))

    assert plano[:2000] == novos["treino"]
    print("Plano conferido: mesmos 5000 episódios do histórico.", flush=True)

    print("Conferindo integridade dos 1449 caches de 512...", flush=True)
    for i, r in enumerate(por_indice.values(), 1):
        assert sha(DADOS / r["cache"]) == r["cache_sha256"], r["cache"]
        if i % 300 == 0:
            print(f"Caches: {i}/1449", flush=True)

    return {"treino": plano, "validacao": novos["validacao"]}

def versao(nome):
    try:
        return importlib.metadata.version(nome)
    except importlib.metadata.PackageNotFoundError:
        return None

def main(pasta):
    assert torch.cuda.is_available()
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)

    plano = preparar_plano()
    fontes = [
        Path(__file__), CODIGO / "eft_modelo_512.py",
        CODIGO / "eft_alinhamento_seguro.py",
        ROOT / "models/vmamba.py",
        ROOT / "models/grid_proto_fewshot.py",
        ROOT / "models/alpmodule.py",
    ]
    pesos = ROOT / "pretrained_model/vmamba_tiny_v2.pth"
    digest_pesos = sha(pesos)
    assert digest_pesos == "09739f6d95638e5caf0d33fcbca85b7cff62b8ca16ec2926d781109939c6b201"

    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    ).stdout.strip()

    config = {
        "formato": 1, "total_passos": 5000, "semente": 1234,
        "inicio": "pesos pre-treinados; sem checkpoint EFT anterior",
        "checkpoint_origem": str(pesos),
        "checkpoint_origem_sha256": digest_pesos,
        "resolucao_treino": [512, 512], "grade_alp": [32, 32],
        "batch_episodico": 1, "adaptacao": "encoder completo, features da ultima camada",
        "otimizador": "AdamW", "lr": 1e-5, "betas": [0.9, 0.999],
        "eps": 1e-8, "weight_decay": 0.0,
        "amp": False, "scheduler": None, "clipping": None,
        "loss": "CE + alinhamento; pesos 1 e 1",
        "alinhamento_seguro": "ativo desde o primeiro passo",
        "dados": str(DADOS),
        "dados_config_sha256": sha(DADOS / "config.json"),
        "plano512_origem_sha256": sha(DADOS / "planos.json"),
        "plano5000_historico_sha256": sha(H5000 / "planos.json"),
        "checkpoint_a_cada": 50, "checkpoints_preservados": sorted(PRESERVAR),
        "avaliacao_polipos": "separada: encoder672/SAM1024",
        "limitacao_retomada": "estado restaurado; backward observado nao identico bit a bit",
        "fontes_sha256": {str(p): sha(p) for p in fontes},
        "ambiente": {
            "python": platform.python_version(),
            "pacotes": {n: versao(n) for n in (
                "torch", "torchvision", "numpy", "Pillow", "timm", "fvcore", "triton"
            )},
            "cuda": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version(),
            "gpu_driver": gpu_info,
            "tf32_matmul": False, "tf32_cudnn": True,
            "cudnn_benchmark": False, "deterministico": False,
        },
    }

    pasta.mkdir(parents=True, exist_ok=True)
    cfg = pasta / "config.json"
    if cfg.exists():
        assert ler(cfg) == config, "Código/configuração/ambiente mudou. Retomada interrompida."
        assert ler(pasta / "planos.json") == plano, "Plano da execução mudou."
    else:
        assert not (pasta / "checkpoint_latest.pt").exists()
        json_atomico(cfg, config)
        json_atomico(pasta / "planos.json", plano)
        (pasta / "fontes").mkdir(exist_ok=True)
        for i, p in enumerate(fontes):
            shutil.copy2(p, pasta / "fontes" / f"{i:02d}_{p.name}")
        shutil.copy2(DADOS / "config.json", pasta / "config_dados.json")

    print("EXECUÇÃO:", pasta, flush=True)
    modelo = construir_modelo("tiny", repo=ROOT)
    estado_align = instalar(modelo)
    otimizador = torch.optim.AdamW(
        modelo.parameters(), lr=1e-5, betas=(0.9, 0.999),
        eps=1e-8, weight_decay=0.0
    )
    latest = pasta / "checkpoint_latest.pt"

    def preservar(n):
        destino = pasta / f"checkpoint_{n:04d}.pt"
        if n in PRESERVAR and not destino.exists():
            try:
                os.link(latest, destino)
            except OSError:
                tmp = destino.with_suffix(".tmp")
                shutil.copy2(latest, tmp)
                os.replace(tmp, destino)

    def salvar(n):
        tmp = latest.with_suffix(".tmp")
        torch.save({
            "formato": 1, "passo": n,
            "modelo": modelo.state_dict(),
            "otimizador": otimizador.state_dict(),
            "scheduler": None, "rng": rng_atual(), "config": config,
        }, tmp)
        os.replace(tmp, latest)
        preservar(n)

    passo = 0
    if latest.exists():
        salvo = torch.load(latest, map_location="cpu", weights_only=True)
        assert salvo["formato"] == 1 and salvo["config"] == config
        assert salvo["scheduler"] is None
        modelo.load_state_dict(salvo["modelo"], strict=True)
        otimizador.load_state_dict(salvo["otimizador"])
        passo = int(salvo["passo"])
        assert 0 <= passo <= 5000
        restaurar_rng(salvo["rng"])
        del salvo
        preservar(passo)
        print(f"Retomada após passo {passo}.", flush=True)
    else:
        salvar(0)
        print("Início novo: checkpoint 0 salvo, pesos pré-treinados.", flush=True)

    tentativa = time.time_ns()
    log = pasta / f"passos_{tentativa}.jsonl"
    modelo.train()
    torch.cuda.reset_peak_memory_stats()
    inicio_execucao = time.perf_counter()
    passo_inicial = passo

    with log.open("x", encoding="utf-8", buffering=1) as f:
        for n in range(passo + 1, 5001):
            inicio = time.perf_counter()
            r = plano["treino"][n - 1]
            s, fg, q, alvo = episodio(r)
            otimizador.zero_grad(set_to_none=True)
            estado_align["ultimo"] = None

            retorno = modelo(
                [[s]], [[fg]], [[1 - fg]], [q],
                isval=False, val_wsize=2, show_viz=False
            )
            logits, alinhamento = retorno[:2]
            assert logits.shape == (1, 2, 512, 512)
            assert torch.isfinite(logits).all().item(), "Logits não finitos."

            ce = F.cross_entropy(logits, alvo)
            loss = ce + alinhamento
            assert torch.isfinite(loss).item(), "Loss não finita."
            loss.backward()

            grads = [p.grad for p in modelo.encoder.parameters() if p.grad is not None]
            assert grads
            assert torch.stack([torch.isfinite(g).all() for g in grads]).all().item()
            assert torch.stack([g.abs().max() for g in grads]).max().item() > 0

            otimizador.step()
            assert torch.stack([
                torch.isfinite(p).all() for p in modelo.parameters()
            ]).all().item(), "Pesos não finitos."

            linha = {
                "passo": n, "indice": r["indice"], "imagem": r["imagem"],
                "regiao": r["regiao"], "ce": ce.item(),
                "alinhamento": float(alinhamento.detach())
                    if torch.is_tensor(alinhamento) else float(alinhamento),
                "total": loss.item(),
                "alinhamento_excecao": estado_align["ultimo"],
                "segundos": time.perf_counter() - inicio,
            }
            f.write(json.dumps(linha, ensure_ascii=False) + "\n")
            del retorno, logits, alinhamento, ce, loss, grads, s, fg, q, alvo
            otimizador.zero_grad(set_to_none=True)

            if n % 50 == 0:
                salvar(n)
                print(f"Checkpoint salvo: {n}", flush=True)

            if n == passo_inicial + 1 or n % 10 == 0:
                media = (time.perf_counter() - inicio_execucao) / (n - passo_inicial)
                restante_h = media * (5000 - n) / 3600
                print(
                    f"{n}/5000 | loss={linha['total']:.6f} | "
                    f"CE={linha['ce']:.6f} | align={linha['alinhamento']:.6f} | "
                    f"{media:.2f}s/passo | restante estimado={restante_h:.2f}h",
                    flush=True
                )

    json_atomico(pasta / "concluido.json", {
        "passos": 5000,
        "checkpoint": str(latest),
        "pico_memoria_alocada_GiB": torch.cuda.max_memory_allocated() / 1024**3,
        "avaliacao_polipos_pendente": True,
    })
    print("TREINAMENTO CONCLUÍDO: 5000 passos.", flush=True)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--run", required=True)
    args = parser.parse_args()
    pasta = Path(args.run).resolve()
    try:
        main(pasta)
    except BaseException:
        if pasta.is_dir():
            (pasta / f"erro_{time.time_ns()}.txt").write_text(
                traceback.format_exc(), encoding="utf-8"
            )
        raise

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
SCRIPT = ROOT / "exp/eft_tiny_512_v1/treinar_5000.py"

# Nova execução: None.
# Para retomar: "/home/1542521/.../eft_tiny512_5000_..."
RETOMAR = None

if "processo5000" in globals() and processo5000.poll() is None:
    raise RuntimeError("O treinamento desta sessão já está executando.")

if RETOMAR is None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    EXECUCAO5000 = ROOT / "reproducao" / f"eft_tiny512_5000_{stamp}"
    EXECUCAO5000.mkdir(parents=True, exist_ok=False)
else:
    EXECUCAO5000 = Path(RETOMAR)
    assert (EXECUCAO5000 / "config.json").is_file()

# Verifica a sintaxe antes de iniciar.
compile(SCRIPT.read_text(encoding="utf-8"), str(SCRIPT), "exec")

LOG5000 = EXECUCAO5000 / "console.log"
with LOG5000.open("ab", buffering=0) as log:
    processo5000 = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT), "--run", str(EXECUCAO5000)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(EXECUCAO5000 / "processo.pid").write_text(str(processo5000.pid))
print("PID:", processo5000.pid)
print("EXECUÇÃO — guarde este caminho:", EXECUCAO5000)
print("LOG:", LOG5000)

In [ ]:
print("Estado:", "executando" if processo5000.poll() is None
      else f"encerrado, código {processo5000.returncode}")

with LOG5000.open("rb") as f:
    f.seek(0, 2)
    tamanho = f.tell()
    f.seek(max(0, tamanho - 12000))
    texto = f.read().decode("utf-8", errors="replace")

print("\n".join(texto.splitlines()[-25:]))

In [ ]:
import subprocess

for comando in [
    ["ps", "-p", "366785", "-o", "pid,stat,etime,%cpu,%mem,args"],
    ["nvidia-smi"],
]:
    resultado = subprocess.run(comando, capture_output=True, text=True)
    print(resultado.stdout or resultado.stderr)

print("\n=== ÚLTIMAS LINHAS DO TREINO ===")
with LOG5000.open("rb") as f:
    f.seek(0, 2)
    f.seek(max(0, f.tell() - 8000))
    print(f.read().decode("utf-8", errors="replace"))

In [ ]:
from pathlib import Path
import os
import signal
import json
import torch

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXECUCAO5000 = ROOT / "reproducao/eft_tiny512_5000_20261016T220313152819Z"
SCRIPT = ROOT / "exp/eft_tiny_512_v1/treinar_5000.py"

pid = int((EXECUCAO5000 / "processo.pid").read_text())
proc = Path(f"/proc/{pid}/cmdline")

if proc.exists():
    argumentos = [
        x.decode() for x in proc.read_bytes().split(b"\0") if x
    ]
    assert str(SCRIPT) in argumentos, "PID pertence a outro programa."
    assert "--run" in argumentos
    assert argumentos[argumentos.index("--run") + 1] == str(EXECUCAO5000)
    os.kill(pid, signal.SIGTERM)
    print("Encerramento solicitado ao treinamento:", pid)
else:
    print("O processo já não está ativo.")

checkpoint = EXECUCAO5000 / "checkpoint_latest.pt"
assert checkpoint.is_file(), "Checkpoint não encontrado."
salvo = torch.load(checkpoint, map_location="cpu", weights_only=True)
print("Último passo salvo:", salvo["passo"])
print("Checkpoint preservado:", checkpoint)
del salvo

In [ ]:
from pathlib import Path
import importlib.util
import importlib.metadata
import re
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

print("=== AMBIENTE ===")
print("Python:", sys.executable)
for nome in ("torch", "torchvision", "triton", "mamba-ssm", "causal-conv1d"):
    try:
        print(nome, importlib.metadata.version(nome))
    except importlib.metadata.PackageNotFoundError:
        print(nome, "não registrado como pacote")

print("\n=== EXTENSÕES LOCALIZÁVEIS ===")
for nome in (
    "selective_scan_cuda",
    "selective_scan_cuda_core",
    "selective_scan_cuda_oflex",
    "selective_scan",
):
    try:
        spec = importlib.util.find_spec(nome)
        print(nome, "→", spec.origin if spec else "não encontrada")
    except (ImportError, ValueError) as erro:
        print(nome, "→", repr(erro))

print("\n=== SELEÇÃO DE BACKEND NO VMAMBA ===")
arquivo = ROOT / "models/vmamba.py"
linhas = arquivo.read_text(encoding="utf-8").splitlines()
padrao = re.compile(
    r"selective_scan_cuda|WITH_SELECTIVESCAN|"
    r"def selective_scan|class SelectiveScan|"
    r"backend\s*[=:]|v05_noz|v05[\"']"
)

mostrar = set()
for i, linha in enumerate(linhas):
    if padrao.search(linha):
        mostrar.update(range(max(0, i - 5), min(len(linhas), i + 12)))

anterior = -2
for i in sorted(mostrar):
    if i != anterior + 1:
        print("\n...")
    print(f"{i + 1:04d}: {linhas[i]}")
    anterior = i

print("\n=== GPU ===")
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import torch
from torch.utils.cpp_extension import CUDA_HOME

print("=== PYTORCH E GPU ===")
print("PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("ABI C++:", torch._C._GLIBCXX_USE_CXX11_ABI)

print("\n=== COMPILADORES ===")
print("CUDA_HOME detectado:", CUDA_HOME)
print("CUDA_HOME no ambiente:", os.environ.get("CUDA_HOME"))

candidatos = [
    shutil.which("nvcc"),
    str(Path(CUDA_HOME) / "bin/nvcc") if CUDA_HOME else None,
]
nvccs = list(dict.fromkeys(
    p for p in candidatos if p and Path(p).is_file()
))

for executavel in nvccs + [
    p for p in (shutil.which("g++"), shutil.which("ninja")) if p
]:
    print("\nExecutável:", executavel)
    resultado = subprocess.run(
        [executavel, "--version"],
        capture_output=True, text=True, timeout=20
    )
    print(resultado.stdout or resultado.stderr)

if not nvccs:
    print("\nnvcc não encontrado nos caminhos consultados.")
    print("Instalações CUDA em /usr/local:")
    for p in sorted(Path("/usr/local").glob("cuda*")):
        print(p)

print("\n=== CHECKPOINT PRESERVADO ===")
execucao = Path(
    "/home/1542521/cassio_pereira/ProtoSAM-repro/reproducao/"
    "eft_tiny512_5000_20261016T220313152819Z"
)
checkpoint = execucao / "checkpoint_latest.pt"
salvo = torch.load(checkpoint, map_location="cpu", weights_only=True)
print("Último passo salvo:", salvo["passo"])
del salvo

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys
import urllib.request
import torch

assert sys.version_info[:2] == (3, 10)
assert torch.__version__.startswith("2.7.1+cu128")
assert torch._C._GLIBCXX_USE_CXX11_ABI is True
assert platform.machine() == "x86_64"

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DESTINO = ROOT / "reproducao/dependencias_scan_cuda"
DESTINO.mkdir(parents=True, exist_ok=True)

TAG = "v2.3.2.post1"
NOME = (
    "mamba_ssm-2.3.2.post1+cu12torch2.7cxx11abiTRUE"
    "-cp310-cp310-linux_x86_64.whl"
)

url_api = f"https://api.github.com/repos/state-spaces/mamba/releases/tags/{TAG}"
pedido = urllib.request.Request(
    url_api,
    headers={
        "User-Agent": "ProtoSAM-environment-check",
        "Accept": "application/vnd.github+json",
    },
)

print("Consultando release oficial...", flush=True)
with urllib.request.urlopen(pedido, timeout=60) as resposta:
    release = json.load(resposta)

encontrados = [a for a in release["assets"] if a["name"] == NOME]
if len(encontrados) != 1:
    print("Arquivos próximos disponíveis:")
    for a in release["assets"]:
        if "torch2.7" in a["name"] and "cp310" in a["name"]:
            print(" ", a["name"])
    raise RuntimeError(
        "Wheel exata não encontrada. Nenhuma instalação realizada; envie a saída."
    )

asset = encontrados[0]
wheel = DESTINO / NOME

if not wheel.exists():
    temporario = wheel.with_suffix(".download")
    print(f"Baixando {NOME} ({asset['size'] / 1024**2:.0f} MiB)...", flush=True)
    with urllib.request.urlopen(asset["browser_download_url"], timeout=60) as src:
        with temporario.open("wb") as dst:
            while bloco := src.read(1024 * 1024):
                dst.write(bloco)
    assert temporario.stat().st_size == asset["size"]
    temporario.replace(wheel)

assert wheel.stat().st_size == asset["size"], "Download incompleto."
h = hashlib.sha256()
with wheel.open("rb") as f:
    for bloco in iter(lambda: f.read(1024 * 1024), b""):
        h.update(bloco)
digest = h.hexdigest()

digest_oficial = asset.get("digest")
if digest_oficial:
    assert digest_oficial == f"sha256:{digest}", "Hash diferente do publicado."

print("SHA256:", digest)
registro = {
    "release": TAG,
    "arquivo": NOME,
    "url": asset["browser_download_url"],
    "sha256": digest,
    "digest_publicado": digest_oficial,
}
(DESTINO / "wheel_instalada.json").write_text(
    json.dumps(registro, indent=2), encoding="utf-8"
)

print("Instalando a wheel, sem alterar dependências...", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", str(wheel)],
    check=True
)

print("\nVerificando em um processo novo...", flush=True)
codigo = """
import torch
import selective_scan_cuda
from models import vmamba
print("PyTorch:", torch.__version__)
print("Extensão:", selective_scan_cuda.__file__)
print("WITH_SELECTIVESCAN_MAMBA:", vmamba.WITH_SELECTIVESCAN_MAMBA)
assert vmamba.WITH_SELECTIVESCAN_MAMBA
"""
resultado = subprocess.run(
    [sys.executable, "-u", "-c", codigo],
    cwd=ROOT, capture_output=True, text=True
)
print(resultado.stdout)
print(resultado.stderr)
assert resultado.returncode == 0, "A importação falhou; envie a saída."

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
BENCH512 = ROOT / "reproducao" / f"eft512_benchmark_cuda_{stamp}"
BENCH512.mkdir(parents=True, exist_ok=False)

codigo = r'''
from pathlib import Path
import hashlib
import json
import statistics
import sys
import time
import torch
import torch.nn.functional as F
import selective_scan_cuda

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "exp/eft_tiny_512_v1"))

from models import vmamba
from eft_modelo_512 import construir_modelo
from eft_alinhamento_seguro import instalar
from treinar_5000 import episodio, DADOS

DESTINO = Path(sys.argv[1])
assert vmamba.WITH_SELECTIVESCAN_MAMBA

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(False)

modelo = construir_modelo("tiny", repo=ROOT)
estado = instalar(modelo)
otimizador = torch.optim.AdamW(
    modelo.parameters(), lr=1e-5,
    betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0
)
planos = json.loads((DADOS / "planos.json").read_text())
modelo.train()

# Confere que a execução realmente passa pela extensão CUDA.
chamadas = {"forward_cuda": 0}
forward_original = selective_scan_cuda.fwd

def observar_forward(*args, **kwargs):
    chamadas["forward_cuda"] += 1
    return forward_original(*args, **kwargs)

selective_scan_cuda.fwd = observar_forward

def relogio():
    torch.cuda.synchronize()
    return time.perf_counter()

torch.cuda.reset_peak_memory_stats()
linhas = []

try:
    for i in range(3):
        print(f"\nDiagnóstico {i + 1}/3...", flush=True)
        inicio = relogio()
        s, fg, q, alvo = episodio(planos["treino"][i])
        t_dados = relogio()

        otimizador.zero_grad(set_to_none=True)
        estado["ultimo"] = None
        contagem_antes = chamadas["forward_cuda"]

        retorno = modelo(
            [[s]], [[fg]], [[1 - fg]], [q],
            isval=False, val_wsize=2, show_viz=False
        )
        logits, alinhamento = retorno[:2]
        ce = F.cross_entropy(logits, alvo)
        loss = ce + alinhamento
        t_forward = relogio()

        assert logits.shape == (1, 2, 512, 512)
        assert torch.isfinite(logits).all().item()
        assert torch.isfinite(loss).item()
        assert chamadas["forward_cuda"] > contagem_antes

        t_backward_inicio = relogio()
        loss.backward()
        t_backward = relogio()

        grads = [p.grad for p in modelo.encoder.parameters() if p.grad is not None]
        assert grads
        assert torch.stack([torch.isfinite(g).all() for g in grads]).all().item()
        assert torch.stack([g.abs().max() for g in grads]).max().item() > 0
        t_checks = relogio()

        otimizador.step()
        t_otimizador = relogio()

        assert torch.stack([
            torch.isfinite(p).all() for p in modelo.parameters()
        ]).all().item()
        fim = relogio()

        linha = {
            "passo_diagnostico": i + 1,
            "dados_s": t_dados - inicio,
            "forward_loss_s": t_forward - t_dados,
            "backward_s": t_backward - t_backward_inicio,
            "checagem_gradientes_s": t_checks - t_backward,
            "otimizador_s": t_otimizador - t_checks,
            "total_s": fim - inicio,
            "ce": ce.item(),
            "loss": loss.item(),
            "chamadas_cuda": chamadas["forward_cuda"] - contagem_antes,
            "excecao_alinhamento": estado["ultimo"],
        }
        linhas.append(linha)
        print(json.dumps(linha, indent=2), flush=True)
        del retorno, logits, alinhamento, ce, loss, grads, s, fg, q, alvo
        otimizador.zero_grad(set_to_none=True)
finally:
    selective_scan_cuda.fwd = forward_original

# Exclui o primeiro passo da estimativa por custos de inicialização.
media = statistics.mean(r["total_s"] for r in linhas[1:])
extensao = Path(selective_scan_cuda.__file__)
h = hashlib.sha256()
with extensao.open("rb") as f:
    for bloco in iter(lambda: f.read(1024 * 1024), b""):
        h.update(bloco)

resumo = {
    "backend": "selective_scan_cuda",
    "extensao": str(extensao),
    "extensao_sha256": h.hexdigest(),
    "torch": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0),
    "passos": linhas,
    "media_passos_2_e_3_s": media,
    "estimativa_5000_passos_h_sem_avaliacao_e_salvamento": media * 5000 / 3600,
    "pico_memoria_alocada_GiB": torch.cuda.max_memory_allocated() / 1024**3,
}
(DESTINO / "resultado.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("\nRESUMO:", json.dumps(resumo, indent=2), flush=True)
print("Diagnóstico concluído; pesos descartados ao encerrar o processo.", flush=True)
'''

script = BENCH512 / "benchmark.py"
script.write_text(codigo, encoding="utf-8")

with (BENCH512 / "console.log").open("w", encoding="utf-8") as log:
    processo = subprocess.Popen(
        [sys.executable, "-u", str(script), str(BENCH512)],
        cwd=ROOT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for linha in processo.stdout:
        print(linha, end="", flush=True)
        log.write(linha)
        log.flush()
    retorno = processo.wait()

print("\nArquivos do diagnóstico:", BENCH512)
assert retorno == 0, "O teste falhou; envie o traceback acima."

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
ANTIGA = ROOT / "reproducao/eft_tiny512_5000_20261016T220313152819Z"
BENCH = ROOT / "reproducao/eft512_benchmark_cuda_20261017T101300393042Z"

if "processoCUDA" in globals() and processoCUDA.poll() is None:
    raise RuntimeError("O treinamento CUDA já está executando nesta sessão.")

def sha_arquivo(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

origem = CODIGO / "treinar_5000.py"
config_antiga = json.loads((ANTIGA / "config.json").read_text())
assert sha_arquivo(origem) == config_antiga["fontes_sha256"][str(origem)], (
    "O programa de treinamento mudou desde a execução anterior."
)

benchmark = json.loads((BENCH / "resultado.json").read_text())
assert benchmark["backend"] == "selective_scan_cuda"
assert all(p["chamadas_cuda"] > 0 for p in benchmark["passos"])
hash_extensao = benchmark["extensao_sha256"]

texto = origem.read_text(encoding="utf-8")

# Falha explicitamente se a extensão desaparecer ou mudar.
ancora = "def main(pasta):\n    assert torch.cuda.is_available()"
assert texto.count(ancora) == 1
texto = texto.replace(
    ancora,
    "def main(pasta):\n"
    "    import selective_scan_cuda\n"
    "    from models import vmamba\n"
    "    assert vmamba.WITH_SELECTIVESCAN_MAMBA, 'Backend CUDA indisponivel.'\n"
    f"    assert sha(selective_scan_cuda.__file__) == {hash_extensao!r}, "
    "'Extensao CUDA diferente da validada.'\n"
    "    assert torch.cuda.is_available()"
)

# Registra backend e versão no config e em todos os checkpoints.
ancora = '        "formato": 1, "total_passos": 5000, "semente": 1234,'
assert texto.count(ancora) == 1
texto = texto.replace(
    ancora,
    ancora + "\n"
    '        "backend_scan": "selective_scan_cuda",\n'
    '        "mamba_ssm": versao("mamba-ssm"),\n'
    '        "extensao_scan": str(Path(selective_scan_cuda.__file__).resolve()),\n'
    '        "extensao_scan_sha256": sha(selective_scan_cuda.__file__),\n'
    f'        "benchmark_previo": {str(BENCH)!r},\n'
    f'        "benchmark_sha256": {sha_arquivo(BENCH / "resultado.json")!r},'
)

# A observação de não determinismo anterior foi medida no backend Torch.
antigo = '"estado restaurado; backward observado nao identico bit a bit"'
novo = (
    '"retomada salva pesos, otimizador e RNG; identidade bit a bit nao garantida; '
    'diagnostico anterior de variacao realizado no backend Torch"'
)
assert texto.count(antigo) == 1
texto = texto.replace(antigo, novo)

SCRIPT_CUDA = CODIGO / "treinar_5000_cuda.py"
compile(texto, str(SCRIPT_CUDA), "exec")
if SCRIPT_CUDA.exists():
    assert SCRIPT_CUDA.read_text(encoding="utf-8") == texto, (
        "Já existe uma versão diferente do programa CUDA."
    )
else:
    with SCRIPT_CUDA.open("x", encoding="utf-8") as f:
        f.write(texto)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EXECUCAO_CUDA = ROOT / "reproducao" / f"eft_tiny512_cuda_5000_{stamp}"
EXECUCAO_CUDA.mkdir(parents=True, exist_ok=False)

comando = [
    sys.executable, "-u", str(SCRIPT_CUDA),
    "--run", str(EXECUCAO_CUDA)
]
(EXECUCAO_CUDA / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

LOG_CUDA = EXECUCAO_CUDA / "console.log"
with LOG_CUDA.open("ab", buffering=0) as log:
    processoCUDA = subprocess.Popen(
        comando, cwd=ROOT,
        stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True
    )

(EXECUCAO_CUDA / "processo.pid").write_text(str(processoCUDA.pid))
print("PID:", processoCUDA.pid)
print("NOVA EXECUÇÃO:", EXECUCAO_CUDA)
print("LOG:", LOG_CUDA)
print("Início nos pesos pré-treinados; execução anterior preservada.")

In [ ]:
print(
    "Estado:",
    "executando" if processoCUDA.poll() is None
    else f"encerrado, código {processoCUDA.returncode}"
)

with LOG_CUDA.open("rb") as f:
    f.seek(0, 2)
    tamanho = f.tell()
    f.seek(max(0, tamanho - 12000))
    texto = f.read().decode("utf-8", errors="replace")

print("\n".join(texto.splitlines()[-25:]))

In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXECUCAO = ROOT / (
    "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"
)

print("=== CONCLUSÃO DO TREINAMENTO ===")
print((EXECUCAO / "concluido.json").read_text(encoding="utf-8"))

print("\n=== CHECKPOINTS PARA AVALIAÇÃO ===")
for passo in (0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000):
    p = EXECUCAO / f"checkpoint_{passo:04d}.pt"
    print(f"Passo {passo:4d}: {'OK' if p.is_file() else 'AUSENTE'}")
    assert p.is_file(), f"Checkpoint ausente: {p}"

print("\n=== CONFIGURAÇÃO DA NOVA EXECUÇÃO ===")
config = json.loads((EXECUCAO / "config.json").read_text())
for chave in (
    "inicio", "resolucao_treino", "grade_alp", "backend_scan",
    "mamba_ssm", "checkpoint_origem_sha256", "avaliacao_polipos"
):
    print(f"{chave}: {config.get(chave)}")

for nome in (
    "exp/eft_validacao_polipos_v1/eft_modelo_validacao.py",
    "exp/eft_validacao_polipos_v1/validar_marcos_5000.py",
):
    p = ROOT / nome
    print(f"\n{'=' * 70}\nARQUIVO: {nome}\n{'=' * 70}")
    assert p.is_file(), f"Arquivo ausente: {p}"
    print(p.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

print("=== LOCALIZAÇÃO E CARREGAMENTO DO DATASET ===")
p = ROOT / "dataloaders/PolypDataset.py"
print(p.read_text(encoding="utf-8"))

print("\n=== INTEGRIDADE DOS ARQUIVOS DE INFERÊNCIA ===")
referencia = ROOT / (
    "reproducao/eft_tiny_passo_2000_validacao_20261011T221527472147Z/"
    "config.json"
)
assert referencia.is_file(), f"Referência não encontrada: {referencia}"
ref = json.loads(referencia.read_text(encoding="utf-8"))

arquivos = [
    "models/grid_proto_fewshot.py",
    "models/alpmodule.py",
    "models/ProtoSAM.py",
    "dataloaders/PolypDataset.py",
    "dataloaders/PolypTransforms.py",
    "models/segment_anything/utils/transforms.py",
    "models/vmamba.py",
]

for relativo in arquivos:
    atual = ROOT / relativo
    digest = hashlib.sha256(atual.read_bytes()).hexdigest()
    candidatos = [
        valor for nome, valor in ref["fontes_sha256"].items()
        if nome.endswith("/" + relativo)
    ]
    if len(candidatos) != 1:
        print("SEM REFERÊNCIA ÚNICA:", relativo, "| SHA256:", digest)
    else:
        print(
            "CONFERE" if digest == candidatos[0] else "DIFERENTE",
            "|", relativo, "| SHA256:", digest
        )

print("\n=== SUPORTE REGISTRADO NA AVALIAÇÃO ANTERIOR ===")
print(json.dumps(
    {
        "suporte": ref.get("suporte"),
        "suporte_tensor_sha256": ref.get("suporte_tensor_sha256"),
        "checkpoint_sam_sha256": ref.get("checkpoint_sam_sha256"),
        "numero_queries": len(ref.get("episodios", [])),
    },
    indent=2, ensure_ascii=False
))


In [ ]:
%%writefile /home/1542521/cassio_pereira/ProtoSAM-repro/exp/eft_tiny_512_v1/avaliar_checkpoints.py
from pathlib import Path
import argparse
import csv
import gc
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import time
import traceback

import numpy as np
from PIL import Image
import torch
import selective_scan_cuda

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO = ROOT / "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"
REFERENCIA = ROOT / "reproducao/eft_tiny_passo_2000_validacao_20261011T221527472147Z"
LEGADO = ROOT / "exp/eft_validacao_polipos_v1"
PASSOS = [0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000]

os.chdir(ROOT)
sys.path.insert(0, str(LEGADO))
sys.path.insert(0, str(ROOT))

# Reutiliza somente as funções de construção e métricas já inspecionadas.
# Não chama executar() nem modifica os arquivos antigos.
import validar_marcos_5000 as legado
legado.REPO = ROOT

from dataloaders.PolypDataset import PolypDataset
from dataloaders.PolypTransforms import get_polyp_transform
from models.segment_anything.utils.transforms import ResizeLongestSide
from models.ProtoSAM import ProtoSAM, ALPNetWrapper, ALPNetInput
from models import vmamba

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def salvar(p, obj):
    p = Path(p)
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, p)

def csv_atomico(p, linhas):
    p = Path(p)
    tmp = p.with_suffix(".tmp")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0]))
        w.writeheader()
        w.writerows(linhas)
    os.replace(tmp, p)

def atual(p):
    p = Path(p)
    if p.is_relative_to("/workspace"):
        p = ROOT.parent / p.relative_to("/workspace")
    return p.resolve()

def hash_tensor(t):
    return hashlib.sha256(t.detach().cpu().numpy().tobytes()).hexdigest()

def main(out):
    assert torch.cuda.is_available() and vmamba.WITH_SELECTIVESCAN_MAMBA
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.use_deterministic_algorithms(False)

    ref = ler(REFERENCIA / "config.json")
    cfg_treino = ler(TREINO / "config.json")
    planos = ler(TREINO / "planos.json")
    assert ler(TREINO / "concluido.json")["passos"] == 5000
    assert sha(selective_scan_cuda.__file__) == cfg_treino["extensao_scan_sha256"]

    oficiais = [
        "models/grid_proto_fewshot.py", "models/alpmodule.py",
        "models/ProtoSAM.py", "dataloaders/PolypDataset.py",
        "dataloaders/PolypTransforms.py",
        "models/segment_anything/utils/transforms.py", "models/vmamba.py",
    ]
    for relativo in oficiais:
        hashes = [
            v for k, v in ref["fontes_sha256"].items()
            if k.endswith("/" + relativo)
        ]
        assert len(hashes) == 1 and sha(ROOT / relativo) == hashes[0], relativo

    sup_meta = dict(ref["suporte"])
    for chave in ("imagem", "mascara"):
        sup_meta[chave] = str(atual(sup_meta[chave]))
        assert sha(sup_meta[chave]) == sup_meta[chave + "_sha256"]

    reservadas = {f'{r["indice"]:04d}': r for r in planos["validacao"]}
    assert len(reservadas) == 145
    assert not (
        {r["indice"] for r in planos["validacao"]}
        & {r["indice"] for r in planos["treino"]}
    )
    assert {r["id"] for r in ref["episodios"]} == set(reservadas)

    # Mesma construção do ds_train da factory; somente caminhos atualizados.
    _, transform_test = get_polyp_transform()
    image_root = str(Path(sup_meta["imagem"]).parent)
    gt_root = str(Path(sup_meta["mascara"]).parent)
    ds = PolypDataset(
        root=image_root, image_root=image_root, gt_root=gt_root,
        augmentations=transform_test, sam_trans=ResizeLongestSide(1024),
        train=True, image_size=(1024, 1024)
    )
    mapa = {str(Path(p).resolve()): i for i, p in enumerate(ds.images)}
    assert len(mapa) == len(ds.images)

    itens = []
    for antigo in ref["episodios"]:
        item = dict(antigo)
        r = reservadas[item["id"]]
        for chave in ("imagem", "mascara"):
            item[chave] = str(atual(item[chave]))
            assert sha(item[chave]) == item[chave + "_sha256"]
        assert item["imagem"] == str(atual(r["imagem"]))
        assert item["imagem_sha256"] == r["imagem_sha256"]
        i = mapa[item["imagem"]]
        assert Path(ds.gts[i]).resolve() == Path(item["mascara"])
        item["indice_dataset"] = i
        itens.append(item)

    assert sup_meta["imagem"] not in {i["imagem"] for i in itens}
    indice_sup = mapa[sup_meta["imagem"]]
    assert Path(ds.gts[indice_sup]).resolve() == Path(sup_meta["mascara"])
    suporte = ds[indice_sup]
    legado.conferir_amostra(suporte)
    assert int((suporte["label"] > .5).sum()) == 51940
    assert hash_tensor(suporte["image"]) == ref["suporte_tensor_sha256"], (
        "Tensor do suporte difere da referência."
    )

    cfg_sam = dict(ref["sam"])
    cfg_sam["sam_pretrained_path"] = str(ROOT / "pretrained_model/sam_vit_h.pth")
    assert sha(cfg_sam["sam_pretrained_path"]) == ref["checkpoint_sam_sha256"]

    fontes = [Path(__file__), LEGADO / "validar_marcos_5000.py",
              LEGADO / "eft_modelo_validacao.py"] + [ROOT / r for r in oficiais]
    checkpoints = {
        str(n): sha(TREINO / f"checkpoint_{n:04d}.pt") for n in PASSOS
    }
    config = {
        "finalidade": "validacao anotada; selecao exploratoria de checkpoint",
        "treino": str(TREINO), "treino_config_sha256": sha(TREINO / "config.json"),
        "passos": PASSOS, "checkpoints_sha256": checkpoints,
        "episodios": itens, "suporte": sup_meta,
        "suporte_tensor_sha256": hash_tensor(suporte["image"]),
        "encoder": 672, "grade_alp": [32, 32], "canvas": 1024,
        "sam": cfg_sam, "sam_sha256": ref["checkpoint_sam_sha256"],
        "selecao_sam": ref["selecao_sam"],
        "alp_inferencia": ref["alp_inferencia"],
        "referencia_config_sha256": sha(REFERENCIA / "config.json"),
        "fontes_sha256": {str(p): sha(p) for p in fontes},
        "backend": "selective_scan_cuda",
        "extensao_sha256": sha(selective_scan_cuda.__file__),
        "torch": str(torch.__version__), "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "pacotes": {p: importlib.metadata.version(p) for p in [
            "numpy", "Pillow", "torchvision", "timm", "scipy",
            "opencv-python", "mamba-ssm"
        ] if importlib.util.find_spec(
            {"Pillow": "PIL", "opencv-python": "cv2", "mamba-ssm": "mamba_ssm"}.get(p, p)
        ) is not None},
        "gpu_driver": subprocess.check_output([
            "nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"
        ], text=True).strip(),
    }
    out.mkdir(parents=True, exist_ok=True)
    if (out / "config.json").exists():
        assert ler(out / "config.json") == config, "Configuração mudou; retomada recusada."
    else:
        salvar(out / "config.json", config)
        (out / "fontes").mkdir(exist_ok=True)
        for i, p in enumerate(fontes):
            shutil.copy2(p, out / "fontes" / f"{i:02d}_{p.name}")

    print("AVALIAÇÃO:", out, flush=True)
    print("Suporte, dados e código conferidos. 145 queries por checkpoint.", flush=True)

    original_fwd = selective_scan_cuda.fwd
    original_fallback = vmamba.selective_scan_torch
    chamadas = {"cuda": 0}
    def contar(*args, **kwargs):
        chamadas["cuda"] += 1
        return original_fwd(*args, **kwargs)
    def recusar(*args, **kwargs):
        raise RuntimeError("Fallback Torch não permitido nesta avaliação.")
    selective_scan_cuda.fwd = contar
    vmamba.selective_scan_torch = recusar

    tabela = []
    try:
        for passo in PASSOS:
            pasta = out / f"passo_{passo:04d}"
            for sub in ("registros", "alp", "sam", "gt"):
                (pasta / sub).mkdir(parents=True, exist_ok=True)

            modelo = legado.construir_672("tiny")
            ckpt = torch.load(
                TREINO / f"checkpoint_{passo:04d}.pt",
                map_location="cpu", weights_only=True
            )
            assert ckpt["passo"] == passo and ckpt["config"] == cfg_treino
            if passo == 0:
                inicial = modelo.state_dict()
                assert inicial.keys() == ckpt["modelo"].keys()
                assert all(
                    torch.equal(inicial[k].cpu(), ckpt["modelo"][k])
                    for k in inicial
                ), "Checkpoint 0 difere da inicialização pré-treinada."
                del inicial
            modelo.load_state_dict(ckpt["modelo"], strict=True)
            del ckpt

            wrapper = ALPNetWrapper(modelo)
            protosam = ProtoSAM(
                coarse_segmentation_model=wrapper, **cfg_sam
            ).to("cuda", dtype=torch.float32)
            protosam.eval().requires_grad_(False)
            modelo.eval().requires_grad_(False)
            sup_img = suporte["image"].unsqueeze(0).cuda().float()
            sup_mask = suporte["label"].unsqueeze(0).cuda().float()
            linhas = []

            with torch.no_grad(), torch.autocast("cuda", enabled=False):
                for pos, item in enumerate(itens, 1):
                    registro = pasta / "registros" / f'{item["id"]}.json'
                    if registro.exists():
                        linha = ler(registro)
                        assert linha["item"] == item and linha["passo"] == passo
                        for relativo, digest in linha["arquivos_sha256"].items():
                            assert sha(pasta / relativo) == digest
                    else:
                        inicio = time.perf_counter()
                        amostra = ds[item["indice_dataset"]]
                        legado.conferir_amostra(amostra)
                        gt = amostra["label"].cpu().numpy() > .5
                        input_sha = hash_tensor(amostra["image"])

                        # Confere o pré-processamento anterior quando o registro está disponível.
                        antigo = REFERENCIA / "antes/registros" / f'{item["id"]}.json'
                        if antigo.is_file():
                            assert input_sha == ler(antigo)["query_tensor_sha256"], (
                                f'Pré-processamento mudou: {item["id"]}'
                            )
                        if passo != 0:
                            base = ler(out / "passo_0000/registros" / f'{item["id"]}.json')
                            assert input_sha == base["query_tensor_sha256"]
                            with Image.open(out / "passo_0000/gt" / f'{item["id"]}.png') as im:
                                assert np.array_equal(gt, np.asarray(im) > 0)

                        qry = amostra["image"].unsqueeze(0).cuda().float()
                        def entrada():
                            return ALPNetInput(
                                support_images=[sup_img], support_labels=[sup_mask],
                                query_images=qry, isval=True, val_wsize=2,
                                show_viz=False, supp_fts=None
                            )
                        chamadas["cuda"] = 0
                        logits = wrapper(entrada())
                        assert tuple(logits.shape) == (1, 2, 1024, 1024)
                        assert torch.isfinite(logits).all().item()
                        alp = (logits.argmax(1)[0] == 1).cpu().numpy()
                        del logits

                        saida, scores = protosam(
                            query_image=qry, coarse_model_input=entrada(), degrees_rotate=0
                        )
                        sam = saida.detach().cpu().numpy().squeeze()
                        assert sam.shape == (1024, 1024) and np.isin(sam, [0, 1]).all()
                        assert chamadas["cuda"] > 0
                        sam = sam.astype(bool)

                        hashes = {}
                        for tipo, mask in (("alp", alp), ("sam", sam), ("gt", gt)):
                            arquivo = pasta / tipo / f'{item["id"]}.png'
                            Image.fromarray(mask.astype(np.uint8) * 255).save(arquivo)
                            hashes[str(arquivo.relative_to(pasta))] = sha(arquivo)

                        linha = {
                            "passo": passo, "item": item,
                            "metricas": {"ALP": legado.medir(alp, gt), "SAM": legado.medir(sam, gt)},
                            "query_tensor_sha256": input_sha,
                            "arquivos_sha256": hashes,
                            "chamadas_cuda": chamadas["cuda"],
                            "scores_sam": len(scores),
                            "segundos": time.perf_counter() - inicio,
                        }
                        salvar(registro, linha)
                        del saida, scores, qry

                    linhas.append(linha)
                    print(
                        f"Passo {passo} | {pos}/145 | "
                        f"ALP={linha['metricas']['ALP']['Dice']:.4f} | "
                        f"SAM={linha['metricas']['SAM']['Dice']:.4f}", flush=True
                    )

            resumo = {"passo": passo, "n": len(linhas)}
            por_imagem = []
            for linha in linhas:
                row = {"id": linha["item"]["id"], "imagem": linha["item"]["imagem"]}
                for etapa in ("ALP", "SAM"):
                    for metrica in ("Dice", "IoU", "precisao", "recall"):
                        row[f"{etapa}_{metrica}"] = linha["metricas"][etapa][metrica]
                por_imagem.append(row)
            for etapa in ("ALP", "SAM"):
                for metrica in ("Dice", "IoU", "precisao", "recall"):
                    chave = f"{etapa}_{metrica}"
                    resumo[chave] = float(np.mean([r[chave] for r in por_imagem]))

            csv_atomico(pasta / "por_imagem.csv", por_imagem)
            salvar(pasta / "resumo.json", resumo)
            tabela.append(resumo)
            csv_atomico(out / "por_checkpoint.csv", tabela)
            print("RESUMO:", json.dumps(resumo), flush=True)

            del protosam, wrapper, modelo, sup_img, sup_mask
            gc.collect()
            torch.cuda.empty_cache()

        melhor = max(tabela, key=lambda r: (r["SAM_Dice"], -r["passo"]))
        salvar(out / "concluido.json", {
            "checkpoints": len(tabela), "queries_por_checkpoint": 145,
            "melhor_incluindo_passo0": melhor,
            "criterio": "maior Dice medio ALP+SAM; empate favorece menor passo",
            "limite": "selecao na validacao S00; nao e resultado de teste",
        })
        print("AVALIAÇÃO CONCLUÍDA.", flush=True)
    finally:
        selective_scan_cuda.fwd = original_fwd
        vmamba.selective_scan_torch = original_fallback

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    out = Path(args.out).resolve()
    try:
        main(out)
    except BaseException:
        out.mkdir(parents=True, exist_ok=True)
        (out / f"erro_{time.time_ns()}.txt").write_text(
            traceback.format_exc(), encoding="utf-8"
        )
        raise

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
SCRIPT_AVAL = ROOT / "exp/eft_tiny_512_v1/avaliar_checkpoints.py"

# Para retomar, substitua None pelo caminho da avaliação já criada.
RETOMAR_AVALIACAO = None

if "processoAVAL" in globals() and processoAVAL.poll() is None:
    raise RuntimeError("A avaliação já está executando.")

compile(SCRIPT_AVAL.read_text(encoding="utf-8"), str(SCRIPT_AVAL), "exec")

if RETOMAR_AVALIACAO is None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    AVAL512 = ROOT / "reproducao" / f"avaliacao_eft512_9checkpoints_{stamp}"
    AVAL512.mkdir(parents=True, exist_ok=False)
else:
    AVAL512 = Path(RETOMAR_AVALIACAO)
    assert (AVAL512 / "config.json").is_file()

LOG_AVAL = AVAL512 / "console.log"
with LOG_AVAL.open("ab", buffering=0) as log:
    processoAVAL = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT_AVAL), "--out", str(AVAL512)],
        cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True
    )

(AVAL512 / "processo.pid").write_text(str(processoAVAL.pid))
print("PID:", processoAVAL.pid)
print("AVALIAÇÃO:", AVAL512)
print("LOG:", LOG_AVAL)

In [ ]:
print(
    "Estado:", "executando" if processoAVAL.poll() is None
    else f"encerrado, código {processoAVAL.returncode}"
)
with LOG_AVAL.open("rb") as f:
    f.seek(0, 2)
    tamanho = f.tell()
    f.seek(max(0, tamanho - 10000))
    texto = f.read().decode("utf-8", errors="replace")
print("\n".join(texto.splitlines()[-25:]))

In [ ]:
from pathlib import Path
import csv
import json

assert "AVAL512" in globals(), "Informe o caminho da pasta de avaliação."
pasta = Path(AVAL512)

with (pasta / "por_checkpoint.csv").open(encoding="utf-8", newline="") as f:
    linhas = list(csv.DictReader(f))

print("Checkpoint | Dice ALP | Dice ALP+SAM | Δ SAM vs passo 0 (p.p.)")
baseline = next(float(r["SAM_Dice"]) for r in linhas if int(r["passo"]) == 0)

for r in sorted(linhas, key=lambda x: int(x["passo"])):
    alp = float(r["ALP_Dice"])
    sam = float(r["SAM_Dice"])
    print(
        f'{int(r["passo"]):>10} | {alp:.6f} | {sam:.6f}'
        f'     | {(sam - baseline) * 100:+.2f}'
    )

print("\n=== SELEÇÃO REGISTRADA ===")
print(json.dumps(
    json.loads((pasta / "concluido.json").read_text()),
    indent=2, ensure_ascii=False
))
print("\nPasta:", pasta)

In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

script = ROOT / "exp/eft_validacao_polipos_v1/validar_multissuporte_v1.py"
plano = ROOT / (
    "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
)

assert script.is_file(), script
assert plano.is_file(), plano

print("=== AVALIADOR MULTISSUPORTE EXISTENTE ===")
print(script.read_text(encoding="utf-8"))

def resumir(obj):
    if isinstance(obj, dict):
        return {k: resumir(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return {
            "quantidade": len(obj),
            "primeiro": resumir(obj[0]) if obj else None,
        }
    return obj

print("\n=== ESTRUTURA DO PLANO FIXO ===")
dados = json.loads(plano.read_text(encoding="utf-8"))
print(json.dumps(resumir(dados), indent=2, ensure_ascii=False))

In [ ]:
from pathlib import Path
import hashlib

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
AVAL_S00 = ROOT / (
    "reproducao/avaliacao_eft512_9checkpoints_20261017T122928229285Z"
)
PLANO_SUPORTES = ROOT / (
    "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
)

import json
origem = CODIGO / "avaliar_checkpoints.py"
config_s00 = json.loads((AVAL_S00 / "config.json").read_text())
digest = hashlib.sha256(origem.read_bytes()).hexdigest()
assert digest == config_s00["fontes_sha256"][str(origem)], (
    "O avaliador mudou desde a avaliação concluída."
)

texto = origem.read_text(encoding="utf-8")

def substituir_unico(antigo, novo):
    global texto
    assert texto.count(antigo) == 1, f"Trecho inesperado: {antigo[:100]}"
    texto = texto.replace(antigo, novo)

substituir_unico(
    "PASSOS = [0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000]",
    "PASSOS = [0, 200]\n"
    f"PLANO_SUPORTES = Path({str(PLANO_SUPORTES)!r})"
)

substituir_unico("def main(out):", "def main(out, suporte_id):")

substituir_unico(
    '    ref = ler(REFERENCIA / "config.json")',
    '''    ref = ler(REFERENCIA / "config.json")
    plano_suportes = ler(PLANO_SUPORTES)
    assert plano_suportes["n_suportes"] == 20
    assert plano_suportes["n_queries"] == 145
    assert plano_suportes["resolucao_encoder"] == 672
    assert plano_suportes["canvas_sam"] == 1024

    selecionados = [
        s for s in plano_suportes["suportes"] if s["id"] == suporte_id
    ]
    assert len(selecionados) == 1
    escolhido = selecionados[0]
    historico_suporte = PLANO_SUPORTES.parent / suporte_id / "config.json"
    ref_sup = ler(historico_suporte)

    # Identidade do suporte fixada pelo plano; nenhum novo sorteio.
    meta = dict(escolhido["meta"])
    for chave in ("imagem", "mascara"):
        assert atual(meta[chave]) == atual(ref_sup["suporte"][chave])
        assert escolhido[chave + "_sha256"] == ref_sup["suporte"][chave + "_sha256"]
        meta[chave + "_sha256"] = escolhido[chave + "_sha256"]

    assert ref_sup["checkpoint_original_sha256"] == ref["checkpoint_original_sha256"]
    assert ref_sup["checkpoint_sam_sha256"] == ref["checkpoint_sam_sha256"]
    assert {r["id"] for r in ref_sup["episodios"]} == {
        r["id"] for r in ref["episodios"]
    }

    ref["suporte"] = meta
    ref["suporte_tensor_sha256"] = ref_sup["suporte_tensor_sha256"]
'''
)

substituir_unico(
    '    assert int((suporte["label"] > .5).sum()) == 51940',
    '''    area = int((suporte["label"] > .5).sum())
    assert 0 < area < 1024 * 1024
    if suporte_id == "S00":
        assert area == 51940
    else:
        imagens_treino = {str(atual(r["imagem"])) for r in planos["treino"]}
        assert sup_meta["imagem"] in imagens_treino
'''
)

substituir_unico(
    '        "finalidade": "validacao anotada; selecao exploratoria de checkpoint",',
    '''        "finalidade": "multissuporte fixo; inicial versus EFT512 passo200",
        "suporte_id": suporte_id,
        "plano_suportes_sha256": sha(PLANO_SUPORTES),
        "historico_suporte_config_sha256": sha(historico_suporte),'''
)

substituir_unico(
    "        main(out)",
    '''        import fcntl
        out.mkdir(parents=True, exist_ok=True)
        trava = (out / "execucao.lock").open("a")
        fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)

        plano_fixo = ler(PLANO_SUPORTES)
        suportes = plano_fixo["suportes"]
        assert len(suportes) == 20
        assert [s["id"] for s in suportes] == [f"S{i:02d}" for i in range(20)]
        assert len({str(atual(s["meta"]["imagem"])) for s in suportes}) == 20

        config_lote = {
            "plano_original": str(PLANO_SUPORTES),
            "plano_sha256": sha(PLANO_SUPORTES),
            "avaliador_sha256": sha(__file__),
            "treino": str(TREINO),
            "passos": PASSOS,
            "suportes": [s["id"] for s in suportes],
            "observacao": "S00 usado na selecao; resumo principal S01-S19",
        }
        if (out / "config_lote.json").exists():
            assert ler(out / "config_lote.json") == config_lote
        else:
            salvar(out / "config_lote.json", config_lote)
            shutil.copy2(PLANO_SUPORTES, out / "plano_original.json")
            shutil.copy2(__file__, out / "avaliador_usado.py")

        tabela_suportes = []
        for s in suportes:
            sid = s["id"]
            print(f"\\\\n=== SUPORTE {sid} ===", flush=True)
            main(out / sid, sid)

            inicial = ler(out / sid / "passo_0000/resumo.json")
            eft = ler(out / sid / "passo_0200/resumo.json")
            row = {"suporte": sid, "nome": s["meta"]["nome"]}
            for etapa in ("ALP", "SAM"):
                for metrica in ("Dice", "IoU", "precisao", "recall"):
                    chave = f"{etapa}_{metrica}"
                    row[f"inicial_{chave}"] = inicial[chave]
                    row[f"eft512_200_{chave}"] = eft[chave]
                row[f"delta_{etapa}_Dice"] = eft[f"{etapa}_Dice"] - inicial[f"{etapa}_Dice"]
            tabela_suportes.append(row)
            csv_atomico(out / "por_suporte.csv", tabela_suportes)
            print("SUPORTE CONCLUIDO:", json.dumps(row), flush=True)

        sorteados = [r for r in tabela_suportes if r["suporte"] != "S00"]
        resumo_multi = {
            "n_suportes_sorteados": len(sorteados),
            "n_queries_por_suporte": 145,
            "S00": tabela_suportes[0],
            "S01_S19": {},
            "limite": "validacao exploratoria; nao mede variacao entre treinamentos",
        }
        for etapa in ("ALP", "SAM"):
            dados = {}
            for condicao in ("inicial", "eft512_200"):
                valores = np.array([r[f"{condicao}_{etapa}_Dice"] for r in sorteados])
                dados[condicao] = {
                    "media": float(valores.mean()),
                    "desvio_padrao_entre_suportes_ddof1": float(valores.std(ddof=1)),
                    "minimo": float(valores.min()),
                    "maximo": float(valores.max()),
                }
            delta = np.array([r[f"delta_{etapa}_Dice"] for r in sorteados])
            dados["delta_medio_pp"] = float(delta.mean() * 100)
            dados["suportes_com_melhora"] = int((delta > 1e-8).sum())
            dados["suportes_com_piora"] = int((delta < -1e-8).sum())
            resumo_multi["S01_S19"][etapa] = dados

        salvar(out / "resumo_multissuporte.json", resumo_multi)
        print(json.dumps(resumo_multi, indent=2, ensure_ascii=False), flush=True)
        print("MULTISSUPORTE CONCLUIDO: 20 suportes.", flush=True)
'''
)

SCRIPT_MULTI = CODIGO / "avaliar_multissuporte512.py"
compile(texto, str(SCRIPT_MULTI), "exec")
if SCRIPT_MULTI.exists():
    assert SCRIPT_MULTI.read_text(encoding="utf-8") == texto, (
        "Já existe outra versão do avaliador multissuporte."
    )
else:
    with SCRIPT_MULTI.open("x", encoding="utf-8") as f:
        f.write(texto)

print("Avaliador preparado:", SCRIPT_MULTI)
print("Suportes fixos: S00–S19. Checkpoints: 0 e 200.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
SCRIPT_MULTI = ROOT / "exp/eft_tiny_512_v1/avaliar_multissuporte512.py"

# Em caso de interrupção, coloque aqui a pasta impressa abaixo.
RETOMAR_MULTI = None

if "processoMULTI" in globals() and processoMULTI.poll() is None:
    raise RuntimeError("A avaliação multissuporte já está executando.")

if RETOMAR_MULTI is None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    MULTI512 = ROOT / "reproducao" / f"multissuporte_eft512_200_{stamp}"
    MULTI512.mkdir(parents=True, exist_ok=False)
else:
    MULTI512 = Path(RETOMAR_MULTI)
    assert (MULTI512 / "config_lote.json").is_file()

LOG_MULTI = MULTI512 / "console.log"
with LOG_MULTI.open("ab", buffering=0) as log:
    processoMULTI = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT_MULTI), "--out", str(MULTI512)],
        cwd=ROOT, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True
    )

(MULTI512 / "processo.pid").write_text(str(processoMULTI.pid))
print("PID:", processoMULTI.pid)
print("MULTISSUPORTE:", MULTI512)
print("LOG:", LOG_MULTI)

In [ ]:
print(
    "Estado:", "executando" if processoMULTI.poll() is None
    else f"encerrado, código {processoMULTI.returncode}"
)
with LOG_MULTI.open("rb") as f:
    f.seek(0, 2)
    tamanho = f.tell()
    f.seek(max(0, tamanho - 10000))
    texto = f.read().decode("utf-8", errors="replace")
print("\n".join(texto.splitlines()[-25:]))

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

# Implementação exata da extração dos estágios.
arquivo = ROOT / "models/vmamba.py"
texto = arquivo.read_text(encoding="utf-8")
arvore = ast.parse(texto)

print("=== Backbone_VSSM ===")
for no in arvore.body:
    if isinstance(no, ast.ClassDef) and no.name == "Backbone_VSSM":
        print(ast.get_source_segment(texto, no))
        break
else:
    raise RuntimeError("Classe Backbone_VSSM não encontrada.")

# Avaliador que já funcionou nesta execução.
arquivo = ROOT / "exp/eft_tiny_512_v1/avaliar_checkpoints.py"
assert arquivo.is_file(), f"Arquivo não encontrado: {arquivo}"

print("\n=== AVALIADOR ATUAL ===")
print(arquivo.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import hashlib
import json
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
BASE = ROOT / "exp/eft_tiny_512_v1/avaliar_checkpoints.py"
SCRIPT_PENULTIMA = BASE.with_name("avaliar_penultima512_sem_norm.py")

AVALIACAO_ULTIMA = ROOT / (
    "reproducao/avaliacao_eft512_9checkpoints_20261017T122928229285Z"
)

def sha_arquivo(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

# Confere que estamos adaptando o avaliador da execução anterior.
config_anterior = json.loads(
    (AVALIACAO_ULTIMA / "config.json").read_text(encoding="utf-8")
)
assert config_anterior["fontes_sha256"][str(BASE)] == sha_arquivo(BASE)

codigo = BASE.read_text(encoding="utf-8")

def trocar_unico(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

trocar_unico(
    "PASSOS = [0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000]",
    "PASSOS = [0, 200]",
)

adaptacao = textwrap.dedent("""
def instalar_penultima(modelo):
    from types import MethodType
    import torch.nn.functional as F
    from models.grid_proto_fewshot import MultiProtoAsConv

    encoder = modelo.encoder
    assert encoder.channel_first
    assert len(encoder.layers) == 4
    assert int(encoder.dims[2]) == 384

    # Não descartar silenciosamente parâmetros aprendidos do ALP.
    assert len(modelo.cls_unit.state_dict()) == 0, (
        "ALP contém estado; é necessário conferir antes da adaptação."
    )
    cls_nova = MultiProtoAsConv(
        proto_grid=[8, 8],
        feature_hw=[32, 32],
        embed_dim=384,
    ).to("cuda", dtype=torch.float32)
    assert len(cls_nova.state_dict()) == 0
    modelo.cls_unit = cls_nova

    def get_features_penultima(self, imgs_concat):
        x = F.interpolate(
            imgs_concat, size=(672, 672),
            mode="bilinear", align_corners=False,
        )
        x = self.encoder.patch_embed(x)

        # Mesmo percurso do Backbone_VSSM até a saída do estágio 2.
        for i in range(3):
            camada = self.encoder.layers[i]
            x = camada.blocks(x)
            if i < 2:
                x = camada.downsample(x)

        assert tuple(x.shape[1:]) == (384, 42, 42), tuple(x.shape)
        assert torch.isfinite(x).all().item()

        # Nenhuma outnorm adicional. Norms internas preservadas.
        return F.interpolate(
            x.contiguous(), size=(32, 32),
            mode="bilinear", align_corners=False,
        )

    modelo.get_features = MethodType(get_features_penultima, modelo)
    print(
        "Extração: estágio 2 antes do downsampling | "
        "384x42x42 -> 384x32x32 | sem outnorm",
        flush=True,
    )

""")

trocar_unico("def main(out):", adaptacao + "\ndef main(out):")

trocar_unico(
    '            modelo.load_state_dict(ckpt["modelo"], strict=True)\n'
    '            del ckpt',
    '            modelo.load_state_dict(ckpt["modelo"], strict=True)\n'
    '            del ckpt\n'
    '            instalar_penultima(modelo)',
)

trocar_unico(
    '"finalidade": "validacao anotada; selecao exploratoria de checkpoint",',
    '"finalidade": "avaliacao penultima sem outnorm; validacao S00",\n'
    '        "extracao": {\n'
    '            "estagio_indice": 2,\n'
    '            "ponto": "apos blocks, antes de downsample",\n'
    '            "canais": 384,\n'
    '            "grade_nativa": [42, 42],\n'
    '            "normalizacao_saida": "nenhuma",\n'
    '            "normalizacoes_internas": "preservadas",\n'
    '            "novo_treinamento": False,\n'
    '        },',
)

trocar_unico(
    '"limite": "selecao na validacao S00; nao e resultado de teste",',
    '"limite": "validacao S00; camada e norm de saida diferem '
    'da referencia; nao e resultado de teste",',
)

compile(codigo, str(SCRIPT_PENULTIMA), "exec")

if SCRIPT_PENULTIMA.exists():
    assert SCRIPT_PENULTIMA.read_text(encoding="utf-8") == codigo, (
        "Já existe um script diferente nesse caminho."
    )
else:
    with SCRIPT_PENULTIMA.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Script:", SCRIPT_PENULTIMA)
print("SHA256:", sha_arquivo(SCRIPT_PENULTIMA))
print("Avaliador original preservado. Nenhuma avaliação iniciada ainda.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoPENULTIMA")
assert anterior is None or anterior.poll() is not None, (
    "A avaliação da penúltima já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_PENULTIMA = ROOT / "reproducao" / (
    f"avaliacao_eft512_penultima_sem_norm_{stamp}"
)
RUN_PENULTIMA.mkdir(parents=True, exist_ok=False)
LOG_PENULTIMA = RUN_PENULTIMA / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_PENULTIMA),
    "--out", str(RUN_PENULTIMA),
]
(RUN_PENULTIMA / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_PENULTIMA.open("w", encoding="utf-8") as log:
    processoPENULTIMA = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_PENULTIMA / "pid.txt").write_text(
    str(processoPENULTIMA.pid), encoding="utf-8"
)

print("PID:", processoPENULTIMA.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_PENULTIMA)
print("LOG:", LOG_PENULTIMA)

In [ ]:
estado = processoPENULTIMA.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)

linhas = LOG_PENULTIMA.read_text(
    encoding="utf-8", errors="replace"
).splitlines()
print("\n".join(linhas[-20:]))

if (RUN_PENULTIMA / "concluido.json").is_file():
    print("\n=== COMPARAÇÃO S00 — MESMAS 145 QUERIES ===")
    print("Passo | Extração                     | Dice ALP | Dice ALP+SAM")

    for passo in (0, 200):
        for nome, pasta in (
            ("Última + norm aprendida", AVALIACAO_ULTIMA),
            ("Penúltima sem outnorm", RUN_PENULTIMA),
        ):
            resumo = json.loads(
                (pasta / f"passo_{passo:04d}" / "resumo.json")
                .read_text(encoding="utf-8")
            )
            print(
                f"{passo:5d} | {nome:28s} | "
                f"{resumo['ALP_Dice']:.6f} | "
                f"{resumo['SAM_Dice']:.6f}"
            )

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
ORIGEM = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima512_sem_norm.py"
SCRIPT_MARCOS = ORIGEM.with_name("avaliar_penultima512_marcos.py")

# Pasta da avaliação da penúltima que acabou de concluir.
ANTERIOR = Path(RUN_PENULTIMA)
assert (ANTERIOR / "concluido.json").is_file()
assert (ANTERIOR / "passo_0000/resumo.json").is_file()

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg_anterior = ler_json(ANTERIOR / "config.json")
assert cfg_anterior["fontes_sha256"][str(ORIGEM)] == sha_local(ORIGEM)

codigo = ORIGEM.read_text(encoding="utf-8")
assert codigo.count("PASSOS = [0, 200]") == 1
codigo = codigo.replace(
    "PASSOS = [0, 200]",
    "PASSOS = [0, 2000, 3000, 5000]",
    1,
)

# Antes de reutilizar registros, verifica que apenas o script e
# a lista de checkpoints mudaram em relação à avaliação anterior.
marcador = (
    '    out.mkdir(parents=True, exist_ok=True)\n'
    '    if (out / "config.json").exists():\n'
)
assert codigo.count(marcador) == 1

verificacao = '''
    config_origem = ler(out / "config_origem.json")
    permitidas = {"passos", "checkpoints_sha256", "fontes_sha256"}
    assert {
        k: v for k, v in config.items() if k not in permitidas
    } == {
        k: v for k, v in config_origem.items() if k not in permitidas
    }, "Protocolo diferente da avaliação anterior."

    assert config["checkpoints_sha256"]["0"] == (
        config_origem["checkpoints_sha256"]["0"]
    ), "Checkpoint inicial mudou."

    fontes_atuais = dict(config["fontes_sha256"])
    fontes_anteriores = dict(config_origem["fontes_sha256"])
    fontes_atuais.pop(str(Path(__file__)))
    fontes_anteriores.pop(str(
        ROOT / "exp/eft_tiny_512_v1/avaliar_penultima512_sem_norm.py"
    ))
    assert fontes_atuais == fontes_anteriores, "Dependências mudaram."

'''
codigo = codigo.replace(marcador, verificacao + marcador, 1)
compile(codigo, str(SCRIPT_MARCOS), "exec")

if SCRIPT_MARCOS.exists():
    assert SCRIPT_MARCOS.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_MARCOS.open("x", encoding="utf-8") as f:
        f.write(codigo)

processo_anterior = globals().get("processoPEN_MARCOS")
assert processo_anterior is None or processo_anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_PEN_MARCOS = ROOT / "reproducao" / (
    f"avaliacao_penultima512_marcos_{stamp}"
)
RUN_PEN_MARCOS.mkdir(parents=True, exist_ok=False)

shutil.copy2(
    ANTERIOR / "config.json",
    RUN_PEN_MARCOS / "config_origem.json",
)
shutil.copytree(
    ANTERIOR / "passo_0000",
    RUN_PEN_MARCOS / "passo_0000",
)
(RUN_PEN_MARCOS / "proveniencia.json").write_text(
    json.dumps({
        "passo0_reutilizado_de": str(ANTERIOR),
        "config_origem_sha256": sha_local(ANTERIOR / "config.json"),
        "novos_passos": [2000, 3000, 5000],
    }, indent=2),
    encoding="utf-8",
)

LOG_PEN_MARCOS = RUN_PEN_MARCOS / "console.log"
comando = [
    sys.executable, "-u", str(SCRIPT_MARCOS),
    "--out", str(RUN_PEN_MARCOS),
]
(RUN_PEN_MARCOS / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_PEN_MARCOS.open("w", encoding="utf-8") as log:
    processoPEN_MARCOS = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_PEN_MARCOS / "pid.txt").write_text(
    str(processoPEN_MARCOS.pid), encoding="utf-8"
)

print("PID:", processoPEN_MARCOS.pid)
print("EXECUÇÃO:", RUN_PEN_MARCOS)
print("435 novas avaliações: 145 queries × 3 checkpoints.")
print("O passo 0 aparecerá no log, mas suas predições serão reutilizadas.")

In [ ]:
estado = processoPEN_MARCOS.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_PEN_MARCOS.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

if (RUN_PEN_MARCOS / "concluido.json").is_file():
    ultima = ROOT / (
        "reproducao/avaliacao_eft512_9checkpoints_20261017T122928229285Z"
    )

    print("\n=== S00 — DICE NAS MESMAS 145 QUERIES ===")
    print("Passo | Última ALP | Penúltima ALP | Última+SAM | Penúltima+SAM")

    for passo in (0, 200, 2000, 3000, 5000):
        pasta_pen = ANTERIOR if passo in (0, 200) else RUN_PEN_MARCOS
        a = ler_json(ultima / f"passo_{passo:04d}/resumo.json")
        b = ler_json(pasta_pen / f"passo_{passo:04d}/resumo.json")
        print(
            f"{passo:5d} | {a['ALP_Dice']:.6f}   | "
            f"{b['ALP_Dice']:.6f}      | "
            f"{a['SAM_Dice']:.6f}   | {b['SAM_Dice']:.6f}"
        )

In [ ]:
from pathlib import Path
import ast
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
BASE_PEN = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima512_sem_norm.py"
SCRIPT_MULTI_PEN = BASE_PEN.with_name("avaliar_multissuporte_penultima2000.py")
PLANO_SUPORTES = ROOT / (
    "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
)

assert BASE_PEN.is_file()
assert PLANO_SUPORTES.is_file()

codigo = BASE_PEN.read_text(encoding="utf-8")

def substituir_unico(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, f"Trecho ausente ou repetido:\n{antigo}"
    codigo = codigo.replace(antigo, novo, 1)

substituir_unico("PASSOS = [0, 200]", "PASSOS = [2000]")
substituir_unico("def main(out):", "def main(out, suporte_id):")

# Mantém a referência original de código e queries.
# Substitui apenas o suporte pelo mesmo usado no multissuporte anterior.
substituir_unico(
    '    ref = ler(REFERENCIA / "config.json")',
    textwrap.dedent('''
        ref = ler(REFERENCIA / "config.json")
        plano_path = ROOT / (
            "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
        )
        plano_sup = ler(plano_path)
        candidatos = [
            s for s in plano_sup["suportes"] if s["id"] == suporte_id
        ]
        assert len(candidatos) == 1
        escolhido = candidatos[0]

        referencia_sup = plano_path.parent / suporte_id / "config.json"
        ref_sup = ler(referencia_sup)
        assert {r["id"] for r in ref_sup["episodios"]} == {
            r["id"] for r in ref["episodios"]
        }
        assert ref_sup["checkpoint_sam_sha256"] == ref["checkpoint_sam_sha256"]

        novo_suporte = dict(escolhido["meta"])
        for chave in ("imagem", "mascara"):
            assert atual(novo_suporte[chave]) == atual(ref_sup["suporte"][chave])
            digest = escolhido[chave + "_sha256"]
            assert digest == ref_sup["suporte"][chave + "_sha256"]
            novo_suporte[chave + "_sha256"] = digest

        ref["suporte"] = novo_suporte
        ref["suporte_tensor_sha256"] = ref_sup["suporte_tensor_sha256"]
    ''').strip("\n").replace("\n", "\n    ").join(["    ", ""])
)

substituir_unico(
    '    assert int((suporte["label"] > .5).sum()) == 51940',
    '    assert 0 < int((suporte["label"] > .5).sum()) < 1024 * 1024\n'
    '    assert sup_meta["imagem"] in {\n'
    '        str(atual(r["imagem"])) for r in planos["treino"]\n'
    '    }, "Suporte sorteado fora do conjunto de treino registrado."'
)

substituir_unico(
    '"finalidade": "avaliacao penultima sem outnorm; validacao S00",',
    '"finalidade": "penultima sem outnorm; checkpoint 2000; S01-S19",\n'
    '        "suporte_id": suporte_id,\n'
    '        "plano_suportes_sha256": sha(plano_path),\n'
    '        "referencia_suporte_sha256": sha(referencia_sup),'
)

# Sem executar o passo 0 novamente.
# A conferência com a referência histórica do pré-processamento permanece.
arvore = ast.parse(codigo)
alvos = [
    no for no in ast.walk(arvore)
    if isinstance(no, ast.If)
    and ast.unparse(no.test) == "passo != 0"
]
assert len(alvos) == 1
no = alvos[0]
linhas = codigo.splitlines(keepends=True)
del linhas[no.lineno - 1:no.end_lineno]
codigo = "".join(linhas)

substituir_unico(
    '"limite": "validacao S00; camada e norm de saida diferem '
    'da referencia; nao e resultado de teste",',
    '"limite": "validacao exploratoria por suporte; '
    'nao e resultado de teste",'
)

# Substitui apenas a entrada de execução por um lote sequencial.
marcador = 'if __name__ == "__main__":'
assert codigo.count(marcador) == 1
codigo = codigo.split(marcador)[0] + textwrap.dedent('''
if __name__ == "__main__":
    import fcntl

    parser = argparse.ArgumentParser()
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    out = Path(args.out).resolve()
    out.mkdir(parents=True, exist_ok=True)

    trava = (out / "execucao.lock").open("w")
    fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)

    try:
        plano_path = ROOT / (
            "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
        )
        plano = ler(plano_path)
        assert [s["id"] for s in plano["suportes"]] == [
            f"S{i:02d}" for i in range(20)
        ]
        assert len({
            str(atual(s["meta"]["imagem"])) for s in plano["suportes"]
        }) == 20

        config_lote = {
            "suportes": [f"S{i:02d}" for i in range(1, 20)],
            "checkpoint": 2000,
            "script_sha256": sha(__file__),
            "plano_sha256": sha(plano_path),
            "treino": str(TREINO),
            "extracao": "penultima sem outnorm; 672; grade ALP 32",
        }
        if (out / "config_lote.json").exists():
            assert ler(out / "config_lote.json") == config_lote
        else:
            salvar(out / "config_lote.json", config_lote)
            shutil.copy2(plano_path, out / "plano_suportes.json")
            shutil.copy2(__file__, out / Path(__file__).name)

        resultados = []
        for sid in config_lote["suportes"]:
            print(f"\\n=== SUPORTE {sid} | PENULTIMA | PASSO 2000 ===", flush=True)
            main(out / sid, sid)
            resumo = ler(out / sid / "passo_2000/resumo.json")
            assert resumo["n"] == 145
            resultados.append({"suporte": sid, **resumo})
            csv_atomico(out / "por_suporte.csv", resultados)

        agregado = {
            "n_suportes": len(resultados),
            "queries_por_suporte": 145,
            "checkpoint": 2000,
            "limite": "validacao S01-S19; variacao entre suportes, nao entre treinos",
        }
        for etapa in ("ALP", "SAM"):
            valores = np.array([r[f"{etapa}_Dice"] for r in resultados])
            agregado[etapa] = {
                "media": float(valores.mean()),
                "desvio_padrao_entre_suportes_ddof1": float(valores.std(ddof=1)),
                "minimo": float(valores.min()),
                "maximo": float(valores.max()),
            }

        salvar(out / "resumo_multissuporte.json", agregado)
        salvar(out / "concluido.json", agregado)
        print(json.dumps(agregado, indent=2, ensure_ascii=False), flush=True)
        print("MULTISSUPORTE PENULTIMA CONCLUIDO.", flush=True)

    except BaseException:
        (out / f"erro_{time.time_ns()}.txt").write_text(
            traceback.format_exc(), encoding="utf-8"
        )
        raise
''')

compile(codigo, str(SCRIPT_MULTI_PEN), "exec")

if SCRIPT_MULTI_PEN.exists():
    assert SCRIPT_MULTI_PEN.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_MULTI_PEN.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Script criado:", SCRIPT_MULTI_PEN)
print("Avaliadores anteriores preservados.")

In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoMULTI_PEN")
assert anterior is None or anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_MULTI_PEN = ROOT / "reproducao" / (
    f"multissuporte_penultima512_2000_{stamp}"
)
RUN_MULTI_PEN.mkdir(parents=True, exist_ok=False)
LOG_MULTI_PEN = RUN_MULTI_PEN / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_MULTI_PEN),
    "--out", str(RUN_MULTI_PEN),
]
(RUN_MULTI_PEN / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_MULTI_PEN.open("w", encoding="utf-8") as log:
    processoMULTI_PEN = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_MULTI_PEN / "pid.txt").write_text(
    str(processoMULTI_PEN.pid), encoding="utf-8"
)

print("PID:", processoMULTI_PEN.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_MULTI_PEN)
print("LOG:", LOG_MULTI_PEN)

In [ ]:
estado = processoMULTI_PEN.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_MULTI_PEN.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

## EFT512 — treinamento direto pela penúltima camada

In [ ]:
from pathlib import Path

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO_ANTERIOR = ROOT / (
    "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"
)

arquivos = [
    ROOT / "exp/eft_tiny_512_v1/treinar_5000_cuda.py",
    ROOT / "exp/eft_tiny_512_v1/eft_alinhamento_seguro.py",
    TREINO_ANTERIOR / "config.json",
]

for arquivo in arquivos:
    assert arquivo.is_file(), f"Arquivo não encontrado: {arquivo}"
    print(f"\n=== {arquivo.relative_to(ROOT)} ===\n")
    print(arquivo.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import json
import hashlib
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
TREINO_ANTERIOR = ROOT / (
    "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"
)

BASE = CODIGO / "treinar_5000_cuda.py"
MODELO_PEN = CODIGO / "eft_modelo_penultima_512.py"
SCRIPT_TREINO_PEN = CODIGO / "treinar_penultima_2000_cuda.py"

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg_anterior = json.loads(
    (TREINO_ANTERIOR / "config.json").read_text(encoding="utf-8")
)

# Confere todos os arquivos usados pelo treinamento anterior.
for caminho, digest in cfg_anterior["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

modelo_codigo = textwrap.dedent('''
from pathlib import Path
from types import MethodType

import torch
import torch.nn.functional as F

from eft_modelo_512 import construir_modelo as construir_original
from models.grid_proto_fewshot import MultiProtoAsConv


def construir_modelo(variante, repo):
    assert variante == "tiny"
    modelo = construir_original(variante, repo=Path(repo))
    encoder = modelo.encoder

    assert encoder.channel_first
    assert len(encoder.layers) == 4
    assert int(encoder.dims[2]) == 384
    assert len(modelo.cls_unit.state_dict()) == 0

    # Preserva o checkpoint completo, mas congela módulos não utilizados.
    encoder.layers[2].downsample.requires_grad_(False)
    encoder.layers[3].requires_grad_(False)
    encoder.outnorm3.requires_grad_(False)

    modelo.cls_unit = MultiProtoAsConv(
        proto_grid=[8, 8],
        feature_hw=[32, 32],
        embed_dim=384,
    ).to("cuda", dtype=torch.float32)
    assert len(modelo.cls_unit.state_dict()) == 0

    def get_features_penultima(self, imgs_concat):
        x = F.interpolate(
            imgs_concat, size=(512, 512),
            mode="bilinear", align_corners=False,
        )
        x = self.encoder.patch_embed(x)

        for i in range(3):
            camada = self.encoder.layers[i]
            x = camada.blocks(x)
            if i < 2:
                x = camada.downsample(x)

        assert tuple(x.shape[1:]) == (384, 32, 32), tuple(x.shape)
        return x.contiguous()

    modelo.get_features = MethodType(get_features_penultima, modelo)
    modelo.train()
    modelo.zero_grad(set_to_none=True)

    print(
        "EFT direto: penultima, 384x32x32, sem outnorm; "
        "patch_embed e estagios 0-2 treinaveis.",
        flush=True,
    )
    print(
        "Parametros treinaveis:",
        sum(p.numel() for p in modelo.parameters() if p.requires_grad),
        flush=True,
    )
    return modelo
''')

treinador = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global treinador
    assert treinador.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    treinador = treinador.replace(antigo, novo, 1)

trocar(
    "from eft_modelo_512 import construir_modelo",
    "from eft_modelo_penultima_512 import construir_modelo",
)
trocar(
    "PRESERVAR = {0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000}",
    "PRESERVAR = {0, 50, 100, 200, 500, 1000, 1500, 2000}",
)
trocar(
    'print("Plano conferido: mesmos 5000 episódios do histórico.", flush=True)',
    'print("Plano conferido: usando os primeiros 2000 episódios '
    'do mesmo histórico.", flush=True)',
)
trocar(
    'return {"treino": plano, "validacao": novos["validacao"]}',
    'return {"treino": plano[:2000], "validacao": novos["validacao"]}',
)
trocar(
    'Path(__file__), CODIGO / "eft_modelo_512.py",',
    'Path(__file__), CODIGO / "eft_modelo_512.py",\n'
    '        CODIGO / "eft_modelo_penultima_512.py",',
)
trocar(
    '"formato": 1, "total_passos": 5000, "semente": 1234,',
    '"formato": 1, "total_passos": 2000, "semente": 1234,',
)
trocar(
    '"batch_episodico": 1, "adaptacao": "encoder completo, features da ultima camada",',
    '"batch_episodico": 1,\n'
    '        "adaptacao": "EFT direto pela penultima; patch_embed e estagios 0-2",\n'
    '        "extracao": {\n'
    '            "estagio_indice": 2,\n'
    '            "ponto": "apos blocks, antes de downsample",\n'
    '            "canais": 384,\n'
    '            "grade_nativa_treino": [32, 32],\n'
    '            "normalizacao_saida": "nenhuma",\n'
    '            "normalizacoes_internas": "preservadas e treinaveis",\n'
    '            "modulos_congelados": [\n'
    '                "encoder.layers.2.downsample",\n'
    '                "encoder.layers.3", "encoder.outnorm3"\n'
    '            ],\n'
    '        },',
)
trocar(
    'modelo.parameters(), lr=1e-5, betas=(0.9, 0.999),',
    '[p for p in modelo.parameters() if p.requires_grad],\n'
    '        lr=1e-5, betas=(0.9, 0.999),',
)
trocar("assert 0 <= passo <= 5000", "assert 0 <= passo <= 2000")
trocar("range(passo + 1, 5001)", "range(passo + 1, 2001)")
trocar("media * (5000 - n) / 3600", "media * (2000 - n) / 3600")
trocar('f"{n}/5000 | loss=', 'f"{n}/2000 | loss=')
trocar('"passos": 5000,', '"passos": 2000,')
trocar(
    'print("TREINAMENTO CONCLUÍDO: 5000 passos.", flush=True)',
    'print("TREINAMENTO CONCLUÍDO: 2000 passos pela penultima.", flush=True)',
)

# Confere o fluxo dos gradientes em cada atualização.
trocar(
    "            loss.backward()\n",
    "            loss.backward()\n"
    "            assert all(\n"
    "                p.grad is None for p in modelo.parameters()\n"
    "                if not p.requires_grad\n"
    "            ), 'Modulo congelado recebeu gradiente.'\n"
    "            assert any(\n"
    "                p.grad is not None\n"
    "                for p in modelo.encoder.layers[2].blocks.parameters()\n"
    "            ), 'Penultima sem gradientes.'\n",
)

# Falha explicitamente se houver tentativa de usar o backend lento.
trocar(
    "    assert torch.cuda.is_available()\n",
    "    assert torch.cuda.is_available()\n"
    "    def recusar_fallback(*args, **kwargs):\n"
    "        raise RuntimeError('Fallback Torch proibido neste treino.')\n"
    "    vmamba.selective_scan_torch = recusar_fallback\n",
)

# Evita duas execuções simultâneas escrevendo na mesma pasta.
trocar(
    "    try:\n        main(pasta)\n",
    "    import fcntl\n"
    "    pasta.mkdir(parents=True, exist_ok=True)\n"
    "    trava = (pasta / 'execucao.lock').open('w')\n"
    "    fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)\n"
    "    try:\n        main(pasta)\n",
)

# Verifica sintaxe antes de gravar.
for arquivo, conteudo in (
    (MODELO_PEN, modelo_codigo),
    (SCRIPT_TREINO_PEN, treinador),
):
    compile(conteudo, str(arquivo), "exec")

for arquivo, conteudo in (
    (MODELO_PEN, modelo_codigo),
    (SCRIPT_TREINO_PEN, treinador),
):
    if arquivo.exists():
        assert arquivo.read_text(encoding="utf-8") == conteudo, (
            f"Já existe uma versão diferente: {arquivo}"
        )
    else:
        with arquivo.open("x", encoding="utf-8") as f:
            f.write(conteudo)
    print("Arquivo:", arquivo)
    print("SHA256:", sha_local(arquivo))

print("\nPreparação concluída. Treinamento ainda não iniciado.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoTREINO_PEN")
assert anterior is None or anterior.poll() is not None, (
    "O treino da penúltima já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_TREINO_PEN = ROOT / "reproducao" / (
    f"eft_penultima512_sem_norm_2000_{stamp}"
)
RUN_TREINO_PEN.mkdir(parents=True, exist_ok=False)
LOG_TREINO_PEN = RUN_TREINO_PEN / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_TREINO_PEN),
    "--run", str(RUN_TREINO_PEN),
]
(RUN_TREINO_PEN / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_TREINO_PEN.open("w", encoding="utf-8") as log:
    processoTREINO_PEN = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_TREINO_PEN / "pid.txt").write_text(
    str(processoTREINO_PEN.pid), encoding="utf-8"
)

print("PID:", processoTREINO_PEN.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_TREINO_PEN)
print("LOG:", LOG_TREINO_PEN)

In [ ]:
estado = processoTREINO_PEN.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_TREINO_PEN.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO_NOVO = Path(RUN_TREINO_PEN).resolve()

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

assert ler_json(TREINO_NOVO / "concluido.json")["passos"] == 2000
cfg = ler_json(TREINO_NOVO / "config.json")
assert cfg["total_passos"] == 2000
assert cfg["extracao"]["estagio_indice"] == 2
assert cfg["extracao"]["normalizacao_saida"] == "nenhuma"

PASSOS_AVALIAR = [0, 50, 100, 200, 500, 1000, 1500, 2000]
for passo in PASSOS_AVALIAR:
    assert (TREINO_NOVO / f"checkpoint_{passo:04d}.pt").is_file()

BASE = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima512_sem_norm.py"
SCRIPT_AVAL_PEN = BASE.with_name("avaliar_eft_direto_penultima512.py")
codigo = BASE.read_text(encoding="utf-8")

def trocar_unico(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

trocar_unico(
    'TREINO = ROOT / "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"',
    f"TREINO = Path({str(TREINO_NOVO)!r})",
)
trocar_unico(
    "PASSOS = [0, 200]",
    f"PASSOS = {PASSOS_AVALIAR!r}",
)
trocar_unico(
    'assert ler(TREINO / "concluido.json")["passos"] == 5000',
    'assert ler(TREINO / "concluido.json")["passos"] == 2000',
)
trocar_unico(
    '"finalidade": "avaliacao penultima sem outnorm; validacao S00",',
    '"finalidade": "avaliacao do EFT treinado diretamente pela penultima; S00",',
)

# O avaliador mantém a extração em 672, ALP em 32x32 e SAM em 1024.
# Carrega os pesos completos antes de instalar a extração da penúltima.
compile(codigo, str(SCRIPT_AVAL_PEN), "exec")

if SCRIPT_AVAL_PEN.exists():
    assert SCRIPT_AVAL_PEN.read_text(encoding="utf-8") == codigo, (
        "Já existe outro avaliador nesse caminho."
    )
else:
    with SCRIPT_AVAL_PEN.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Treinamento:", TREINO_NOVO)
print("Avaliador:", SCRIPT_AVAL_PEN)
print("Checkpoints:", PASSOS_AVALIAR)
print("Total: 1160 avaliações de query.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoAVAL_PEN_DIRETO")
assert anterior is None or anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_AVAL_PEN_DIRETO = ROOT / "reproducao" / (
    f"avaliacao_eft_direto_penultima512_{stamp}"
)
RUN_AVAL_PEN_DIRETO.mkdir(parents=True, exist_ok=False)
LOG_AVAL_PEN_DIRETO = RUN_AVAL_PEN_DIRETO / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_AVAL_PEN),
    "--out", str(RUN_AVAL_PEN_DIRETO),
]
(RUN_AVAL_PEN_DIRETO / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_AVAL_PEN_DIRETO.open("w", encoding="utf-8") as log:
    processoAVAL_PEN_DIRETO = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_AVAL_PEN_DIRETO / "pid.txt").write_text(
    str(processoAVAL_PEN_DIRETO.pid), encoding="utf-8"
)

print("PID:", processoAVAL_PEN_DIRETO.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_AVAL_PEN_DIRETO)
print("LOG:", LOG_AVAL_PEN_DIRETO)

In [ ]:
estado = processoAVAL_PEN_DIRETO.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_AVAL_PEN_DIRETO.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

resumos = []
for passo in PASSOS_AVALIAR:
    arquivo = RUN_AVAL_PEN_DIRETO / f"passo_{passo:04d}/resumo.json"
    if arquivo.is_file():
        resumos.append(ler_json(arquivo))

if resumos:
    print("\n=== EFT DIRETO PELA PENÚLTIMA — S00 ===")
    print("Passo | Dice ALP | Dice ALP+SAM")
    for r in resumos:
        print(
            f"{r['passo']:5d} | {r['ALP_Dice']:.6f} | "
            f"{r['SAM_Dice']:.6f}"
        )

if (RUN_AVAL_PEN_DIRETO / "concluido.json").is_file():
    melhor = max(resumos, key=lambda r: (r["SAM_Dice"], -r["passo"]))
    referencia = 0.644332  # Penúltima do treino anterior, passo 2000, S00.
    print(f"\nMelhor checkpoint avaliado: {melhor['passo']}")
    print(f"Dice ALP+SAM: {melhor['SAM_Dice']:.2%}")
    print(
        "Diferença para a penúltima do treino anterior no S00: "
        f"{100 * (melhor['SAM_Dice'] - referencia):+.2f} p.p."
    )

In [ ]:
from pathlib import Path
import hashlib
import json

# Interrompe apenas o processo da proposta anterior, se ainda estiver ativo.
anterior_errado = globals().get("processoPEN_10K")
if anterior_errado is not None and anterior_errado.poll() is None:
    anterior_errado.terminate()
    try:
        anterior_errado.wait(timeout=30)
    except TimeoutError:
        raise RuntimeError(
            "O processo anterior ainda está encerrando. "
            "Aguarde e execute esta célula novamente."
        )
    except __import__("subprocess").TimeoutExpired:
        raise RuntimeError(
            "O processo anterior ainda está encerrando. "
            "Aguarde e execute esta célula novamente."
        )
    print("Treino direto pela penúltima interrompido.")

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
BASE = CODIGO / "treinar_5000_cuda.py"
SCRIPT_ULTIMA_10K = CODIGO / "treinar_ultima_10000_wd001_cuda.py"

REFERENCIA = ROOT / (
    "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"
)
cfg_ref = json.loads(
    (REFERENCIA / "config.json").read_text(encoding="utf-8")
)

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

for caminho, digest in cfg_ref["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

codigo = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

preservar = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]

trocar(
    "PRESERVAR = {0, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000}",
    f"PRESERVAR = set({preservar!r})",
)
trocar(
    'print("Plano conferido: mesmos 5000 episódios do histórico.", flush=True)',
    'print("Plano conferido: 5000 episódios históricos, '
    'repetidos duas vezes na mesma ordem.", flush=True)',
)
trocar(
    'return {"treino": plano, "validacao": novos["validacao"]}',
    'assert len(plano) == 5000\n'
    '    plano10k = [dict(r) for r in plano] + [dict(r) for r in plano]\n'
    '    assert plano10k[:5000] == plano10k[5000:]\n'
    '    return {"treino": plano10k, "validacao": novos["validacao"]}',
)
trocar(
    '"formato": 1, "total_passos": 5000, "semente": 1234,',
    '"formato": 1, "total_passos": 10000, "semente": 1234,\n'
    '        "experimento": "treino ultima; avaliacao planejada penultima; WD 0.01",\n'
    '        "repeticao_plano": {\n'
    '            "episodios_por_ciclo": 5000,\n'
    '            "ciclos": 2,\n'
    '            "ordem": "historica, identica nos dois ciclos",\n'
    '        },\n'
    '        "decay_aplicado_a": "todos os parametros do otimizador",',
)
trocar(
    '"eps": 1e-8, "weight_decay": 0.0,',
    '"eps": 1e-8, "weight_decay": 0.01,',
)
trocar(
    "eps=1e-8, weight_decay=0.0",
    "eps=1e-8, weight_decay=0.01",
)
trocar("assert 0 <= passo <= 5000", "assert 0 <= passo <= 10000")
trocar("range(passo + 1, 5001)", "range(passo + 1, 10001)")
trocar("media * (5000 - n) / 3600", "media * (10000 - n) / 3600")
trocar('f"{n}/5000 | loss=', 'f"{n}/10000 | loss=')
trocar('"passos": 5000,', '"passos": 10000,')
trocar(
    'print("TREINAMENTO CONCLUÍDO: 5000 passos.", flush=True)',
    'print("TREINAMENTO CONCLUÍDO: 10000 passos pela ultima; WD=0.01.", '
    'flush=True)',
)
trocar(
    '"passo": n, "indice": r["indice"], "imagem": r["imagem"],',
    '"passo": n, "ciclo_plano": (n - 1) // 5000 + 1,\n'
    '                "posicao_no_ciclo": (n - 1) % 5000 + 1,\n'
    '                "indice": r["indice"], "imagem": r["imagem"],',
)
trocar(
    "    assert torch.cuda.is_available()\n",
    "    assert torch.cuda.is_available()\n"
    "    def recusar_fallback(*args, **kwargs):\n"
    "        raise RuntimeError('Fallback Torch proibido neste treino.')\n"
    "    vmamba.selective_scan_torch = recusar_fallback\n",
)
trocar(
    "    try:\n        main(pasta)\n",
    "    import fcntl\n"
    "    pasta.mkdir(parents=True, exist_ok=True)\n"
    "    trava = (pasta / 'execucao.lock').open('w')\n"
    "    fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)\n"
    "    try:\n        main(pasta)\n",
)

# Confirma que usa o modelo original: treino pela última, encoder completo.
assert "from eft_modelo_512 import construir_modelo" in codigo
assert '"adaptacao": "encoder completo, features da ultima camada"' in codigo

compile(codigo, str(SCRIPT_ULTIMA_10K), "exec")

if SCRIPT_ULTIMA_10K.exists():
    assert SCRIPT_ULTIMA_10K.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_ULTIMA_10K.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Script:", SCRIPT_ULTIMA_10K)
print("Treino: última camada, encoder completo.")
print("Avaliação posterior: penúltima, sem outnorm.")
print("Checkpoints:", preservar)

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoULTIMA_10K")
assert anterior is None or anterior.poll() is not None, (
    "Este treinamento já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ULTIMA_10K = ROOT / "reproducao" / (
    f"eft_ultima512_10000_wd001_{stamp}"
)
RUN_ULTIMA_10K.mkdir(parents=True, exist_ok=False)
LOG_ULTIMA_10K = RUN_ULTIMA_10K / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_ULTIMA_10K),
    "--run", str(RUN_ULTIMA_10K),
]
(RUN_ULTIMA_10K / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_ULTIMA_10K.open("w", encoding="utf-8") as log:
    processoULTIMA_10K = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_ULTIMA_10K / "pid.txt").write_text(
    str(processoULTIMA_10K.pid), encoding="utf-8"
)

print("PID:", processoULTIMA_10K.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_ULTIMA_10K)
print("LOG:", LOG_ULTIMA_10K)

In [ ]:
estado = processoULTIMA_10K.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_ULTIMA_10K.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

In [ ]:
from pathlib import Path
import json
import hashlib

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO_10K = Path(RUN_ULTIMA_10K).resolve()

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

cfg = ler_json(TREINO_10K / "config.json")
assert ler_json(TREINO_10K / "concluido.json")["passos"] == 10000
assert cfg["total_passos"] == 10000
assert cfg["weight_decay"] == 0.01
assert cfg["adaptacao"] == "encoder completo, features da ultima camada"

# Confere que os arquivos do treinamento permanecem iguais.
for caminho, digest in cfg["fontes_sha256"].items():
    assert hashlib.sha256(Path(caminho).read_bytes()).hexdigest() == digest, (
        f"Arquivo mudou: {caminho}"
    )

PASSOS_10K = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]
for passo in PASSOS_10K:
    assert (TREINO_10K / f"checkpoint_{passo:04d}.pt").is_file()

BASE = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima512_sem_norm.py"
SCRIPT_AVAL_10K = BASE.with_name("avaliar_penultima_ultima10k_wd001.py")
codigo = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

trocar(
    'TREINO = ROOT / "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"',
    f"TREINO = Path({str(TREINO_10K)!r})",
)
trocar("PASSOS = [0, 200]", f"PASSOS = {PASSOS_10K!r}")
trocar(
    'assert ler(TREINO / "concluido.json")["passos"] == 5000',
    'assert ler(TREINO / "concluido.json")["passos"] == 10000',
)
trocar(
    '"finalidade": "avaliacao penultima sem outnorm; validacao S00",',
    '"finalidade": "penultima sem outnorm; treino ultima 10k WD0.01; S00",',
)

compile(codigo, str(SCRIPT_AVAL_10K), "exec")

if SCRIPT_AVAL_10K.exists():
    assert SCRIPT_AVAL_10K.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_AVAL_10K.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Treinamento:", TREINO_10K)
print("Avaliador:", SCRIPT_AVAL_10K)
print("Checkpoints:", PASSOS_10K)
print("Total: 2320 avaliações de query.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoAVAL_10K")
assert anterior is None or anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_AVAL_10K = ROOT / "reproducao" / (
    f"avaliacao_penultima_ultima10k_wd001_{stamp}"
)
RUN_AVAL_10K.mkdir(parents=True, exist_ok=False)
LOG_AVAL_10K = RUN_AVAL_10K / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_AVAL_10K),
    "--out", str(RUN_AVAL_10K),
]
(RUN_AVAL_10K / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_AVAL_10K.open("w", encoding="utf-8") as log:
    processoAVAL_10K = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_AVAL_10K / "pid.txt").write_text(
    str(processoAVAL_10K.pid), encoding="utf-8"
)

print("PID:", processoAVAL_10K.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_AVAL_10K)
print("LOG:", LOG_AVAL_10K)

In [ ]:
estado = processoAVAL_10K.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_AVAL_10K.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

resumos = []
for passo in PASSOS_10K:
    arquivo = RUN_AVAL_10K / f"passo_{passo:04d}/resumo.json"
    if arquivo.is_file():
        resumos.append(ler_json(arquivo))

if resumos:
    print("\n=== TREINO ÚLTIMA, WD 0.01 — AVALIAÇÃO PENÚLTIMA, S00 ===")
    print("Passo | Dice ALP | Dice ALP+SAM")
    for r in resumos:
        print(
            f"{r['passo']:5d} | {r['ALP_Dice']:.6f} | "
            f"{r['SAM_Dice']:.6f}"
        )

if (RUN_AVAL_10K / "concluido.json").is_file():
    melhor = max(resumos, key=lambda r: (r["SAM_Dice"], -r["passo"]))
    print(f"\nMelhor checkpoint avaliado: {melhor['passo']}")
    print(f"Dice ALP+SAM: {melhor['SAM_Dice']:.2%}")
    print(
        "Diferença para a referência anterior no S00 (64,43%): "
        f"{100 * (melhor['SAM_Dice'] - 0.644332):+.2f} p.p."
    )

In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

BASE_MODELO = CODIGO / "eft_modelo_512.py"
BASE_TREINO = CODIGO / "treinar_ultima_10000_wd001_cuda.py"
MODELO_672 = CODIGO / "eft_modelo_672.py"
SCRIPT_672 = CODIGO / "treinar_ultima672_10000_wd001_cuda.py"

TREINO_REF = Path(RUN_ULTIMA_10K).resolve()
cfg_ref = json.loads(
    (TREINO_REF / "config.json").read_text(encoding="utf-8")
)
assert cfg_ref["total_passos"] == 10000
assert cfg_ref["weight_decay"] == 0.01

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

for caminho, digest in cfg_ref["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

# Modelo original: altera somente a resolução e a checagem da grade nativa.
modelo_codigo = BASE_MODELO.read_text(encoding="utf-8")

def substituir(texto, antigo, novo):
    assert texto.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    return texto.replace(antigo, novo, 1)

modelo_codigo = substituir(
    modelo_codigo, "size=(512, 512)", "size=(672, 672)"
)
modelo_codigo = substituir(
    modelo_codigo, "(canais, 16, 16)", "(canais, 21, 21)"
)
modelo_codigo = substituir(
    modelo_codigo, "image_size=512", "image_size=672"
)

codigo = BASE_TREINO.read_text(encoding="utf-8")

codigo = substituir(
    codigo,
    "from eft_modelo_512 import construir_modelo",
    "from eft_modelo_672 import construir_modelo",
)
codigo = substituir(
    codigo,
    'Path(__file__), CODIGO / "eft_modelo_512.py",',
    'Path(__file__), CODIGO / "eft_modelo_672.py",',
)
codigo = substituir(
    codigo,
    '"resolucao_treino": [512, 512], "grade_alp": [32, 32],',
    '"resolucao_treino": [672, 672], "grade_alp": [32, 32],\n'
    '        "preparo_672": {\n'
    '            "rgb": "original RGB -> PIL bilinear 672",\n'
    '            "mascara": "regiao256 recuperada do cache512 -> nearest 672",\n'
    '            "superpixels_recalculados": False,\n'
    '            "aumento": "query por flip horizontal apos resize",\n'
    '        },',
)
codigo = substituir(
    codigo,
    '"experimento": "treino ultima; avaliacao planejada penultima; WD 0.01",',
    '"experimento": "treino672 ultima; avaliacao penultima; 10k; WD 0.01",',
)
codigo = substituir(
    codigo,
    "assert logits.shape == (1, 2, 512, 512)",
    "assert logits.shape == (1, 2, 672, 672)",
)
codigo = substituir(
    codigo,
    'print("TREINAMENTO CONCLUÍDO: 10000 passos pela ultima; WD=0.01.", '
    'flush=True)',
    'print("TREINAMENTO CONCLUÍDO: 10000 passos em 672 pela ultima; '
    'WD=0.01.", flush=True)',
)

# Substitui a preparação do episódio inteiro, sem alterar o plano.
arvore = ast.parse(codigo)
funcoes = [
    no for no in arvore.body
    if isinstance(no, ast.FunctionDef) and no.name == "episodio"
]
assert len(funcoes) == 1
funcao_antiga = ast.get_source_segment(codigo, funcoes[0])

funcao_nova = textwrap.dedent('''
def episodio(registro):
    from PIL import Image

    # Recupera exatamente a região256 que originou o cache512.
    with np.load(DADOS / registro["cache"], allow_pickle=False) as d:
        mask512 = (d["segmentos"] == registro["regiao"]) & d["util"]

    assert mask512.shape == (512, 512)
    mask256 = mask512[::2, ::2].copy()
    assert np.array_equal(
        mask512, mask256.repeat(2, axis=0).repeat(2, axis=1)
    ), "Cache512 não corresponde à ampliação exata da região256."

    # RGB original: não amplia o RGB512 do cache.
    imagem_original = Path(registro["imagem"])
    assert imagem_original.is_file()
    assert sha(imagem_original) == registro["imagem_sha256"]

    with Image.open(imagem_original) as im:
        rgb = np.array(
            im.convert("RGB").resize(
                (672, 672), resample=Image.Resampling.BILINEAR
            )
        )

    mask = np.array(
        Image.fromarray(mask256.astype(np.uint8)).resize(
            (672, 672), resample=Image.Resampling.NEAREST
        )
    ) > 0

    assert rgb.shape == (672, 672, 3)
    assert mask.shape == (672, 672) and mask.any()

    def imagem(a):
        t = torch.from_numpy(np.ascontiguousarray(a))
        t = t.permute(2, 0, 1)[None].to("cuda", dtype=torch.float32)
        media = t.new_tensor([123.675, 116.28, 103.53])[None, :, None, None]
        desvio = t.new_tensor([58.395, 57.12, 57.375])[None, :, None, None]
        return (t - media) / desvio

    return (
        imagem(rgb),
        torch.from_numpy(mask.copy())[None].to("cuda", dtype=torch.float32),
        imagem(np.fliplr(rgb)),
        torch.from_numpy(np.fliplr(mask).copy())[None].to(
            "cuda", dtype=torch.long
        ),
    )
''').strip()

codigo = substituir(codigo, funcao_antiga, funcao_nova)

# Mantém as verificações de loss, gradientes e backend CUDA do treinador.
assert "range(passo + 1, 10001)" in codigo
assert "eps=1e-8, weight_decay=0.01" in codigo
assert "Fallback Torch proibido" in codigo

for arquivo, conteudo in (
    (MODELO_672, modelo_codigo),
    (SCRIPT_672, codigo),
):
    compile(conteudo, str(arquivo), "exec")

for arquivo, conteudo in (
    (MODELO_672, modelo_codigo),
    (SCRIPT_672, codigo),
):
    if arquivo.exists():
        assert arquivo.read_text(encoding="utf-8") == conteudo, (
            f"Já existe outra versão: {arquivo}"
        )
    else:
        with arquivo.open("x", encoding="utf-8") as f:
            f.write(conteudo)

    print("Arquivo:", arquivo)
    print("SHA256:", sha_local(arquivo))

print("\nPreparado: treino672 pela última, 10000 passos, WD=0.01.")
print("Checkpoints:", cfg_ref["checkpoints_preservados"])
print("Treinamento ainda não iniciado.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoTREINO_672")
assert anterior is None or anterior.poll() is not None, (
    "Este treinamento já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_TREINO_672 = ROOT / "reproducao" / (
    f"eft_ultima672_10000_wd001_{stamp}"
)
RUN_TREINO_672.mkdir(parents=True, exist_ok=False)
LOG_TREINO_672 = RUN_TREINO_672 / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_672),
    "--run", str(RUN_TREINO_672),
]
(RUN_TREINO_672 / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_TREINO_672.open("w", encoding="utf-8") as log:
    processoTREINO_672 = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_TREINO_672 / "pid.txt").write_text(
    str(processoTREINO_672.pid), encoding="utf-8"
)

print("PID:", processoTREINO_672.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_TREINO_672)
print("LOG:", LOG_TREINO_672)

In [ ]:
estado = processoTREINO_672.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_TREINO_672.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

arquivo_final = RUN_TREINO_672 / "concluido.json"
if arquivo_final.is_file():
    print("\nRESUMO:")
    print(arquivo_final.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import json
import hashlib

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO_672 = ROOT / (
    "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

cfg = ler_json(TREINO_672 / "config.json")
assert ler_json(TREINO_672 / "concluido.json")["passos"] == 10000
assert cfg["resolucao_treino"] == [672, 672]
assert cfg["weight_decay"] == 0.01
assert cfg["adaptacao"] == "encoder completo, features da ultima camada"

for caminho, digest in cfg["fontes_sha256"].items():
    assert hashlib.sha256(Path(caminho).read_bytes()).hexdigest() == digest, (
        f"Arquivo mudou: {caminho}"
    )

PASSOS_AVAL_672 = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]
for passo in PASSOS_AVAL_672:
    assert (TREINO_672 / f"checkpoint_{passo:04d}.pt").is_file()

# Reutiliza o avaliador da penúltima que já funcionou.
# A avaliação permanece em 672, independentemente da resolução do treino.
BASE = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima512_sem_norm.py"
SCRIPT_AVAL_672 = BASE.with_name("avaliar_penultima_treino672_10k_wd001.py")
codigo = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

trocar(
    'TREINO = ROOT / "reproducao/eft_tiny512_cuda_5000_20261017T103006205469Z"',
    f"TREINO = Path({str(TREINO_672)!r})",
)
trocar("PASSOS = [0, 200]", f"PASSOS = {PASSOS_AVAL_672!r}")
trocar(
    'assert ler(TREINO / "concluido.json")["passos"] == 5000',
    'assert ler(TREINO / "concluido.json")["passos"] == 10000',
)
trocar(
    '"finalidade": "avaliacao penultima sem outnorm; validacao S00",',
    '"finalidade": "penultima sem outnorm; treino672 ultima 10k WD0.01; S00",',
)

compile(codigo, str(SCRIPT_AVAL_672), "exec")

if SCRIPT_AVAL_672.exists():
    assert SCRIPT_AVAL_672.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_AVAL_672.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Avaliador:", SCRIPT_AVAL_672)
print("16 checkpoints × 145 queries = 2320 avaliações.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoAVAL_672")
assert anterior is None or anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_AVAL_672 = ROOT / "reproducao" / (
    f"avaliacao_penultima_treino672_10k_wd001_{stamp}"
)
RUN_AVAL_672.mkdir(parents=True, exist_ok=False)
LOG_AVAL_672 = RUN_AVAL_672 / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_AVAL_672),
    "--out", str(RUN_AVAL_672),
]
(RUN_AVAL_672 / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_AVAL_672.open("w", encoding="utf-8") as log:
    processoAVAL_672 = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_AVAL_672 / "pid.txt").write_text(
    str(processoAVAL_672.pid), encoding="utf-8"
)

print("PID:", processoAVAL_672.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_AVAL_672)
print("LOG:", LOG_AVAL_672)

In [ ]:
estado = processoAVAL_672.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_AVAL_672.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

sam_treino512 = {
    0: 0.509673,
    50: 0.582620,
    100: 0.604211,
    200: 0.598868,
    500: 0.645722,
    1000: 0.629862,
    1500: 0.647818,
    2000: 0.617840,
    3000: 0.595406,
    4000: 0.571801,
    5000: 0.550919,
    6000: 0.490283,
    7000: 0.515679,
    8000: 0.515777,
    9000: 0.514812,
    10000: 0.483808,
}

resumos = []
for passo in PASSOS_AVAL_672:
    arquivo = RUN_AVAL_672 / f"passo_{passo:04d}/resumo.json"
    if arquivo.is_file():
        resumos.append(ler_json(arquivo))

if resumos:
    print("\n=== S00 — PENÚLTIMA, WD 0.01 ===")
    print("Passo | ALP treino672 | SAM treino512 | SAM treino672 | Δ SAM (p.p.)")
    for r in resumos:
        ref = sam_treino512[r["passo"]]
        print(
            f"{r['passo']:5d} | {r['ALP_Dice']:.6f}      | "
            f"{ref:.6f}      | {r['SAM_Dice']:.6f}      | "
            f"{100 * (r['SAM_Dice'] - ref):+.2f}"
        )

if (RUN_AVAL_672 / "concluido.json").is_file():
    melhor = max(resumos, key=lambda r: (r["SAM_Dice"], -r["passo"]))
    print(f"\nMelhor checkpoint: {melhor['passo']}")
    print(f"Dice ALP+SAM: {melhor['SAM_Dice']:.2%}")
    print(
        "Diferença para o melhor treino512 com decay: "
        f"{100 * (melhor['SAM_Dice'] - max(sam_treino512.values())):+.2f} p.p."
    )

In [ ]:
from pathlib import Path

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
arquivo = ROOT / "models/ProtoSAM.py"

assert arquivo.is_file(), f"Arquivo não encontrado: {arquivo}"
print(arquivo.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import hashlib
import json
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
BASELINE_672 = Path(RUN_AVAL_672).resolve()
BASE = ROOT / (
    "exp/eft_tiny_512_v1/avaliar_penultima_treino672_10k_wd001.py"
)
SCRIPT_COMPARACAO = BASE.with_name("avaliar_672_camadas_selecao_sam.py")

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

assert (BASELINE_672 / "concluido.json").is_file()
cfg_base = ler_json(BASELINE_672 / "config.json")
assert cfg_base["fontes_sha256"][str(BASE)] == sha_local(BASE)
assert cfg_base["sam"]["use_cca"] is False
assert cfg_base["sam"]["use_mask"] is False

codigo = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

# Intercepta apenas o retorno do predictor.
# O ProtoSAM continua escolhendo [0]; no modo max_score,
# colocamos nessa posição a candidata com maior score.
helper = textwrap.dedent('''
def instalar_selecao_sam(protosam, selecao):
    assert selecao in ("indice0", "max_score")
    assert not protosam.use_cca
    assert not protosam.use_mask

    original = protosam.predictor.predict
    auditoria = []
    protosam.auditoria_selecao = auditoria

    def predict(*args, **kwargs):
        assert kwargs.get("multimask_output") is True
        assert not kwargs.get("return_logits", False)

        masks, scores, low_res = original(*args, **kwargs)
        scores = np.asarray(scores)
        assert scores.ndim == 1 and len(scores) > 1
        assert masks.shape[0] == low_res.shape[0] == len(scores)
        assert np.isfinite(scores).all()

        escolhido = int(np.argmax(scores)) if selecao == "max_score" else 0
        auditoria.append({
            "scores": scores.astype(float).tolist(),
            "indice_escolhido": escolhido,
            "numero_candidatas": len(scores),
        })

        ordem = [escolhido] + [
            i for i in range(len(scores)) if i != escolhido
        ]
        return masks[ordem], scores[ordem], low_res[ordem]

    protosam.predictor.predict = predict

''')

trocar(
    "def main(out):",
    helper + "\ndef main(out, camada, selecao):",
)

trocar(
    "            instalar_penultima(modelo)",
    '            if camada == "penultima":\n'
    '                instalar_penultima(modelo)\n'
    '            else:\n'
    '                assert camada == "ultima"\n'
    '                print("Extração: última com norm aprendida.", flush=True)',
)

trocar(
    "            protosam.eval().requires_grad_(False)",
    "            instalar_selecao_sam(protosam, selecao)\n"
    "            protosam.eval().requires_grad_(False)",
)

trocar(
    '                        chamadas["cuda"] = 0',
    '                        protosam.auditoria_selecao.clear()\n'
    '                        chamadas["cuda"] = 0',
)

trocar(
    '                            "scores_sam": len(scores),',
    '                            "scores_sam": len(scores),\n'
    '                            "selecao_sam": selecao,\n'
    '                            "candidatas_por_componente": '
    'list(protosam.auditoria_selecao),',
)

trocar(
    '"finalidade": "penultima sem outnorm; treino672 ultima 10k WD0.01; S00",',
    '"finalidade": "comparacao de camada e selecao SAM; treino672; S00",\n'
    '        "camada_avaliada": camada,',
)

trocar(
    '"selecao_sam": ref["selecao_sam"],',
    '"selecao_sam": {\n'
    '            "regra": selecao,\n'
    '            "unidade": "candidatas de cada componente",\n'
    '            "agregacao": "uniao original do ProtoSAM",\n'
    '            "usa_ground_truth": False,\n'
    '        },',
)

# O bloco de extração do script original descreve somente a penúltima.
# Sobrescreve essa descrição conforme a condição executada.
marcador = (
    '    out.mkdir(parents=True, exist_ok=True)\n'
    '    if (out / "config.json").exists():\n'
)
verificacao = textwrap.dedent('''
    config["extracao"] = {
        "estagio_indice": 2 if camada == "penultima" else 3,
        "ponto": "apos blocks, antes de downsample",
        "canais": 384 if camada == "penultima" else 768,
        "grade_nativa": [42, 42] if camada == "penultima" else [21, 21],
        "normalizacao_saida": "nenhuma" if camada == "penultima" else "outnorm3 do checkpoint",
        "novo_treinamento": False,
    }
    baseline = ler(BASELINE / "config.json")
    for chave in (
        "treino", "treino_config_sha256", "passos",
        "checkpoints_sha256", "episodios", "suporte",
        "suporte_tensor_sha256", "encoder", "grade_alp", "canvas",
        "sam", "sam_sha256", "alp_inferencia",
        "backend", "extensao_sha256", "torch", "cuda", "cudnn",
    ):
        assert config[chave] == baseline[chave], (
            "Comparacao com protocolo diferente: " + chave
        )
    config["baseline_reutilizado"] = str(BASELINE)
    config["baseline_config_sha256"] = sha(BASELINE / "config.json")
''')
verificacao = textwrap.indent(verificacao.strip() + "\n", "    ")
trocar(marcador, verificacao + "\n" + marcador)

# Confirma que mudar a escolha do SAM não mudou o ALP da penúltima.
trecho_alp = (
    '                        alp = (logits.argmax(1)[0] == 1).cpu().numpy()'
)
trocar(
    trecho_alp,
    trecho_alp + '\n'
    '                        if camada == "penultima":\n'
    '                            arquivo_base = BASELINE / '
    'f"passo_{passo:04d}" / "alp" / f\'{item["id"]}.png\'\n'
    '                            with Image.open(arquivo_base) as im:\n'
    '                                assert np.array_equal('
    'alp, np.asarray(im) > 0), (\n'
    '                                    "ALP mudou em relação ao baseline: " '
    '+ item["id"]\n'
    '                                )',
)

# Novo executor: três condições em sequência e tabela com as quatro.
marcador_main = 'if __name__ == "__main__":'
assert codigo.count(marcador_main) == 1
codigo = codigo.split(marcador_main)[0]
codigo += f"\nBASELINE = Path({str(BASELINE_672)!r})\n"

codigo += textwrap.dedent('''
if __name__ == "__main__":
    import fcntl

    parser = argparse.ArgumentParser()
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    out = Path(args.out).resolve()
    out.mkdir(parents=True, exist_ok=True)

    trava = (out / "execucao.lock").open("w")
    fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)

    condicoes = [
        ("penultima", "max_score"),
        ("ultima", "indice0"),
        ("ultima", "max_score"),
    ]

    def atualizar_tabela():
        linhas = []
        for passo in PASSOS:
            r = ler(BASELINE / f"passo_{passo:04d}/resumo.json")
            linha = {
                "passo": passo,
                "penultima_ALP": r["ALP_Dice"],
                "penultima_indice0_SAM": r["SAM_Dice"],
            }
            for camada, selecao in condicoes:
                p = out / f"{camada}_{selecao}" / f"passo_{passo:04d}/resumo.json"
                resultado = ler(p) if p.is_file() else None
                linha[f"{camada}_{selecao}_SAM"] = (
                    resultado["SAM_Dice"] if resultado else None
                )
                linha[f"{camada}_{selecao}_ALP"] = (
                    resultado["ALP_Dice"] if resultado else None
                )
            linhas.append(linha)
        csv_atomico(out / "comparacao.csv", linhas)
        salvar(out / "comparacao.json", linhas)
        return linhas

    try:
        lote = {
            "script_sha256": sha(__file__),
            "baseline": str(BASELINE),
            "baseline_config_sha256": sha(BASELINE / "config.json"),
            "condicoes": [list(c) for c in condicoes],
            "passos": PASSOS,
        }
        if (out / "config_lote.json").exists():
            assert ler(out / "config_lote.json") == lote
        else:
            salvar(out / "config_lote.json", lote)
            shutil.copy2(__file__, out / Path(__file__).name)

        atualizar_tabela()
        for camada, selecao in condicoes:
            print(f"\\n=== {camada.upper()} | {selecao} ===", flush=True)
            main(out / f"{camada}_{selecao}", camada, selecao)
            atualizar_tabela()

        linhas = atualizar_tabela()

        # ALP deve ser idêntico entre as duas seleções da última camada.
        for r in linhas:
            assert r["ultima_indice0_ALP"] == r["ultima_max_score_ALP"]

        colunas = [
            "penultima_indice0_SAM", "penultima_max_score_SAM",
            "ultima_indice0_SAM", "ultima_max_score_SAM",
        ]
        melhores = {}
        for coluna in colunas:
            melhor = max(linhas, key=lambda r: (r[coluna], -r["passo"]))
            melhores[coluna] = {
                "passo": melhor["passo"],
                "Dice": melhor[coluna],
            }

        salvar(out / "concluido.json", {
            "melhores_por_condicao": melhores,
            "limite": "selecao exploratoria na validacao S00; nao e teste",
        })
        print(json.dumps(melhores, indent=2), flush=True)
        print("COMPARACAO CONCLUIDA.", flush=True)

    except BaseException:
        (out / f"erro_{time.time_ns()}.txt").write_text(
            traceback.format_exc(), encoding="utf-8"
        )
        raise
''')

compile(codigo, str(SCRIPT_COMPARACAO), "exec")

if SCRIPT_COMPARACAO.exists():
    assert SCRIPT_COMPARACAO.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_COMPARACAO.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Script:", SCRIPT_COMPARACAO)
print("Baseline reutilizado:", BASELINE_672)
print("Novas avaliações: 3 condições × 16 checkpoints × 145 queries = 6960.")
print("Preparado; avaliação ainda não iniciada.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoCOMPARACAO_SAM")
assert anterior is None or anterior.poll() is not None, (
    "Esta comparação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_COMPARACAO_SAM = ROOT / "reproducao" / (
    f"comparacao672_camadas_selecaoSAM_{stamp}"
)
RUN_COMPARACAO_SAM.mkdir(parents=True, exist_ok=False)
LOG_COMPARACAO_SAM = RUN_COMPARACAO_SAM / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_COMPARACAO),
    "--out", str(RUN_COMPARACAO_SAM),
]
(RUN_COMPARACAO_SAM / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_COMPARACAO_SAM.open("w", encoding="utf-8") as log:
    processoCOMPARACAO_SAM = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_COMPARACAO_SAM / "pid.txt").write_text(
    str(processoCOMPARACAO_SAM.pid), encoding="utf-8"
)

print("PID:", processoCOMPARACAO_SAM.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_COMPARACAO_SAM)
print("LOG:", LOG_COMPARACAO_SAM)

In [ ]:
estado = processoCOMPARACAO_SAM.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_COMPARACAO_SAM.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

final = RUN_COMPARACAO_SAM / "concluido.json"
if final.is_file():
    print("\n=== MELHOR CHECKPOINT POR CONDIÇÃO ===")
    resultado = ler_json(final)
    for condicao, r in resultado["melhores_por_condicao"].items():
        print(
            f"{condicao}: passo {r['passo']} | Dice={r['Dice']:.2%}"
        )
    print("\nTabela completa:", RUN_COMPARACAO_SAM / "comparacao.csv")

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

TREINO_REF_672 = ROOT / (
    "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
)

BASE_MODELO = CODIGO / "eft_modelo_672.py"
BASE_TREINO = CODIGO / "treinar_ultima672_10000_wd001_cuda.py"

MODELO_1024 = CODIGO / "eft_modelo_1024.py"
SCRIPT_TREINO_1024 = CODIGO / "treinar_ultima1024_10000_wd001_cuda.py"

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg_ref = ler_json(TREINO_REF_672 / "config.json")
assert ler_json(TREINO_REF_672 / "concluido.json")["passos"] == 10000
assert cfg_ref["resolucao_treino"] == [672, 672]
assert cfg_ref["weight_decay"] == 0.01

for caminho, digest in cfg_ref["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

def trocar_unico(texto, antigo, novo):
    assert texto.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    return texto.replace(antigo, novo, 1)

modelo = BASE_MODELO.read_text(encoding="utf-8")
modelo = trocar_unico(modelo, "size=(672, 672)", "size=(1024, 1024)")
modelo = trocar_unico(modelo, "(canais, 21, 21)", "(canais, 32, 32)")
modelo = trocar_unico(modelo, "image_size=672", "image_size=1024")

# Neste treinador, as referências a 672 identificam a resolução,
# o módulo do modelo e os metadados desse experimento.
treinador = BASE_TREINO.read_text(encoding="utf-8")
treinador = treinador.replace("672", "1024")

# A avaliação planejada permanece em encoder672/SAM1024.
treinador = trocar_unico(
    treinador,
    '"avaliacao_polipos": "separada: encoder1024/SAM1024",',
    '"avaliacao_polipos": "separada: encoder672/SAM1024",',
)

# Explicita as condições de avaliação planejadas.
treinador = trocar_unico(
    treinador,
    '"formato": 1, "total_passos": 10000, "semente": 1234,',
    '"formato": 1, "total_passos": 10000, "semente": 1234,\n'
    '        "comparacoes_planejadas": {\n'
    '            "encoder_avaliacao": 672,\n'
    '            "canvas_sam": 1024,\n'
    '            "camadas": ["ultima", "penultima"],\n'
    '            "selecao_sam": ["indice0", "max_score"],\n'
    '            "suporte_inicial": "S00",\n'
    '        },',
)

assert "from eft_modelo_1024 import construir_modelo" in treinador
assert 'CODIGO / "eft_modelo_1024.py"' in treinador
assert '"resolucao_treino": [1024, 1024]' in treinador
assert "assert logits.shape == (1, 2, 1024, 1024)" in treinador
assert "range(passo + 1, 10001)" in treinador
assert "eps=1e-8, weight_decay=0.01" in treinador
assert "Fallback Torch proibido" in treinador

for arquivo, conteudo in (
    (MODELO_1024, modelo),
    (SCRIPT_TREINO_1024, treinador),
):
    compile(conteudo, str(arquivo), "exec")

for arquivo, conteudo in (
    (MODELO_1024, modelo),
    (SCRIPT_TREINO_1024, treinador),
):
    if arquivo.exists():
        assert arquivo.read_text(encoding="utf-8") == conteudo, (
            f"Já existe outra versão: {arquivo}"
        )
    else:
        with arquivo.open("x", encoding="utf-8") as f:
            f.write(conteudo)

    print("Arquivo:", arquivo)
    print("SHA256:", sha_local(arquivo))

print("\nTreino: última, entrada1024, features nativas768×32×32.")
print("Início: pesos pré-treinados; 10000 passos; WD=0.01.")
print("Checkpoints:", cfg_ref["checkpoints_preservados"])
print("Preparado; treinamento ainda não iniciado.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoTREINO_1024")
assert anterior is None or anterior.poll() is not None, (
    "Este treinamento já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_TREINO_1024 = ROOT / "reproducao" / (
    f"eft_ultima1024_10000_wd001_{stamp}"
)
RUN_TREINO_1024.mkdir(parents=True, exist_ok=False)
LOG_TREINO_1024 = RUN_TREINO_1024 / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_TREINO_1024),
    "--run", str(RUN_TREINO_1024),
]
(RUN_TREINO_1024 / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_TREINO_1024.open("w", encoding="utf-8") as log:
    processoTREINO_1024 = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_TREINO_1024 / "pid.txt").write_text(
    str(processoTREINO_1024.pid), encoding="utf-8"
)

print("PID:", processoTREINO_1024.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_TREINO_1024)
print("LOG:", LOG_TREINO_1024)

In [ ]:
estado = processoTREINO_1024.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_TREINO_1024.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

final = RUN_TREINO_1024 / "concluido.json"
if final.is_file():
    print("\nRESUMO:")
    print(final.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import textwrap
from datetime import datetime, timezone

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
TREINO_1024 = ROOT / (
    "reproducao/eft_ultima1024_10000_wd001_20261018T202558239976Z"
)

BASE_AVAL = CODIGO / "avaliar_penultima_treino672_10k_wd001.py"
BASE_COMP = CODIGO / "avaliar_672_camadas_selecao_sam.py"

COMP_ANTERIOR = ROOT / (
    "reproducao/comparacao672_camadas_selecaoSAM_20261018T175942960894Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg = ler_json(TREINO_1024 / "config.json")
assert ler_json(TREINO_1024 / "concluido.json")["passos"] == 10000
assert cfg["resolucao_treino"] == [1024, 1024]
assert cfg["weight_decay"] == 0.01

for caminho, digest in cfg["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

# Confere os avaliadores que produziram os resultados anteriores.
lote_anterior = ler_json(COMP_ANTERIOR / "config_lote.json")
assert lote_anterior["script_sha256"] == sha_local(BASE_COMP)

baseline_anterior = Path(lote_anterior["baseline"])
cfg_baseline_anterior = ler_json(baseline_anterior / "config.json")
assert cfg_baseline_anterior["fontes_sha256"][str(BASE_AVAL)] == sha_local(BASE_AVAL)

PASSOS_1024 = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]
assert cfg_baseline_anterior["passos"] == PASSOS_1024
for passo in PASSOS_1024:
    assert (TREINO_1024 / f"checkpoint_{passo:04d}.pt").is_file()

# Impede preparar outra execução enquanto a anterior estiver ativa.
anterior = globals().get("processoTESTES_1024")
assert anterior is None or anterior.poll() is not None

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_TESTES_1024 = ROOT / "reproducao" / (
    f"avaliacao_treino1024_quatro_condicoes_{stamp}"
)
BASELINE_NOVO = RUN_TESTES_1024 / "penultima_indice0"
COMPARACAO_NOVA = RUN_TESTES_1024 / "comparacao"

# Altera atribuições pelo AST, evitando substituir caminhos por engano.
def trocar_atribuicao(texto, nome, expressao):
    arvore = ast.parse(texto)
    encontrados = [
        no for no in arvore.body
        if isinstance(no, ast.Assign)
        and any(isinstance(t, ast.Name) and t.id == nome for t in no.targets)
    ]
    assert len(encontrados) == 1, nome
    no = encontrados[0]
    linhas = texto.splitlines(keepends=True)
    linhas[no.lineno - 1:no.end_lineno] = [f"{nome} = {expressao}\n"]
    return "".join(linhas)

codigo_aval = BASE_AVAL.read_text(encoding="utf-8")
codigo_aval = trocar_atribuicao(
    codigo_aval, "TREINO", f"Path({str(TREINO_1024)!r})"
)
codigo_aval = codigo_aval.replace(
    "penultima sem outnorm; treino672 ultima 10k WD0.01; S00",
    "penultima sem outnorm; treino1024; avaliacao672; WD0.01; S00",
)

codigo_comp = BASE_COMP.read_text(encoding="utf-8")
codigo_comp = trocar_atribuicao(
    codigo_comp, "TREINO", f"Path({str(TREINO_1024)!r})"
)
codigo_comp = trocar_atribuicao(
    codigo_comp, "BASELINE", f"Path({str(BASELINE_NOVO)!r})"
)
codigo_comp = codigo_comp.replace(
    "comparacao de camada e selecao SAM; treino672; S00",
    "comparacao de camada e selecao SAM; treino1024; avaliacao672; S00",
)

SCRIPT_BASE_1024 = RUN_TESTES_1024 / "avaliar_baseline.py"
SCRIPT_COMP_1024 = RUN_TESTES_1024 / "avaliar_comparacao.py"
SCRIPT_LOTE_1024 = RUN_TESTES_1024 / "executar_lote.py"

executor = textwrap.dedent(f'''
from pathlib import Path
import fcntl
import json
import subprocess
import sys
import time
import traceback

ROOT = Path({str(ROOT)!r})
OUT = Path({str(RUN_TESTES_1024)!r})

trava = (OUT / "execucao.lock").open("w")
fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)

try:
    print("ETAPA 1/2: penultima, indice0, todos os checkpoints.", flush=True)
    subprocess.run([
        sys.executable, "-u", {str(SCRIPT_BASE_1024)!r},
        "--out", {str(BASELINE_NOVO)!r},
    ], cwd=ROOT, check=True)

    print("ETAPA 2/2: demais tres condicoes, todos os checkpoints.", flush=True)
    subprocess.run([
        sys.executable, "-u", {str(SCRIPT_COMP_1024)!r},
        "--out", {str(COMPARACAO_NOVA)!r},
    ], cwd=ROOT, check=True)

    resultado = json.loads(
        (Path({str(COMPARACAO_NOVA)!r}) / "concluido.json")
        .read_text(encoding="utf-8")
    )
    (OUT / "concluido.json").write_text(
        json.dumps(resultado, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(json.dumps(resultado, indent=2, ensure_ascii=False), flush=True)
    print("QUATRO CONDICOES CONCLUIDAS.", flush=True)

except BaseException:
    (OUT / f"erro_{{time.time_ns()}}.txt").write_text(
        traceback.format_exc(), encoding="utf-8"
    )
    raise
''')

arquivos = [
    (SCRIPT_BASE_1024, codigo_aval),
    (SCRIPT_COMP_1024, codigo_comp),
    (SCRIPT_LOTE_1024, executor),
]
for arquivo, conteudo in arquivos:
    compile(conteudo, str(arquivo), "exec")

RUN_TESTES_1024.mkdir(parents=True, exist_ok=False)
for arquivo, conteudo in arquivos:
    arquivo.write_text(conteudo, encoding="utf-8")

(RUN_TESTES_1024 / "preparacao.json").write_text(
    json.dumps({
        "treino": str(TREINO_1024),
        "resolucao_treino": 1024,
        "resolucao_encoder_avaliacao": 672,
        "canvas_sam": 1024,
        "passos": PASSOS_1024,
        "scripts_sha256": {str(p): sha_local(p) for p, _ in arquivos},
        "avaliadores_origem_sha256": {
            str(BASE_AVAL): sha_local(BASE_AVAL),
            str(BASE_COMP): sha_local(BASE_COMP),
        },
    }, indent=2),
    encoding="utf-8",
)

print("EXECUÇÃO — guarde este caminho:", RUN_TESTES_1024)
print("Preparado: 4 condições × 16 checkpoints × 145 queries.")
print("Avaliação ainda não iniciada.")

In [ ]:
import subprocess
import sys

anterior = globals().get("processoTESTES_1024")
assert anterior is None or anterior.poll() is not None

LOG_TESTES_1024 = RUN_TESTES_1024 / "console.log"

with LOG_TESTES_1024.open("a", encoding="utf-8") as log:
    processoTESTES_1024 = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT_LOTE_1024)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_TESTES_1024 / "pid.txt").write_text(
    str(processoTESTES_1024.pid), encoding="utf-8"
)

print("PID do executor:", processoTESTES_1024.pid)
print("LOG:", LOG_TESTES_1024)

In [ ]:
estado = processoTESTES_1024.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_TESTES_1024.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

final = RUN_TESTES_1024 / "concluido.json"
if final.is_file():
    resultado = ler_json(final)
    print("\n=== TREINO1024 — MELHOR CHECKPOINT POR CONDIÇÃO ===")

    referencias_672 = {
        "penultima_indice0_SAM": 0.6807229550208969,
        "penultima_max_score_SAM": 0.6484592853566966,
        "ultima_indice0_SAM": 0.6129957132064672,
        "ultima_max_score_SAM": 0.6515275985672543,
    }

    for condicao, r in resultado["melhores_por_condicao"].items():
        delta = 100 * (r["Dice"] - referencias_672[condicao])
        print(
            f"{condicao}: passo {r['passo']} | "
            f"Dice={r['Dice']:.2%} | "
            f"Δ melhor treino672={delta:+.2f} p.p."
        )

    print("\nTabela por checkpoint:", COMPARACAO_NOVA / "comparacao.csv")

In [ ]:
from pathlib import Path
import ast
import json
import textwrap
from datetime import datetime, timezone

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

# Reutiliza a preparação dos suportes que já funcionou.
BASE_MULTI = CODIGO / "avaliar_multissuporte_penultima2000.py"

# Reutiliza a seleção por score que já foi avaliada.
BASE_SELECAO = CODIGO / "avaliar_672_camadas_selecao_sam.py"

PLANO = ROOT / (
    "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
)

CANDIDATOS = {
    "A_672_penultima_500_indice0": {
        "treino": ROOT / (
            "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
        ),
        "passo": 500,
        "camada": "penultima",
        "selecao": "indice0",
    },
    "B_1024_ultima_1500_maxscore": {
        "treino": ROOT / (
            "reproducao/eft_ultima1024_10000_wd001_20261018T202558239976Z"
        ),
        "passo": 1500,
        "camada": "ultima",
        "selecao": "max_score",
    },
}

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

assert PLANO.is_file()
for c in CANDIDATOS.values():
    assert ler_json(c["treino"] / "concluido.json")["passos"] == 10000
    assert (c["treino"] / f"checkpoint_{c['passo']:04d}.pt").is_file()

anterior = globals().get("processoMULTI_AB")
assert anterior is None or anterior.poll() is not None

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_MULTI_AB = ROOT / "reproducao" / f"multissuporte_candidatos_AB_{stamp}"

def trocar(texto, antigo, novo):
    assert texto.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    return texto.replace(antigo, novo, 1)

def atribuir(texto, nome, expressao):
    arvore = ast.parse(texto)
    nos = [
        n for n in arvore.body
        if isinstance(n, ast.Assign)
        and any(isinstance(t, ast.Name) and t.id == nome for t in n.targets)
    ]
    assert len(nos) == 1, nome
    n = nos[0]
    linhas = texto.splitlines(keepends=True)
    linhas[n.lineno - 1:n.end_lineno] = [f"{nome} = {expressao}\n"]
    return "".join(linhas)

# Copia a função de seleção já utilizada, sem alterar o ProtoSAM.
fonte_selecao = BASE_SELECAO.read_text(encoding="utf-8")
arvore = ast.parse(fonte_selecao)
nos = [
    n for n in arvore.body
    if isinstance(n, ast.FunctionDef) and n.name == "instalar_selecao_sam"
]
assert len(nos) == 1
helper = ast.get_source_segment(fonte_selecao, nos[0])

base = BASE_MULTI.read_text(encoding="utf-8")
marcador_main = 'if __name__ == "__main__":'
assert base.count(marcador_main) == 1
base = base.split(marcador_main)[0]

arquivos = []

for nome, c in CANDIDATOS.items():
    codigo = atribuir(base, "TREINO", f"Path({str(c['treino'])!r})")
    codigo = atribuir(codigo, "PASSOS", repr([c["passo"]]))

    codigo = trocar(
        codigo,
        'assert ler(TREINO / "concluido.json")["passos"] == 5000',
        'assert ler(TREINO / "concluido.json")["passos"] == 10000',
    )
    codigo = trocar(
        codigo,
        "def main(out, suporte_id):",
        helper + "\n\ndef main(out, suporte_id):",
    )

    if c["camada"] == "ultima":
        codigo = trocar(
            codigo,
            "            instalar_penultima(modelo)",
            '            print("Extração: última com norm do checkpoint.", flush=True)',
        )

    codigo = trocar(
        codigo,
        "            protosam.eval().requires_grad_(False)",
        f"            instalar_selecao_sam(protosam, {c['selecao']!r})\n"
        "            protosam.eval().requires_grad_(False)",
    )
    codigo = trocar(
        codigo,
        '                        chamadas["cuda"] = 0',
        '                        protosam.auditoria_selecao.clear()\n'
        '                        chamadas["cuda"] = 0',
    )
    codigo = trocar(
        codigo,
        '                            "scores_sam": len(scores),',
        '                            "scores_sam": len(scores),\n'
        '                            "candidatas_por_componente": '
        'list(protosam.auditoria_selecao),',
    )

    # Atualiza a descrição da condição antes de gravar a configuração.
    extracao = {
        "estagio_indice": 2 if c["camada"] == "penultima" else 3,
        "canais": 384 if c["camada"] == "penultima" else 768,
        "grade_nativa": [42, 42] if c["camada"] == "penultima" else [21, 21],
        "normalizacao_saida": (
            "nenhuma" if c["camada"] == "penultima" else "outnorm3 do checkpoint"
        ),
        "novo_treinamento": False,
    }
    marcador = (
        '    out.mkdir(parents=True, exist_ok=True)\n'
        '    if (out / "config.json").exists():\n'
    )
    atualizacao = (
        f'    config["finalidade"] = "comparacao multissuporte dos candidatos A e B"\n'
        f'    config["candidato"] = {nome!r}\n'
        f'    config["extracao"] = {extracao!r}\n'
        f'    config["selecao_sam"] = {c["selecao"]!r}\n'
        f'    assert cfg_sam["use_cca"] is False\n'
        f'    assert cfg_sam["use_mask"] is False\n'
    )
    codigo = trocar(codigo, marcador, atualizacao + marcador)

    # Executor de um candidato, com retomada pelos registros existentes.
    codigo += textwrap.dedent(f'''
if __name__ == "__main__":
    import fcntl

    parser = argparse.ArgumentParser()
    parser.add_argument("--out", required=True)
    args = parser.parse_args()
    out = Path(args.out).resolve()
    out.mkdir(parents=True, exist_ok=True)

    trava = (out / "execucao.lock").open("w")
    fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)

    try:
        plano_path = Path({str(PLANO)!r})
        plano = ler(plano_path)
        assert [s["id"] for s in plano["suportes"]] == [
            f"S{{i:02d}}" for i in range(20)
        ]

        lote = {{
            "candidato": {nome!r},
            "script_sha256": sha(__file__),
            "plano_sha256": sha(plano_path),
            "treino": str(TREINO),
            "passo": {c["passo"]},
            "suportes": [f"S{{i:02d}}" for i in range(1, 20)],
        }}
        if (out / "config_lote.json").exists():
            assert ler(out / "config_lote.json") == lote
        else:
            salvar(out / "config_lote.json", lote)
            shutil.copy2(plano_path, out / "plano_suportes.json")

        resultados = []
        for sid in lote["suportes"]:
            print(f"\\n=== {nome} | {{sid}} ===", flush=True)
            main(out / sid, sid)
            resumo = ler(out / sid / "passo_{c["passo"]:04d}/resumo.json")
            assert resumo["n"] == 145
            resultados.append({{"suporte": sid, **resumo}})
            csv_atomico(out / "por_suporte.csv", resultados)
            salvar(out / "por_suporte.json", resultados)

        agregado = {{
            "candidato": {nome!r},
            "n_suportes": 19,
            "queries_por_suporte": 145,
            "limite": "validacao exploratoria; variacao entre suportes",
        }}
        for etapa in ("ALP", "SAM"):
            valores = np.array([r[f"{{etapa}}_Dice"] for r in resultados])
            agregado[etapa] = {{
                "media": float(valores.mean()),
                "desvio_padrao_entre_suportes_ddof1": float(valores.std(ddof=1)),
                "minimo": float(valores.min()),
                "maximo": float(valores.max()),
            }}

        salvar(out / "concluido.json", agregado)
        print(json.dumps(agregado, indent=2), flush=True)

    except BaseException:
        (out / f"erro_{{time.time_ns()}}.txt").write_text(
            traceback.format_exc(), encoding="utf-8"
        )
        raise
''')

    arquivo = RUN_MULTI_AB / f"{nome}.py"
    compile(codigo, str(arquivo), "exec")
    arquivos.append((arquivo, codigo))

RUN_MULTI_AB.mkdir(parents=True, exist_ok=False)
for arquivo, codigo in arquivos:
    arquivo.write_text(codigo, encoding="utf-8")

print("EXECUÇÃO — guarde este caminho:", RUN_MULTI_AB)
print("Avaliadores preparados. Nenhuma avaliação iniciada ainda.")

In [ ]:
import subprocess
import sys

nomes = list(CANDIDATOS)
SCRIPT_LOTE_AB = RUN_MULTI_AB / "executar_AB.py"

executor = textwrap.dedent(f'''
from pathlib import Path
import csv
import fcntl
import json
import subprocess
import sys
import traceback
import numpy as np

ROOT = Path({str(ROOT)!r})
OUT = Path({str(RUN_MULTI_AB)!r})
NOMES = {nomes!r}

def ler(p):
    return json.loads(p.read_text(encoding="utf-8"))

trava = (OUT / "execucao.lock").open("w")
fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)

try:
    for nome in NOMES:
        subprocess.run([
            sys.executable, "-u", str(OUT / (nome + ".py")),
            "--out", str(OUT / nome),
        ], cwd=ROOT, check=True)

    a = {{r["suporte"]: r for r in ler(OUT / NOMES[0] / "por_suporte.json")}}
    b = {{r["suporte"]: r for r in ler(OUT / NOMES[1] / "por_suporte.json")}}
    assert set(a) == set(b) == {{f"S{{i:02d}}" for i in range(1, 20)}}

    linhas = []
    for sid in sorted(a):
        ca = ler(OUT / NOMES[0] / sid / "config.json")
        cb = ler(OUT / NOMES[1] / sid / "config.json")

        # Confere o pareamento, incluindo os tensores e o protocolo SAM.
        for chave in (
            "episodios", "suporte", "suporte_tensor_sha256",
            "encoder", "grade_alp", "canvas", "sam", "sam_sha256",
            "plano_suportes_sha256",
        ):
            assert ca[chave] == cb[chave], (sid, chave)

        linhas.append({{
            "suporte": sid,
            "A_ALP": a[sid]["ALP_Dice"],
            "B_ALP": b[sid]["ALP_Dice"],
            "A_SAM": a[sid]["SAM_Dice"],
            "B_SAM": b[sid]["SAM_Dice"],
            "delta_B_menos_A_pp": 100 * (
                b[sid]["SAM_Dice"] - a[sid]["SAM_Dice"]
            ),
        }})

    with (OUT / "comparacao_pareada.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0]))
        w.writeheader()
        w.writerows(linhas)

    deltas = np.array([r["delta_B_menos_A_pp"] for r in linhas])
    resumo = {{
        "A": ler(OUT / NOMES[0] / "concluido.json"),
        "B": ler(OUT / NOMES[1] / "concluido.json"),
        "comparacao_SAM": {{
            "delta_medio_B_menos_A_pp": float(deltas.mean()),
            "suportes_B_maior": int((deltas > 0).sum()),
            "suportes_A_maior": int((deltas < 0).sum()),
            "empates": int((deltas == 0).sum()),
        }},
        "limite": "validacao exploratoria; nao mede variacao entre treinamentos",
    }}
    (OUT / "concluido.json").write_text(
        json.dumps(resumo, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(json.dumps(resumo, indent=2, ensure_ascii=False), flush=True)
    print("MULTISSUPORTE A/B CONCLUIDO.", flush=True)

except BaseException:
    (OUT / "erro_executor.txt").write_text(
        traceback.format_exc(), encoding="utf-8"
    )
    raise
''')

compile(executor, str(SCRIPT_LOTE_AB), "exec")
SCRIPT_LOTE_AB.write_text(executor, encoding="utf-8")

anterior = globals().get("processoMULTI_AB")
assert anterior is None or anterior.poll() is not None

LOG_MULTI_AB = RUN_MULTI_AB / "console.log"
with LOG_MULTI_AB.open("a", encoding="utf-8") as log:
    processoMULTI_AB = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT_LOTE_AB)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_MULTI_AB / "pid.txt").write_text(
    str(processoMULTI_AB.pid), encoding="utf-8"
)
print("PID do executor:", processoMULTI_AB.pid)
print("LOG:", LOG_MULTI_AB)

In [ ]:
estado = processoMULTI_AB.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_MULTI_AB.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

final = RUN_MULTI_AB / "concluido.json"
if final.is_file():
    r = ler_json(final)
    print("\n=== RESULTADO MULTISSUPORTE ===")
    for nome in ("A", "B"):
        s = r[nome]["SAM"]
        print(
            f"{nome}: Dice SAM = {s['media']:.2%} | "
            f"DP entre suportes = "
            f"{100 * s['desvio_padrao_entre_suportes_ddof1']:.2f} p.p."
        )
    print("\nComparação pareada:", r["comparacao_SAM"])
    print("Tabela:", RUN_MULTI_AB / "comparacao_pareada.csv")

In [ ]:
from pathlib import Path
import hashlib
import json
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
BASE = CODIGO / "treinar_ultima672_10000_wd001_cuda.py"
SCRIPT_CONGELADO = CODIGO / "treinar_672_estagios23_10000_cuda.py"

TREINO_REFERENCIA = ROOT / (
    "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg_ref = ler_json(TREINO_REFERENCIA / "config.json")
assert cfg_ref["resolucao_treino"] == [672, 672]
assert cfg_ref["lr"] == 1e-5
assert cfg_ref["weight_decay"] == 0.01

for caminho, digest in cfg_ref["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

codigo = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

helper = textwrap.dedent('''
def configurar_congelamento(modelo):
    encoder = modelo.encoder
    assert len(encoder.layers) == 4

    congelados = (
        encoder.patch_embed,
        encoder.layers[0],
        encoder.layers[1],
    )
    for modulo in congelados:
        modulo.requires_grad_(False)
        modulo.eval()

    for modulo in (encoder.layers[2], encoder.layers[3], encoder.outnorm3):
        assert all(p.requires_grad for p in modulo.parameters())

    parametros_fixos = {
        nome: p for nome, p in modelo.named_parameters()
        if not p.requires_grad
    }
    assert parametros_fixos

    print("Congelados: patch_embed, estagios 0 e 1; modo eval.", flush=True)
    print("Treinaveis: estagios 2 e 3, outnorm3.", flush=True)
    print(
        "Parametros treinaveis:",
        sum(p.numel() for p in modelo.parameters() if p.requires_grad),
        flush=True,
    )
    print(
        "Parametros congelados:",
        sum(p.numel() for p in parametros_fixos.values()),
        flush=True,
    )
    return congelados, parametros_fixos

''')

trocar("def main(pasta):", helper + "\ndef main(pasta):")

trocar(
    '    modelo = construir_modelo("tiny", repo=ROOT)',
    '    modelo = construir_modelo("tiny", repo=ROOT)\n'
    '    modulos_fixos, parametros_fixos = configurar_congelamento(modelo)',
)

trocar(
    "modelo.parameters(), lr=1e-5, betas=(0.9, 0.999),",
    "[p for p in modelo.parameters() if p.requires_grad],\n"
    "        lr=1e-5, betas=(0.9, 0.999),",
)

trocar(
    '"adaptacao": "encoder completo, features da ultima camada",',
    '"adaptacao": "estagios 2 e 3 treinaveis; features da ultima camada",\n'
    '        "congelamento": {\n'
    '            "modulos": ["patch_embed", "layers.0", "layers.1"],\n'
    '            "modo": "eval durante treinamento",\n'
    '            "inclui_downsample_dos_estagios_congelados": True,\n'
    '            "outnorm3_treinavel": True,\n'
    '        },',
)

trocar(
    '"experimento": "treino672 ultima; avaliacao penultima; 10k; WD 0.01",',
    '"experimento": "treino672 estagios23; avaliacao penultima; 10k; WD0.01",',
)

# modelo.train() reativa treinamento em todos os submódulos.
# Reaplica eval aos congelados depois dessa chamada.
trocar(
    "    modelo.train()\n",
    "    modelo.train()\n"
    "    for modulo in modulos_fixos:\n"
    "        modulo.eval()\n"
    "    assert all(\n"
    "        not sub.training\n"
    "        for modulo in modulos_fixos for sub in modulo.modules()\n"
    "    )\n"
    "    referencia_fixa = {\n"
    "        nome: p.detach().cpu().clone()\n"
    "        for nome, p in parametros_fixos.items()\n"
    "    }\n",
)

trocar(
    "            loss.backward()\n",
    "            loss.backward()\n"
    "            assert all(p.grad is None for p in parametros_fixos.values()), (\n"
    "                'Parametro congelado recebeu gradiente.'\n"
    "            )\n"
    "            for indice_estagio in (2, 3):\n"
    "                assert any(\n"
    "                    p.grad is not None and p.grad.abs().max().item() > 0\n"
    "                    for p in modelo.encoder.layers[indice_estagio].parameters()\n"
    "                ), f'Estagio {indice_estagio} sem gradiente nao zero.'\n",
)

# Confere periodicamente que os pesos congelados não mudaram.
trocar(
    "            if n % 50 == 0:\n                salvar(n)",
    "            if n % 50 == 0:\n"
    "                assert all(\n"
    "                    torch.equal(p.detach().cpu(), referencia_fixa[nome])\n"
    "                    for nome, p in parametros_fixos.items()\n"
    "                ), 'Pesos congelados foram alterados.'\n"
    "                salvar(n)",
)

trocar(
    '"avaliacao_polipos_pendente": True,',
    '"avaliacao_polipos_pendente": True,\n'
    '        "pesos_congelados_inalterados_na_tentativa": True,',
)

trocar(
    'print("TREINAMENTO CONCLUÍDO: 10000 passos em 672 pela ultima; '
    'WD=0.01.", flush=True)',
    'print("TREINAMENTO CONCLUÍDO: 672, estagios 2-3 treinaveis, '
    '10000 passos, WD=0.01.", flush=True)',
)

compile(codigo, str(SCRIPT_CONGELADO), "exec")

if SCRIPT_CONGELADO.exists():
    assert SCRIPT_CONGELADO.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_CONGELADO.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Script:", SCRIPT_CONGELADO)
print("SHA256:", sha_local(SCRIPT_CONGELADO))
print("Checkpoints:", cfg_ref["checkpoints_preservados"])
print("Preparado; treinamento ainda não iniciado.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoCONGELADO")
assert anterior is None or anterior.poll() is not None, (
    "Este treinamento já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_CONGELADO = ROOT / "reproducao" / (
    f"eft672_estagios23_10000_wd001_{stamp}"
)
RUN_CONGELADO.mkdir(parents=True, exist_ok=False)
LOG_CONGELADO = RUN_CONGELADO / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_CONGELADO),
    "--run", str(RUN_CONGELADO),
]
(RUN_CONGELADO / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_CONGELADO.open("w", encoding="utf-8") as log:
    processoCONGELADO = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_CONGELADO / "pid.txt").write_text(
    str(processoCONGELADO.pid), encoding="utf-8"
)

print("PID:", processoCONGELADO.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_CONGELADO)
print("LOG:", LOG_CONGELADO)

In [ ]:
estado = processoCONGELADO.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_CONGELADO.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

final = RUN_CONGELADO / "concluido.json"
if final.is_file():
    print("\nRESUMO:")
    print(final.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import ast
import hashlib
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO_CONGELADO = ROOT / (
    "reproducao/eft672_estagios23_10000_wd001_20261019T101744349108Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

cfg = ler_json(TREINO_CONGELADO / "config.json")
fim = ler_json(TREINO_CONGELADO / "concluido.json")

assert fim["passos"] == 10000
assert fim["pesos_congelados_inalterados_na_tentativa"] is True
assert cfg["resolucao_treino"] == [672, 672]
assert cfg["congelamento"]["modulos"] == [
    "patch_embed", "layers.0", "layers.1"
]

for caminho, digest in cfg["fontes_sha256"].items():
    assert hashlib.sha256(Path(caminho).read_bytes()).hexdigest() == digest, (
        f"Arquivo mudou: {caminho}"
    )

PASSOS_CONGELADO = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]
for passo in PASSOS_CONGELADO:
    assert (TREINO_CONGELADO / f"checkpoint_{passo:04d}.pt").is_file()

BASE = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima_treino672_10k_wd001.py"
SCRIPT_AVAL_CONGELADO = BASE.with_name("avaliar_penultima_672_estagios23.py")

# Confere o avaliador contra a avaliação anterior.
comparacao_anterior = ROOT / (
    "reproducao/comparacao672_camadas_selecaoSAM_20261018T175942960894Z"
)
BASELINE_COMPLETO = Path(
    ler_json(comparacao_anterior / "config_lote.json")["baseline"]
)
cfg_baseline = ler_json(BASELINE_COMPLETO / "config.json")
assert cfg_baseline["fontes_sha256"][str(BASE)] == (
    hashlib.sha256(BASE.read_bytes()).hexdigest()
)
assert cfg_baseline["passos"] == PASSOS_CONGELADO

codigo = BASE.read_text(encoding="utf-8")

# Troca apenas o caminho do treinamento.
arvore = ast.parse(codigo)
nos = [
    n for n in arvore.body
    if isinstance(n, ast.Assign)
    and any(isinstance(t, ast.Name) and t.id == "TREINO" for t in n.targets)
]
assert len(nos) == 1
no = nos[0]
linhas = codigo.splitlines(keepends=True)
linhas[no.lineno - 1:no.end_lineno] = [
    f"TREINO = Path({str(TREINO_CONGELADO)!r})\n"
]
codigo = "".join(linhas)

antigo = (
    '"finalidade": "penultima sem outnorm; treino672 ultima 10k WD0.01; S00",'
)
assert codigo.count(antigo) == 1
codigo = codigo.replace(
    antigo,
    '"finalidade": "penultima sem outnorm; treino672 estagios23; S00",',
    1,
)

compile(codigo, str(SCRIPT_AVAL_CONGELADO), "exec")

if SCRIPT_AVAL_CONGELADO.exists():
    assert SCRIPT_AVAL_CONGELADO.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_AVAL_CONGELADO.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Avaliador:", SCRIPT_AVAL_CONGELADO)
print("16 checkpoints × 145 queries = 2320 avaliações.")
print("Extração: penúltima sem outnorm; SAM índice [0].")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoAVAL_CONGELADO")
assert anterior is None or anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_AVAL_CONGELADO = ROOT / "reproducao" / (
    f"avaliacao_penultima_672_estagios23_{stamp}"
)
RUN_AVAL_CONGELADO.mkdir(parents=True, exist_ok=False)
LOG_AVAL_CONGELADO = RUN_AVAL_CONGELADO / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_AVAL_CONGELADO),
    "--out", str(RUN_AVAL_CONGELADO),
]
(RUN_AVAL_CONGELADO / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_AVAL_CONGELADO.open("w", encoding="utf-8") as log:
    processoAVAL_CONGELADO = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_AVAL_CONGELADO / "pid.txt").write_text(
    str(processoAVAL_CONGELADO.pid), encoding="utf-8"
)

print("PID:", processoAVAL_CONGELADO.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_AVAL_CONGELADO)
print("LOG:", LOG_AVAL_CONGELADO)

In [ ]:
estado = processoAVAL_CONGELADO.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_AVAL_CONGELADO.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

# Confere que os dados e o protocolo de avaliação são os mesmos.
config_nova = RUN_AVAL_CONGELADO / "config.json"
if config_nova.is_file():
    nova = ler_json(config_nova)
    for chave in (
        "episodios", "suporte", "suporte_tensor_sha256",
        "encoder", "grade_alp", "canvas", "sam",
        "sam_sha256", "selecao_sam", "alp_inferencia",
    ):
        assert nova[chave] == cfg_baseline[chave], chave

resultados = []
referencias = []

for passo in PASSOS_CONGELADO:
    ref = ler_json(
        BASELINE_COMPLETO / f"passo_{passo:04d}/resumo.json"
    )
    referencias.append(ref)
    arquivo = RUN_AVAL_CONGELADO / f"passo_{passo:04d}/resumo.json"

    if arquivo.is_file():
        r = ler_json(arquivo)
        resultados.append((r, ref))

if resultados:
    print("\n=== S00 — PENÚLTIMA + SAM [0] ===")
    print("Passo | ALP parcial | SAM completo | SAM parcial | Δ SAM (p.p.)")
    for r, ref in resultados:
        print(
            f"{r['passo']:5d} | {r['ALP_Dice']:.6f}    | "
            f"{ref['SAM_Dice']:.6f}     | {r['SAM_Dice']:.6f}    | "
            f"{100 * (r['SAM_Dice'] - ref['SAM_Dice']):+.2f}"
        )

if (RUN_AVAL_CONGELADO / "concluido.json").is_file():
    melhor = max(
        (r for r, _ in resultados),
        key=lambda r: (r["SAM_Dice"], -r["passo"]),
    )
    melhor_ref = max(r["SAM_Dice"] for r in referencias)

    print(f"\nMelhor checkpoint do treino parcial: {melhor['passo']}")
    print(f"Dice ALP+SAM: {melhor['SAM_Dice']:.2%}")
    print(
        "Diferença para o melhor do encoder completo: "
        f"{100 * (melhor['SAM_Dice'] - melhor_ref):+.2f} p.p."
    )

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

# Base: encoder completo, não o treinador com congelamento.
BASE = CODIGO / "treinar_ultima672_10000_wd001_cuda.py"
SCRIPT_LR3 = CODIGO / "treinar_ultima672_10000_lr3e6_wd001_cuda.py"

REFERENCIA = ROOT / (
    "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha_local(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg_ref = ler_json(REFERENCIA / "config.json")
assert cfg_ref["lr"] == 1e-5
assert cfg_ref["weight_decay"] == 0.01
assert cfg_ref["resolucao_treino"] == [672, 672]
assert cfg_ref["adaptacao"] == "encoder completo, features da ultima camada"

for caminho, digest in cfg_ref["fontes_sha256"].items():
    assert sha_local(caminho) == digest, f"Arquivo mudou: {caminho}"

codigo = BASE.read_text(encoding="utf-8")

def trocar(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho ausente ou repetido:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

trocar(
    '"otimizador": "AdamW", "lr": 1e-5, "betas": [0.9, 0.999],',
    '"otimizador": "AdamW", "lr": 3e-6, "betas": [0.9, 0.999],',
)
trocar(
    "modelo.parameters(), lr=1e-5, betas=(0.9, 0.999),",
    "modelo.parameters(), lr=3e-6, betas=(0.9, 0.999),",
)
trocar(
    '"experimento": "treino672 ultima; avaliacao penultima; 10k; WD 0.01",',
    '"experimento": "treino672 encoder completo; LR3e-6; 10k; WD0.01",',
)
trocar(
    'print("TREINAMENTO CONCLUÍDO: 10000 passos em 672 pela ultima; '
    'WD=0.01.", flush=True)',
    'print("TREINAMENTO CONCLUÍDO: 672, encoder completo, '
    '10000 passos, LR=3e-6, WD=0.01.", flush=True)',
)

# Confirma que todos os parâmetros do encoder estão treináveis.
trocar(
    '    modelo = construir_modelo("tiny", repo=ROOT)',
    '    modelo = construir_modelo("tiny", repo=ROOT)\n'
    '    assert all(p.requires_grad for p in modelo.encoder.parameters()), (\n'
    '        "Há parâmetros congelados no encoder."\n'
    '    )\n'
    '    print("Encoder completo treinavel; LR=3e-6.", flush=True)',
)

compile(codigo, str(SCRIPT_LR3), "exec")

if SCRIPT_LR3.exists():
    assert SCRIPT_LR3.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_LR3.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Script:", SCRIPT_LR3)
print("SHA256:", sha_local(SCRIPT_LR3))
print("Checkpoints:", cfg_ref["checkpoints_preservados"])
print("Preparado; treinamento ainda não iniciado.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoLR3")
assert anterior is None or anterior.poll() is not None, (
    "Este treinamento já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_LR3 = ROOT / "reproducao" / f"eft672_completo_lr3e6_10000_{stamp}"
RUN_LR3.mkdir(parents=True, exist_ok=False)
LOG_LR3 = RUN_LR3 / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_LR3),
    "--run", str(RUN_LR3),
]
(RUN_LR3 / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_LR3.open("w", encoding="utf-8") as log:
    processoLR3 = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_LR3 / "pid.txt").write_text(
    str(processoLR3.pid), encoding="utf-8"
)

print("PID:", processoLR3.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_LR3)
print("LOG:", LOG_LR3)

In [ ]:
estado = processoLR3.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_LR3.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

final = RUN_LR3 / "concluido.json"
if final.is_file():
    print("\nRESUMO:")
    print(final.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
import ast
import hashlib
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
TREINO_LR3 = ROOT / (
    "reproducao/eft672_completo_lr3e6_10000_20261019T234014899622Z"
)

def ler_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

cfg = ler_json(TREINO_LR3 / "config.json")
assert ler_json(TREINO_LR3 / "concluido.json")["passos"] == 10000
assert cfg["lr"] == 3e-6
assert cfg["weight_decay"] == 0.01
assert cfg["resolucao_treino"] == [672, 672]
assert cfg["adaptacao"] == "encoder completo, features da ultima camada"

for caminho, digest in cfg["fontes_sha256"].items():
    assert hashlib.sha256(Path(caminho).read_bytes()).hexdigest() == digest, (
        f"Arquivo mudou: {caminho}"
    )

BASE = ROOT / "exp/eft_tiny_512_v1/avaliar_penultima_treino672_10k_wd001.py"
SCRIPT_AVAL_LR3 = BASE.with_name("avaliar_penultima672_lr3e6.py")

COMP_ANTERIOR = ROOT / (
    "reproducao/comparacao672_camadas_selecaoSAM_20261018T175942960894Z"
)
BASELINE_LR1 = Path(
    ler_json(COMP_ANTERIOR / "config_lote.json")["baseline"]
)
cfg_baseline = ler_json(BASELINE_LR1 / "config.json")
assert cfg_baseline["fontes_sha256"][str(BASE)] == (
    hashlib.sha256(BASE.read_bytes()).hexdigest()
)

PASSOS_LR3 = cfg_baseline["passos"]
for passo in PASSOS_LR3:
    assert (TREINO_LR3 / f"checkpoint_{passo:04d}.pt").is_file()

codigo = BASE.read_text(encoding="utf-8")
arvore = ast.parse(codigo)
nos = [
    n for n in arvore.body
    if isinstance(n, ast.Assign)
    and any(isinstance(t, ast.Name) and t.id == "TREINO" for t in n.targets)
]
assert len(nos) == 1
no = nos[0]
linhas = codigo.splitlines(keepends=True)
linhas[no.lineno - 1:no.end_lineno] = [
    f"TREINO = Path({str(TREINO_LR3)!r})\n"
]
codigo = "".join(linhas)

antigo = (
    '"finalidade": "penultima sem outnorm; treino672 ultima 10k WD0.01; S00",'
)
assert codigo.count(antigo) == 1
codigo = codigo.replace(
    antigo,
    '"finalidade": "penultima sem outnorm; treino672 completo LR3e-6; S00",',
    1,
)

# Confere o protocolo antes de começar as inferências.
marcador = (
    '    out.mkdir(parents=True, exist_ok=True)\n'
    '    if (out / "config.json").exists():\n'
)
assert codigo.count(marcador) == 1
verificacao = (
    f'    baseline = ler(Path({str(BASELINE_LR1)!r}) / "config.json")\n'
    '    for chave in (\n'
    '        "episodios", "suporte", "suporte_tensor_sha256",\n'
    '        "encoder", "grade_alp", "canvas", "sam",\n'
    '        "sam_sha256", "selecao_sam", "alp_inferencia",\n'
    '        "passos", "backend", "extensao_sha256",\n'
    '    ):\n'
    '        assert config[chave] == baseline[chave], chave\n'
)
codigo = codigo.replace(marcador, verificacao + marcador, 1)

compile(codigo, str(SCRIPT_AVAL_LR3), "exec")
if SCRIPT_AVAL_LR3.exists():
    assert SCRIPT_AVAL_LR3.read_text(encoding="utf-8") == codigo
else:
    with SCRIPT_AVAL_LR3.open("x", encoding="utf-8") as f:
        f.write(codigo)

print("Avaliador:", SCRIPT_AVAL_LR3)
print("Checkpoints:", PASSOS_LR3)
print("Total:", len(PASSOS_LR3) * 145, "avaliações de query.")

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone

anterior = globals().get("processoAVAL_LR3")
assert anterior is None or anterior.poll() is not None, (
    "Esta avaliação já está executando."
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_AVAL_LR3 = ROOT / "reproducao" / f"avaliacao_penultima672_lr3e6_{stamp}"
RUN_AVAL_LR3.mkdir(parents=True, exist_ok=False)
LOG_AVAL_LR3 = RUN_AVAL_LR3 / "console.log"

comando = [
    sys.executable, "-u", str(SCRIPT_AVAL_LR3),
    "--out", str(RUN_AVAL_LR3),
]
(RUN_AVAL_LR3 / "comando.json").write_text(
    json.dumps(comando, indent=2), encoding="utf-8"
)

with LOG_AVAL_LR3.open("w", encoding="utf-8") as log:
    processoAVAL_LR3 = subprocess.Popen(
        comando,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

(RUN_AVAL_LR3 / "pid.txt").write_text(
    str(processoAVAL_LR3.pid), encoding="utf-8"
)

print("PID:", processoAVAL_LR3.pid)
print("EXECUÇÃO — guarde este caminho:", RUN_AVAL_LR3)
print("LOG:", LOG_AVAL_LR3)

In [ ]:
estado = processoAVAL_LR3.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)
print("\n".join(
    LOG_AVAL_LR3.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

resultados = []
referencias = []

for passo in PASSOS_LR3:
    ref = ler_json(BASELINE_LR1 / f"passo_{passo:04d}/resumo.json")
    referencias.append(ref)

    arquivo = RUN_AVAL_LR3 / f"passo_{passo:04d}/resumo.json"
    if arquivo.is_file():
        resultados.append((ler_json(arquivo), ref))

if resultados:
    print("\n=== S00 — PENÚLTIMA + SAM [0] ===")
    print("Passo | ALP LR3e-6 | SAM LR1e-5 | SAM LR3e-6 | Δ SAM (p.p.)")
    for r, ref in resultados:
        print(
            f"{r['passo']:5d} | {r['ALP_Dice']:.6f}   | "
            f"{ref['SAM_Dice']:.6f}   | {r['SAM_Dice']:.6f}   | "
            f"{100 * (r['SAM_Dice'] - ref['SAM_Dice']):+.2f}"
        )

if (RUN_AVAL_LR3 / "concluido.json").is_file():
    melhor = max(
        (r for r, _ in resultados),
        key=lambda r: (r["SAM_Dice"], -r["passo"]),
    )
    melhor_ref = max(referencias, key=lambda r: r["SAM_Dice"])

    print(f"\nMelhor checkpoint LR3e-6: {melhor['passo']}")
    print(f"Dice ALP+SAM: {melhor['SAM_Dice']:.2%}")
    print(
        "Diferença para o melhor LR1e-5: "
        f"{100 * (melhor['SAM_Dice'] - melhor_ref['SAM_Dice']):+.2f} p.p."
    )

In [ ]:
estado = processoTOP1.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}"
)

linhas = LOG_TOP1.read_text(
    encoding="utf-8", errors="replace"
).splitlines()
print("\n".join(linhas[-30:]))

In [ ]:
import numpy as np

assert processoTOP1.poll() == 0, "A avaliação ainda não terminou com sucesso."

comparacao = []

for i in range(1, 20):
    sid = f"S{i:02d}"

    def resumo(base):
        p = base / sid / "passo_0500" / "resumo.json"
        assert p.is_file(), p
        return json.loads(p.read_text(encoding="utf-8"))

    anterior = resumo(BASE_A)
    novo = resumo(OUT_TOP1)

    assert anterior["n"] == novo["n"] == 145

    comparacao.append({
        "suporte": sid,
        "ALP_original": anterior["ALP_Dice"],
        "ALP_top1": novo["ALP_Dice"],
        "SAM_original": anterior["SAM_Dice"],
        "SAM_top1": novo["SAM_Dice"],
        "delta_SAM_pp": 100 * (
            novo["SAM_Dice"] - anterior["SAM_Dice"]
        ),
    })

a = np.array([r["SAM_original"] for r in comparacao])
b = np.array([r["SAM_top1"] for r in comparacao])
delta = b - a

print("Suporte | SAM união | SAM top1 | Δ p.p.")
for r in comparacao:
    print(
        f"{r['suporte']:>7} | "
        f"{r['SAM_original']:.6f}  | "
        f"{r['SAM_top1']:.6f} | "
        f"{r['delta_SAM_pp']:+.2f}"
    )

print(f"\nUnião: {a.mean():.2%} | DP: {100*a.std(ddof=1):.2f} p.p.")
print(f"Top1:  {b.mean():.2%} | DP: {100*b.std(ddof=1):.2f} p.p.")
print(f"Δ médio: {100*delta.mean():+.2f} p.p.")
print("Suportes com melhora:", int((delta > 0).sum()))
print("Suportes com piora:", int((delta < 0).sum()))

diferenca_alp = max(
    abs(r["ALP_original"] - r["ALP_top1"])
    for r in comparacao
)
print("Maior diferença no Dice ALP:", diferenca_alp)

(RUN_TOP1 / "comparacao_pareada.json").write_text(
    json.dumps(comparacao, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
from skimage.segmentation import felzenszwalb, mark_boundaries
from IPython.display import display

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
DADOS = ROOT / (
    "reproducao/"
    "eft_sp_dados512_regioes256_20261016T212513467126Z"
)
planos = json.loads((DADOS / "planos.json").read_text(encoding="utf-8"))

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
INSPECAO_SP512 = ROOT / "reproducao" / f"inspecao_superpixels512_{stamp}"
INSPECAO_SP512.mkdir(parents=True, exist_ok=False)

CONFIGS = {
    "512_mesmos_parametros": {
        "scale": 100, "sigma": 1.0, "min_size": 400,
        "area_minima": 100, "erosao": 3,
    },
    "512_proporcional": {
        "scale": 100, "sigma": 2.0, "min_size": 1600,
        "area_minima": 400, "erosao": 6,
    },
}

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def area_util(rgb, erosao):
    componentes, n = ndi.label(rgb.max(axis=-1) > 10)
    if n == 0:
        return np.zeros(rgb.shape[:2], dtype=bool)

    tamanhos = np.bincount(componentes.ravel())
    tamanhos[0] = 0
    maior = ndi.binary_fill_holes(componentes == tamanhos.argmax())
    return ndi.binary_erosion(
        maior,
        structure=np.ones((3, 3), dtype=bool),
        iterations=erosao,
        border_value=0,
    )

def grade(mask):
    t = torch.from_numpy(np.ascontiguousarray(mask)).float()[None, None]
    g = F.interpolate(t, size=(32, 32), mode="nearest")
    return {
        "pixels_fg": int(g.sum()),
        "celulas_fg": int((F.avg_pool2d(g, 4, 4) > 0.95).sum()),
        "celulas_bg": int((F.avg_pool2d(1 - g, 4, 4) > 0.95).sum()),
    }

def gerar(rgb, cfg):
    segmentos = felzenszwalb(
        rgb.astype(np.float32) / 255.0,
        scale=cfg["scale"],
        sigma=cfg["sigma"],
        min_size=cfg["min_size"],
        channel_axis=-1,
    ).astype(np.int32)

    util = area_util(rgb, cfg["erosao"])
    candidatos = []

    for regiao in np.unique(segmentos):
        inteira = segmentos == regiao
        mask = inteira & util
        if mask.sum() < cfg["area_minima"]:
            continue
        if mask.sum() / inteira.sum() < 0.95:
            continue

        gs = grade(mask)
        gq = grade(np.fliplr(mask))
        if all(
            g["pixels_fg"] > 0 and g["celulas_bg"] > 0
            for g in (gs, gq)
        ):
            candidatos.append(int(regiao))

    return segmentos, util, candidatos

def estatisticas(segmentos, util, candidatos):
    areas = []
    fg_local = 0

    for regiao in candidatos:
        mask = (segmentos == regiao) & util
        areas.append(float(mask.mean() * 100))
        gs, gq = grade(mask), grade(np.fliplr(mask))
        fg_local += int(gs["celulas_fg"] > 0 and gq["celulas_fg"] > 0)

    return {
        "regioes_totais": int(len(np.unique(segmentos))),
        "candidatos": len(candidatos),
        "area_mediana_candidato_pct": (
            float(np.median(areas)) if areas else None
        ),
        "candidatos_com_fg_local_support_e_query": fg_local,
    }

def painel(rgb, segmentos, util, candidatos, titulo):
    # Contornos amarelos: todas as regiões.
    # Áreas escurecidas: fora das regiões elegíveis.
    elegivel = np.isin(segmentos, candidatos) & util
    visual = rgb.astype(np.float32) / 255
    visual[~elegivel] *= 0.25
    visual = mark_boundaries(
        visual, segmentos, color=(1, 1, 0), mode="inner"
    )
    im = Image.fromarray(
        np.clip(visual * 255, 0, 255).astype(np.uint8)
    ).resize((384, 384))

    quadro = Image.new("RGB", (384, 420), "white")
    quadro.paste(im, (0, 36))
    ImageDraw.Draw(quadro).text((8, 10), titulo, fill="black")
    return quadro

# Seleção fixa, somente entre imagens de treino disponíveis no plano.
unicos = {}
for r in planos["treino"]:
    unicos.setdefault(r["indice"], r)

registros = [unicos[k] for k in sorted(unicos)]
assert len(registros) >= 6
rng = np.random.default_rng(1234)
amostra = [registros[int(i)] for i in rng.choice(
    len(registros), size=6, replace=False
)]

relatorio = []
threads_antes = torch.get_num_threads()

try:
    torch.set_num_threads(1)

    for pos, r in enumerate(amostra, 1):
        cache = DADOS / r["cache"]
        assert sha(cache) == r["cache_sha256"], cache

        with np.load(cache, allow_pickle=False) as d:
            rgb = d["rgb"].copy()
            antigo = d["segmentos"].copy()
            util_antigo = d["util"].copy().astype(bool)

        assert rgb.shape == (512, 512, 3)
        assert antigo.shape == util_antigo.shape == (512, 512)

        candidatos_antigos = [int(x) for x in r["candidatos"]]
        variantes = {
            "original256_ampliado": (
                antigo, util_antigo, candidatos_antigos
            )
        }
        for nome, cfg in CONFIGS.items():
            variantes[nome] = gerar(rgb, cfg)

        original = Image.new("RGB", (384, 420), "white")
        original.paste(
            Image.fromarray(rgb).resize((384, 384)), (0, 36)
        )
        ImageDraw.Draw(original).text((8, 10), "RGB 512", fill="black")
        paineis = [original]

        item = {
            "indice": int(r["indice"]),
            "imagem": r.get("imagem_origem", r.get("imagem")),
            "cache_origem": str(cache),
            "cache_sha256": sha(cache),
            "variantes": {},
        }

        titulos = {
            "original256_ampliado": "Regioes antigas: 256 -> 512",
            "512_mesmos_parametros": "512: mesmos parametros",
            "512_proporcional": "512: parametros proporcionais",
        }

        for nome, (seg, util, candidatos) in variantes.items():
            stats = estatisticas(seg, util, candidatos)
            item["variantes"][nome] = stats
            paineis.append(painel(
                rgb, seg, util, candidatos,
                f"{titulos[nome]} | N={len(candidatos)}",
            ))

            np.savez_compressed(
                INSPECAO_SP512 / f"{pos:02d}_{nome}.npz",
                rgb=rgb,
                segmentos=seg,
                util=util,
                candidatos=np.asarray(candidatos, dtype=np.int32),
            )

        figura = Image.new("RGB", (384 * 4, 420), "white")
        for j, p in enumerate(paineis):
            figura.paste(p, (384 * j, 0))

        destino = INSPECAO_SP512 / f"{pos:02d}_comparacao.png"
        figura.save(destino)
        item["figura"] = str(destino)
        relatorio.append(item)

        print(f"\nImagem {pos}/6 | índice {r['indice']}")
        for nome, stats in item["variantes"].items():
            print(nome, ":", stats)
        display(figura)

finally:
    torch.set_num_threads(threads_antes)

registro = {
    "objetivo": "inspecao de superpixels; sem treinamento",
    "semente_amostra": 1234,
    "dados_origem": str(DADOS),
    "plano_origem_sha256": sha(DADOS / "planos.json"),
    "configuracoes512": CONFIGS,
    "filtros": {
        "fracao_interna_minima": 0.95,
        "grade": [32, 32],
        "kernel": 4,
        "elegibilidade": (
            "FG sobrevive e existe celula BG > 0.95 "
            "em support e query"
        ),
    },
    "mascaras_manuais_utilizadas": False,
    "versoes": {
        p: importlib.metadata.version(p)
        for p in ["numpy", "Pillow", "scipy", "scikit-image", "torch"]
    },
    "amostras": relatorio,
}
(INSPECAO_SP512 / "relatorio.json").write_text(
    json.dumps(registro, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nInspeção salva em:", INSPECAO_SP512)
print("Nenhum dado anterior alterado. Nenhum treino iniciado.")

In [ ]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
REFERENCIA = ROOT / (
    "reproducao/"
    "eft_ultima672_10000_wd001_20261018T141350233303Z"
)

arquivos = [
    CODIGO / "treinar_ultima672_10000_wd001_cuda.py",
    CODIGO / "eft_modelo_672.py",
    REFERENCIA / "config.json",
]

for p in arquivos:
    assert p.is_file(), f"Arquivo não encontrado: {p}"
    print(f"\n=== {p.relative_to(ROOT)} ===")
    print(p.read_text(encoding="utf-8"))

plano_path = REFERENCIA / "planos.json"
assert plano_path.is_file(), plano_path
plano = json.loads(plano_path.read_text(encoding="utf-8"))

print("\n=== ESTRUTURA DO PLANO DE REFERÊNCIA ===")
print("SHA256:", hashlib.sha256(plano_path.read_bytes()).hexdigest())
print("Chaves:", list(plano))
print("Episódios de treino:", len(plano["treino"]))
print(
    "Imagens distintas de treino:",
    len({r["indice"] for r in plano["treino"]}),
)
print("Primeiro registro:")
print(json.dumps(plano["treino"][0], indent=2, ensure_ascii=False))

print("\nSomente leitura: nenhum arquivo alterado ou treino iniciado.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys
import textwrap

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"
REFERENCIA = ROOT / (
    "reproducao/"
    "eft_ultima672_10000_wd001_20261018T141350233303Z"
)
ORIGINAL = CODIGO / "treinar_ultima672_10000_wd001_cuda.py"

def digest(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

cfg_ref = json.loads(
    (REFERENCIA / "config.json").read_text(encoding="utf-8")
)

# Confere as fontes usadas no treino de referência.
for nome, esperado in cfg_ref["fontes_sha256"].items():
    p = Path(nome)
    assert p.is_file(), p
    assert digest(p) == esperado, f"Fonte alterada: {p}"

assert digest(REFERENCIA / "planos.json") == (
    "880b04375c4a25fa0efd326afc666011c47fe460c5180ad91653fe0cb4be024b"
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DADOS_SP512 = ROOT / "reproducao" / f"eft_superpixels512_{stamp}"
RUN_SP512 = ROOT / "reproducao" / f"eft672_superpixels512_10k_{stamp}"
SCRIPT_SP512 = CODIGO / f"treinar672_superpixels512_{stamp}.py"
LOG_SP512 = RUN_SP512 / "console.log"

codigo = ORIGINAL.read_text(encoding="utf-8")

def substituir_uma(antigo, novo):
    global codigo
    assert codigo.count(antigo) == 1, (
        f"Trecho não encontrado de forma única:\n{antigo}"
    )
    codigo = codigo.replace(antigo, novo, 1)

# Novo destino dos dados.
substituir_uma(
    'DADOS = ROOT / "reproducao/'
    'eft_sp_dados512_regioes256_20261016T212513467126Z"',
    f"DADOS = Path({str(DADOS_SP512)!r})",
)

# A máscara nova já é nativa em 512: não recuperar 256.
substituir_uma(
    '''    # Recupera exatamente a região256 que originou o cache512.''',
    '''    # Superpixels recalculados diretamente em RGB512.''',
)
substituir_uma(
    '''    mask256 = mask512[::2, ::2].copy()
    assert np.array_equal(
        mask512, mask256.repeat(2, axis=0).repeat(2, axis=1)
    ), "Cache512 não corresponde à ampliação exata da região256."
''',
    "",
)
substituir_uma(
    "Image.fromarray(mask256.astype(np.uint8))",
    "Image.fromarray(mask512.astype(np.uint8))",
)

# Metadados do novo experimento.
substituir_uma(
    '"experimento": "treino672 ultima; avaliacao penultima; 10k; WD 0.01"',
    '"experimento": "superpixels nativos512; treino672 ultima; 10k; WD 0.01"',
)
substituir_uma(
    '"ordem": "historica, identica nos dois ciclos"',
    '"ordem": "imagens historicas; novas regioes; dois ciclos identicos"',
)
substituir_uma(
    '"mascara": "regiao256 recuperada do cache512 -> nearest 672"',
    '"mascara": "superpixel nativo512 -> PIL nearest 672"',
)
substituir_uma(
    '"superpixels_recalculados": False',
    '"superpixels_recalculados": True',
)
substituir_uma(
    '"plano5000_historico_sha256": sha(H5000 / "planos.json"),',
    f'"plano_referencia_sha256": sha(Path({str(REFERENCIA)!r}) / "planos.json"),',
)
substituir_uma(
    "TREINAMENTO CONCLUÍDO: 10000 passos em 672 pela ultima; WD=0.01.",
    "TREINAMENTO CONCLUÍDO: superpixels512, treino672 pela ultima, "
    "10000 passos, WD=0.01.",
)

# Nova preparação: executada no subprocesso antes da construção do modelo.
nova_preparacao = r'''
def preparar_plano():
    from PIL import Image
    from scipy import ndimage as ndi
    from skimage.segmentation import felzenszwalb
    import fcntl

    referencia = ROOT / (
        "reproducao/"
        "eft_ultima672_10000_wd001_20261018T141350233303Z"
    )
    caminho_plano = referencia / "planos.json"
    assert sha(caminho_plano) == (
        "880b04375c4a25fa0efd326afc666011c47fe460c5180ad91653fe0cb4be024b"
    )
    ref = ler(caminho_plano)
    assert len(ref["treino"]) == 10000
    assert ref["treino"][:5000] == ref["treino"][5000:]

    treino_ids = {r["indice"] for r in ref["treino"]}
    val_ids = {r["indice"] for r in ref["validacao"]}
    assert len(treino_ids) == 1304
    assert len(val_ids) == 145
    assert not treino_ids & val_ids

    por_indice = {}
    for r in ref["treino"] + ref["validacao"]:
        if r["indice"] in por_indice:
            anterior = por_indice[r["indice"]]
            for chave in ("imagem", "imagem_sha256", "split"):
                assert anterior[chave] == r[chave]
        else:
            por_indice[r["indice"]] = r
    assert len(por_indice) == 1449

    config_dados = {
        "formato": 1,
        "resolucao_superpixels": [512, 512],
        "rgb": "original RGB -> PIL bilinear 512",
        "superpixels": {"scale": 100, "sigma": 1.0, "min_size": 400},
        "min_area_interna": 100,
        "min_fracao_interna": 0.95,
        "area_util": (
            "maxRGB>10; maior componente conexidade4; preencher buracos; "
            "erosao3x3 por3 iteracoes"
        ),
        "grade": [32, 32],
        "kernel": 4,
        "elegibilidade": (
            "FG sobrevive e existe celula BG>0.95 em support e query"
        ),
        "mascaras_manuais_utilizadas": False,
        "semente_regioes": 1234,
        "amostragem": "uniforme entre candidatos em cada episodio",
        "ciclos": "5000 novos episodios repetidos duas vezes",
        "referencia": str(referencia),
        "plano_referencia_sha256": sha(caminho_plano),
        "script_sha256": sha(__file__),
        "versoes": {
            n: versao(n)
            for n in ("numpy", "Pillow", "scipy", "scikit-image", "torch")
        },
    }

    DADOS.mkdir(parents=True, exist_ok=True)
    with (DADOS / "preparacao.lock").open("a") as trava:
        fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)
        config_path = DADOS / "config.json"

        if config_path.exists():
            assert ler(config_path) == config_dados, (
                "Configuracao da preparacao mudou."
            )
        else:
            json_atomico(config_path, config_dados)
            shutil.copy2(__file__, DADOS / "fonte_preparacao_treino.py")

        (DADOS / "cache").mkdir(exist_ok=True)

        def area_util(rgb):
            cc, n = ndi.label(rgb.max(axis=-1) > 10)
            if n == 0:
                return np.zeros(rgb.shape[:2], dtype=bool)
            tamanhos = np.bincount(cc.ravel())
            tamanhos[0] = 0
            maior = ndi.binary_fill_holes(cc == tamanhos.argmax())
            return ndi.binary_erosion(
                maior,
                structure=np.ones((3, 3), dtype=bool),
                iterations=3,
                border_value=0,
            )

        def grade(mask):
            t = torch.from_numpy(
                np.ascontiguousarray(mask)
            ).float()[None, None]
            g = F.interpolate(t, size=(32, 32), mode="nearest")
            bg = F.avg_pool2d(1 - g, kernel_size=4, stride=4)
            return bool(g.sum() > 0 and (bg > 0.95).any())

        registros = {}
        threads = torch.get_num_threads()
        print("PREPARANDO SUPERPIXELS NATIVOS EM 512:", DADOS, flush=True)

        try:
            torch.set_num_threads(1)

            for pos, indice in enumerate(sorted(por_indice), 1):
                r = por_indice[indice]
                imagem = Path(r["imagem"])
                assert imagem.is_file(), imagem
                assert sha(imagem) == r["imagem_sha256"], imagem
                assert r["split"] == (
                    "treino" if indice in treino_ids else "validacao"
                )

                cache = DADOS / "cache" / f"{indice:04d}.npz"
                meta = cache.with_suffix(".json")

                if cache.exists() and meta.exists():
                    novo = ler(meta)
                    assert novo["imagem_sha256"] == r["imagem_sha256"]
                    assert novo["split"] == r["split"]
                    assert novo["indice"] == indice
                    assert sha(cache) == novo["cache_sha256"]
                else:
                    with Image.open(imagem) as im:
                        rgb = np.array(
                            im.convert("RGB").resize(
                                (512, 512), Image.Resampling.BILINEAR
                            )
                        )

                    segmentos = felzenszwalb(
                        rgb.astype(np.float32) / 255.0,
                        channel_axis=-1,
                        scale=100, sigma=1.0, min_size=400,
                    ).astype(np.int32)
                    util = area_util(rgb)
                    candidatos = []

                    for regiao in np.unique(segmentos):
                        inteira = segmentos == regiao
                        mask = inteira & util
                        if mask.sum() < 100:
                            continue
                        if mask.sum() / inteira.sum() < 0.95:
                            continue
                        if grade(mask) and grade(np.fliplr(mask)):
                            candidatos.append(int(regiao))

                    tmp = cache.with_suffix(".tmp")
                    with tmp.open("wb") as f:
                        np.savez_compressed(
                            f, rgb=rgb, segmentos=segmentos, util=util,
                            candidatos=np.asarray(candidatos, dtype=np.int32),
                        )
                    os.replace(tmp, cache)

                    novo = {
                        "indice": indice,
                        "nome": r["nome"],
                        "imagem": str(imagem),
                        "imagem_sha256": r["imagem_sha256"],
                        "split": r["split"],
                        "cache": str(cache.relative_to(DADOS)),
                        "cache_sha256": sha(cache),
                        "candidatos": candidatos,
                        "regioes_totais": int(len(np.unique(segmentos))),
                    }
                    json_atomico(meta, novo)

                registros[indice] = novo
                if pos == 1 or pos % 25 == 0 or pos == 1449:
                    print(
                        f"Superpixels512: {pos}/1449 | "
                        f"indice={indice} | candidatos={len(novo['candidatos'])}",
                        flush=True,
                    )
        finally:
            torch.set_num_threads(threads)

        sem_treino = [
            registros[i] for i in sorted(treino_ids)
            if not registros[i]["candidatos"]
        ]
        sem_val = [
            registros[i] for i in sorted(val_ids)
            if not registros[i]["candidatos"]
        ]
        resumo = {
            "imagens_treino": 1304,
            "imagens_validacao": 145,
            "treino_sem_candidato": len(sem_treino),
            "validacao_sem_candidato": len(sem_val),
            "candidatos_treino": sum(
                len(registros[i]["candidatos"]) for i in treino_ids
            ),
            "avaliacao_real": "separada; mesmas145 queries preservadas",
        }
        json_atomico(DADOS / "resumo.json", resumo)
        json_atomico(DADOS / "sem_candidato.json", {
            "treino": sem_treino, "validacao": sem_val,
        })
        print(json.dumps(resumo, indent=2), flush=True)

        assert not sem_treino, (
            f"{len(sem_treino)} imagens de treino sem candidato. "
            "Preparacao registrada; treino NAO iniciado. "
            "Nenhuma imagem substituida."
        )

        rng = np.random.default_rng(1234)
        ciclo = []
        for anterior in ref["treino"][:5000]:
            r = registros[anterior["indice"]]
            ciclo.append(dict(
                r, regiao=int(rng.choice(r["candidatos"]))
            ))

        treino = [dict(r) for r in ciclo] + [dict(r) for r in ciclo]
        assert [r["indice"] for r in treino] == [
            r["indice"] for r in ref["treino"]
        ]
        assert treino[:5000] == treino[5000:]

        # Pseudo-validacao apenas registrada; nao usada pelo loop de treino.
        # As 145 queries da avaliacao com mascaras reais nao sao alteradas.
        rng_val = np.random.default_rng(1234)
        validacao = [
            dict(
                registros[i],
                regiao=int(rng_val.choice(registros[i]["candidatos"])),
            )
            for i in sorted(val_ids) if registros[i]["candidatos"]
        ]

        plano = {"treino": treino, "validacao": validacao}
        plano_path = DADOS / "planos.json"
        if plano_path.exists():
            assert ler(plano_path) == plano
        else:
            json_atomico(plano_path, plano)

        json_atomico(DADOS / "divisao.json", [
            {
                "indice": i,
                "imagem": registros[i]["imagem"],
                "split": registros[i]["split"],
            }
            for i in sorted(registros)
        ])
        json_atomico(DADOS / "preparacao_concluida.json", {
            "episodios_treino": len(treino),
            "planos_sha256": sha(plano_path),
            "mesma_sequencia_de_imagens": True,
            "superpixels_recalculados_em512": True,
        })

        print(
            "PREPARACAO CONCLUIDA: novas regioes512; "
            "mesmas10000 imagens na mesma ordem. Iniciando treino.",
            flush=True,
        )
        return plano
'''

# Substitui somente a função preparar_plano, usando a estrutura do Python.
arvore = ast.parse(codigo)
funcoes = [
    n for n in arvore.body
    if isinstance(n, ast.FunctionDef) and n.name == "preparar_plano"
]
assert len(funcoes) == 1
funcao = funcoes[0]
linhas = codigo.splitlines(keepends=True)
linhas[funcao.lineno - 1:funcao.end_lineno] = [
    textwrap.dedent(nova_preparacao).strip() + "\n"
]
codigo = "".join(linhas)
compile(codigo, str(SCRIPT_SP512), "exec")

# Não sobrescreve scripts ou execuções anteriores.
with SCRIPT_SP512.open("x", encoding="utf-8") as f:
    f.write(codigo)

RUN_SP512.mkdir(parents=True, exist_ok=False)
manifesto = {
    "script": str(SCRIPT_SP512),
    "script_sha256": digest(SCRIPT_SP512),
    "script_base": str(ORIGINAL),
    "script_base_sha256": digest(ORIGINAL),
    "referencia": str(REFERENCIA),
    "dados": str(DADOS_SP512),
    "treino": str(RUN_SP512),
    "superpixels": "512, scale100, sigma1, min_size400",
    "treino_config": "672, ultima, encoder completo, LR1e-5, WD0.01, 10k",
    "inicializacao": "pesos pre-treinados; sem retomar EFT anterior",
}
(RUN_SP512 / "lancamento.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

with LOG_SP512.open("x", encoding="utf-8") as log:
    processoSP512 = subprocess.Popen(
        [
            sys.executable, "-u", str(SCRIPT_SP512),
            "--run", str(RUN_SP512),
        ],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID:", processoSP512.pid)
print("DADOS:", DADOS_SP512)
print("TREINO:", RUN_SP512)
print("LOG:", LOG_SP512)
print("Preparação iniciada; treino começa após as verificações.")

In [ ]:
estado = processoSP512.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)
linhas = LOG_SP512.read_text(
    encoding="utf-8", errors="replace"
).splitlines()
print("\n".join(linhas[-30:]))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

# Variável criada na célula que iniciou o treino.
assert "RUN_SP512" in globals(), (
    "Defina RUN_SP512 = Path('caminho da execução concluída')."
)
RUN_SP512 = Path(RUN_SP512)
assert (RUN_SP512 / "concluido.json").is_file()

concluido = json.loads(
    (RUN_SP512 / "concluido.json").read_text(encoding="utf-8")
)
assert concluido["passos"] == 10000

BASE_AVALIADOR = CODIGO / "avaliar_penultima_treino672_10k_wd001.py"
assert BASE_AVALIADOR.is_file(), BASE_AVALIADOR

PASSOS_SP512 = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]
for passo in PASSOS_SP512:
    p = RUN_SP512 / f"checkpoint_{passo:04d}.pt"
    assert p.is_file(), f"Checkpoint ausente: {p}"

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AVALIACAO_SP512 = ROOT / "reproducao" / f"avaliacao_superpixels512_S00_{stamp}"
AVALIACAO_SP512.mkdir(parents=True, exist_ok=False)

SCRIPT_AVAL_SP512 = CODIGO / f"avaliar_superpixels512_{stamp}.py"
LOG_AVAL_SP512 = AVALIACAO_SP512 / "console.log"
RESULTADOS_SP512 = AVALIACAO_SP512 / "resultados"

codigo = BASE_AVALIADOR.read_text(encoding="utf-8")
arvore = ast.parse(codigo)

# Substitui apenas as duas atribuições no nível principal.
alteracoes = []
for nome, expressao in {
    "TREINO": f"Path({str(RUN_SP512)!r})",
    "PASSOS": repr(PASSOS_SP512),
}.items():
    encontrados = [
        n for n in arvore.body
        if isinstance(n, ast.Assign)
        and any(
            isinstance(t, ast.Name) and t.id == nome
            for t in n.targets
        )
    ]
    assert len(encontrados) == 1, (
        f"Atribuição {nome} não encontrada de forma única."
    )
    n = encontrados[0]
    alteracoes.append((n.lineno, n.end_lineno, f"{nome} = {expressao}\n"))

linhas = codigo.splitlines(keepends=True)
for inicio, fim, substituto in sorted(alteracoes, reverse=True):
    linhas[inicio - 1:fim] = [substituto]
codigo = "".join(linhas)

compile(codigo, str(SCRIPT_AVAL_SP512), "exec")
with SCRIPT_AVAL_SP512.open("x", encoding="utf-8") as f:
    f.write(codigo)

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

manifesto = {
    "treino": str(RUN_SP512),
    "treino_config_sha256": sha(RUN_SP512 / "config.json"),
    "avaliador_base": str(BASE_AVALIADOR),
    "avaliador_base_sha256": sha(BASE_AVALIADOR),
    "avaliador_novo_sha256": sha(SCRIPT_AVAL_SP512),
    "alteracoes_no_avaliador": ["TREINO", "PASSOS"],
    "passos": PASSOS_SP512,
    "objetivo": "comparar pseudolabels256 vs pseudolabels512",
    "limite": "validacao exploratoria S00; nao e teste",
}
(AVALIACAO_SP512 / "lancamento.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

with LOG_AVAL_SP512.open("x", encoding="utf-8") as log:
    processoAVAL_SP512 = subprocess.Popen(
        [
            sys.executable, "-u", str(SCRIPT_AVAL_SP512),
            "--out", str(RESULTADOS_SP512),
        ],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID:", processoAVAL_SP512.pid)
print("AVALIAÇÃO:", AVALIACAO_SP512)
print("LOG:", LOG_AVAL_SP512)
print("Checkpoints:", PASSOS_SP512)

In [ ]:
assert processoAVAL_SP512.poll() == 0, (
    "A avaliação ainda não terminou com sucesso."
)

# Referência: treino672, WD0.01, regiões geradas em256,
# avaliação pela penúltima + SAM[0], no S00.
referencia_sam = {
    0: 0.509673, 50: 0.564939, 100: 0.604391,
    200: 0.618045, 500: 0.680723, 1000: 0.666825,
    1500: 0.660086, 2000: 0.611333, 3000: 0.553504,
    4000: 0.549343, 5000: 0.530366, 6000: 0.499942,
    7000: 0.548999, 8000: 0.568081, 9000: 0.586565,
    10000: 0.536894,
}

resultados = []
print("Passo | ALP SP512 | SAM SP256 | SAM SP512 | Δ SAM (p.p.)")

for passo in PASSOS_SP512:
    p = RESULTADOS_SP512 / f"passo_{passo:04d}" / "resumo.json"
    assert p.is_file(), p
    r = json.loads(p.read_text(encoding="utf-8"))
    assert r["n"] == 145
    resultados.append(r)

    antigo = referencia_sam[passo]
    print(
        f"{passo:5d} | {r['ALP_Dice']:.6f}  | "
        f"{antigo:.6f}  | {r['SAM_Dice']:.6f}  | "
        f"{100 * (r['SAM_Dice'] - antigo):+.2f}"
    )

melhor = max(
    resultados,
    key=lambda r: (r["SAM_Dice"], -r["passo"])
)
print("\nMelhor checkpoint:", melhor["passo"])
print(f"Dice ALP+SAM: {melhor['SAM_Dice']:.2%}")
print(
    "Diferença para o melhor anterior: "
    f"{100 * (melhor['SAM_Dice'] - 0.6807229550208969):+.2f} p.p."
)
print("Referência da tabela arredondada a seis casas decimais.")

In [ ]:
estado = processoAVAL_SP512.poll()

if estado is None:
    print("Estado: avaliação ainda executando. Aguarde para comparar.")
elif estado == 0:
    print("Estado: concluída com sucesso. Pode executar a comparação.")
else:
    print(f"Estado: avaliação encerrada com erro, código {estado}.")

print("\n=== ÚLTIMAS LINHAS DO LOG ===")
if LOG_AVAL_SP512.is_file():
    linhas = LOG_AVAL_SP512.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    print("\n".join(linhas[-60:]))
else:
    print("Log não encontrado:", LOG_AVAL_SP512)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
INSPECAO_ORIGEM = ROOT / (
    "reproducao/inspecao_superpixels512_20261020T234211912923Z"
)
assert (INSPECAO_ORIGEM / "relatorio.json").is_file()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_SAM_REGIOES = ROOT / "reproducao" / f"inspecao_regioesSAM_{stamp}"
RUN_SAM_REGIOES.mkdir(parents=True, exist_ok=False)

SCRIPT_SAM_REGIOES = RUN_SAM_REGIOES / "inspecionar_sam.py"
LOG_SAM_REGIOES = RUN_SAM_REGIOES / "console.log"

codigo = r'''
from pathlib import Path
import hashlib
import inspect
import json
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from scipy import ndimage as ndi

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
sys.path.insert(0, str(ROOT))

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

OUT = Path(__file__).resolve().parent
ORIGEM = ROOT / (
    "reproducao/inspecao_superpixels512_20261020T234211912923Z"
)
CHECKPOINT = ROOT / "pretrained_model/sam_vit_h.pth"

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def salvar_json(p, obj):
    p = Path(p)
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    tmp.replace(p)

def area_util(rgb):
    cc, n = ndi.label(rgb.max(axis=-1) > 10)
    if n == 0:
        return np.zeros(rgb.shape[:2], dtype=bool)
    tamanhos = np.bincount(cc.ravel())
    tamanhos[0] = 0
    maior = ndi.binary_fill_holes(cc == tamanhos.argmax())
    return ndi.binary_erosion(
        maior, structure=np.ones((3, 3), dtype=bool),
        iterations=3, border_value=0,
    )

def grade(mask):
    t = torch.from_numpy(np.ascontiguousarray(mask)).float()[None, None]
    g = F.interpolate(t, size=(32, 32), mode="nearest")
    fg = F.avg_pool2d(g, 4, 4)
    bg = F.avg_pool2d(1 - g, 4, 4)
    return {
        "pixels_fg": int(g.sum()),
        "celulas_fg": int((fg > 0.95).sum()),
        "celulas_bg": int((bg > 0.95).sum()),
    }

def painel(rgb, mask, titulo, subtitulo):
    # Máscara mostrada individualmente, sem unir candidatas.
    visual = rgb.astype(np.float32)
    visual[~mask] *= 0.25
    visual[mask] = (
        0.65 * visual[mask] + 0.35 * np.array([0, 255, 80])
    )
    borda = mask & ~ndi.binary_erosion(mask)
    visual[borda] = [255, 255, 0]

    quadro = Image.new("RGB", (384, 430), "white")
    im = Image.fromarray(
        np.clip(visual, 0, 255).astype(np.uint8)
    ).resize((384, 384))
    quadro.paste(im, (0, 46))
    desenho = ImageDraw.Draw(quadro)
    desenho.text((6, 5), titulo, fill="black")
    desenho.text((6, 23), subtitulo, fill="black")
    return quadro

assert torch.cuda.is_available()
assert CHECKPOINT.is_file()
torch.set_num_threads(1)
torch.manual_seed(1234)
np.random.seed(1234)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False

amostras = json.loads(
    (ORIGEM / "relatorio.json").read_text(encoding="utf-8")
)["amostras"]
assert len(amostras) == 6

parametros = {
    "points_per_side": 32,
    "points_per_batch": 16,
    "pred_iou_thresh": 0.88,
    "stability_score_thresh": 0.95,
    "stability_score_offset": 1.0,
    "box_nms_thresh": 0.7,
    "crop_n_layers": 0,
    "min_mask_region_area": 0,
    "output_mode": "binary_mask",
}

config = {
    "objetivo": "inspecao de mascaras SAM como alvos episodicos",
    "modelo": "vit_h",
    "checkpoint": str(CHECKPOINT),
    "checkpoint_sha256": sha(CHECKPOINT),
    "gerador": parametros,
    "entrada": "mesmo RGB512 da inspecao Felzenszwalb",
    "sobreposicoes": "preservadas; candidatas independentes",
    "filtros_eft": {
        "area_interna_minima": 100,
        "fracao_interna_minima": 0.95,
        "erosao": 3,
        "grade": [32, 32],
        "kernel": 4,
        "criterio": "FG sobrevive e existe BG local>0.95 em support/query",
        "duplicatas": "remove apenas mascaras recortadas exatamente iguais",
    },
    "mascaras_manuais_utilizadas": False,
    "DINO_utilizado": False,
    "origem_sha256": sha(ORIGEM / "relatorio.json"),
    "script_sha256": sha(__file__),
    "gerador_fonte": inspect.getfile(SamAutomaticMaskGenerator),
    "gerador_fonte_sha256": sha(inspect.getfile(SamAutomaticMaskGenerator)),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
}
salvar_json(OUT / "config.json", config)

print("Carregando SAM-H...", flush=True)
sam = sam_model_registry["vit_h"](checkpoint=str(CHECKPOINT))
sam = sam.to("cuda").eval().requires_grad_(False)
gerador = SamAutomaticMaskGenerator(sam, **parametros)

resumos = []

for pos, item in enumerate(amostras, 1):
    pasta = OUT / f"imagem_{pos:02d}_indice_{item['indice']}"
    pasta.mkdir(exist_ok=False)

    cache = Path(item["cache_origem"])
    assert sha(cache) == item["cache_sha256"]
    with np.load(cache, allow_pickle=False) as d:
        rgb = d["rgb"].copy()
    assert rgb.shape == (512, 512, 3)
    assert rgb.dtype == np.uint8

    util = area_util(rgb)
    Image.fromarray(rgb).save(pasta / "rgb.png")

    print(
        f"\nImagem {pos}/6 | indice={item['indice']} | gerando...",
        flush=True,
    )
    inicio = time.perf_counter()
    with torch.inference_mode():
        anotacoes = gerador.generate(rgb)
    torch.cuda.synchronize()
    segundos = time.perf_counter() - inicio

    # Ordenação determinística para apresentação e remoção de duplicatas.
    anotacoes.sort(key=lambda a: (
        -float(a["predicted_iou"]),
        -float(a["stability_score"]),
        -int(a["area"]),
    ))

    registros = []
    brutas = []
    aprovadas = []
    vistas = set()
    paineis = []

    for j, a in enumerate(anotacoes):
        original = np.asarray(a["segmentation"], dtype=bool)
        assert original.shape == (512, 512)
        mask = original & util
        area_original = int(original.sum())
        area = int(mask.sum())
        fracao = area / max(area_original, 1)
        gs, gq = grade(mask), grade(np.fliplr(mask))

        motivos = []
        if area < 100:
            motivos.append("area_interna<100")
        if fracao < 0.95:
            motivos.append("fracao_interna<0.95")
        if not all(
            g["pixels_fg"] > 0 and g["celulas_bg"] > 0
            for g in (gs, gq)
        ):
            motivos.append("grade_EFT")

        assinatura = hashlib.sha256(
            np.packbits(mask).tobytes()
        ).hexdigest()
        if not motivos:
            if assinatura in vistas:
                motivos.append("duplicata_exata")
            else:
                vistas.add(assinatura)

        aceita = not motivos
        candidato = len(aprovadas) if aceita else None
        if aceita:
            aprovadas.append(mask)

        registro = {
            "id_gerador_ordenado": j,
            "candidato": candidato,
            "aprovada": aceita,
            "motivos_rejeicao": motivos,
            "area_original": area_original,
            "area_interna": area,
            "fracao_interna": fracao,
            "area_interna_pct": 100 * area / mask.size,
            "predicted_iou": float(a["predicted_iou"]),
            "stability_score": float(a["stability_score"]),
            "grade_support": gs,
            "grade_query": gq,
            "bbox_original": [float(v) for v in a["bbox"]],
            "point_coords": np.asarray(a["point_coords"]).tolist(),
        }
        registros.append(registro)
        brutas.append(original)

        # Guarda cada máscara original e a versão recortada pela área útil.
        Image.fromarray(original.astype(np.uint8) * 255).save(
            pasta / f"mascara_{j:03d}_original.png"
        )
        Image.fromarray(mask.astype(np.uint8) * 255).save(
            pasta / f"mascara_{j:03d}_util.png"
        )

        status = "OK" if aceita else "REJEITADA"
        paineis.append(painel(
            rgb, mask,
            f"ID {j} | {status} | area={100*area/mask.size:.2f}%",
            f"IoUest={a['predicted_iou']:.3f} "
            f"estab={a['stability_score']:.3f}",
        ))

    # Inclui todas as máscaras retornadas pelo AMG, em páginas de seis.
    for inicio_pagina in range(0, len(paineis), 6):
        pagina = Image.new("RGB", (384 * 3, 430 * 2), "white")
        for k, im in enumerate(paineis[inicio_pagina:inicio_pagina + 6]):
            pagina.paste(im, ((k % 3) * 384, (k // 3) * 430))
        pagina.save(
            pasta / f"pagina_{inicio_pagina // 6 + 1:02d}.png"
        )

    def empilhar(lista):
        return (
            np.stack(lista)
            if lista else np.zeros((0, 512, 512), dtype=bool)
        )

    np.savez_compressed(
        pasta / "mascaras.npz",
        rgb=rgb,
        util=util,
        mascaras_amg=empilhar(brutas),
        candidatas=empilhar(aprovadas),
    )
    salvar_json(pasta / "mascaras.json", registros)

    resumo = {
        "indice": item["indice"],
        "mascaras_retornadas_AMG": len(anotacoes),
        "candidatas_EFT": len(aprovadas),
        "com_fg_local_support_e_query": sum(
            r["aprovada"]
            and r["grade_support"]["celulas_fg"] > 0
            and r["grade_query"]["celulas_fg"] > 0
            for r in registros
        ),
        "segundos_geracao": segundos,
        "pasta": str(pasta),
    }
    resumos.append(resumo)
    print(json.dumps(resumo, indent=2), flush=True)
    salvar_json(OUT / "resumo_parcial.json", resumos)

salvar_json(OUT / "concluido.json", {
    "imagens": 6,
    "resumos": resumos,
    "treinamento_iniciado": False,
})
print("\nINSPECAO SAM CONCLUIDA. Nenhum treinamento iniciado.", flush=True)
'''

compile(codigo, str(SCRIPT_SAM_REGIOES), "exec")
SCRIPT_SAM_REGIOES.write_text(codigo, encoding="utf-8")

with LOG_SAM_REGIOES.open("x", encoding="utf-8") as log:
    processoSAM_REGIOES = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT_SAM_REGIOES)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID:", processoSAM_REGIOES.pid)
print("EXECUÇÃO:", RUN_SAM_REGIOES)
print("LOG:", LOG_SAM_REGIOES)

In [ ]:
estado = processoSAM_REGIOES.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)
print("\n".join(
    LOG_SAM_REGIOES.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-35:]
))

In [ ]:
import json
from PIL import Image
from IPython.display import display

assert processoSAM_REGIOES.poll() == 0, (
    "Confira o estado e o log antes de visualizar."
)

resultado = json.loads(
    (RUN_SAM_REGIOES / "concluido.json").read_text(encoding="utf-8")
)

for r in resultado["resumos"]:
    print(
        f"\nÍndice {r['indice']} | "
        f"AMG: {r['mascaras_retornadas_AMG']} | "
        f"Aprovadas EFT: {r['candidatas_EFT']} | "
        f"Com FG local: {r['com_fg_local_support_e_query']}"
    )
    pasta = Path(r["pasta"])
    for figura in sorted(pasta.glob("pagina_*.png")):
        with Image.open(figura) as im:
            display(im.copy())

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import torch
import torch.nn.functional as F

assert "RUN_SAM_REGIOES" in globals(), (
    "Defina RUN_SAM_REGIOES com a pasta da inspeção SAM."
)
ORIGEM = Path(RUN_SAM_REGIOES)
assert (ORIGEM / "concluido.json").is_file()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AUDITORIA_SAM = ORIGEM / f"auditoria_filtros_{stamp}"
AUDITORIA_SAM.mkdir(exist_ok=False)

MIN_FRACAO = 0.005
MAX_FRACAO = 0.80

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def contar_bg_interno(mask, util):
    # Fundo válido: está dentro da área útil e fora da máscara.
    bg_interno = util & ~mask
    t = torch.from_numpy(
        np.ascontiguousarray(bg_interno)
    ).float()[None, None]
    g = F.interpolate(t, size=(32, 32), mode="nearest")
    celulas = F.avg_pool2d(g, kernel_size=4, stride=4)
    return int((celulas > 0.95).sum())

concluido = json.loads(
    (ORIGEM / "concluido.json").read_text(encoding="utf-8")
)

linhas = []
resumos = []
fontes = {}

threads_antes = torch.get_num_threads()
try:
    torch.set_num_threads(1)

    for imagem in concluido["resumos"]:
        pasta = Path(imagem["pasta"])
        arquivo_npz = pasta / "mascaras.npz"
        arquivo_json = pasta / "mascaras.json"

        fontes[str(arquivo_npz)] = sha(arquivo_npz)
        fontes[str(arquivo_json)] = sha(arquivo_json)

        registros = json.loads(
            arquivo_json.read_text(encoding="utf-8")
        )
        with np.load(arquivo_npz, allow_pickle=False) as d:
            util = d["util"].copy().astype(bool)
            mascaras = d["mascaras_amg"].copy().astype(bool)

        assert len(registros) == len(mascaras)
        assert util.any()
        area_util = int(util.sum())
        itens_imagem = []

        for j, r in enumerate(registros):
            assert r["id_gerador_ordenado"] == j
            mask = mascaras[j] & util
            assert int(mask.sum()) == r["area_interna"]

            fracao = float(mask.sum() / area_util)
            bg_s = contar_bg_interno(mask, util)
            bg_q = contar_bg_interno(
                np.fliplr(mask), np.fliplr(util)
            )

            atual = bool(r["aprovada"])
            filtro_area = MIN_FRACAO <= fracao <= MAX_FRACAO
            filtro_bg = bg_s > 0 and bg_q > 0

            linha = {
                "indice": imagem["indice"],
                "id_mascara": j,
                "atual": atual,
                "area_util": atual and filtro_area,
                "area_util_bg": atual and filtro_area and filtro_bg,
                "cobertura_area_util_pct": 100 * fracao,
                "bg_interno_support": bg_s,
                "bg_interno_query": bg_q,
                "predicted_iou": r["predicted_iou"],
                "stability_score": r["stability_score"],
                "motivos_rejeicao_original": r["motivos_rejeicao"],
                "mascara": str(pasta / f"mascara_{j:03d}_util.png"),
            }
            itens_imagem.append(linha)
            linhas.append(linha)

        resumo = {
            "indice": imagem["indice"],
            "AMG": len(registros),
            "atual": sum(r["atual"] for r in itens_imagem),
            "area_util": sum(r["area_util"] for r in itens_imagem),
            "area_util_bg": sum(r["area_util_bg"] for r in itens_imagem),
            "ids_area_util_bg": [
                r["id_mascara"] for r in itens_imagem
                if r["area_util_bg"]
            ],
        }
        resumos.append(resumo)

finally:
    torch.set_num_threads(threads_antes)

print("Índice | AMG | Atual | Área útil | Área + BG interno")
for r in resumos:
    print(
        f"{r['indice']:6d} | {r['AMG']:3d} | "
        f"{r['atual']:5d} | {r['area_util']:9d} | "
        f"{r['area_util_bg']:17d}"
    )
    print("       IDs mantidos no último filtro:", r["ids_area_util_bg"])

print("\n=== CANDIDATAS APROVADAS PELO FILTRO ORIGINAL ===")
for r in linhas:
    if r["atual"]:
        print(
            f"Índice {r['indice']} | ID {r['id_mascara']:2d} | "
            f"área útil={r['cobertura_area_util_pct']:6.2f}% | "
            f"BG interno S/Q={r['bg_interno_support']}/"
            f"{r['bg_interno_query']} | "
            f"novo filtro={'MANTÉM' if r['area_util_bg'] else 'EXCLUI'}"
        )

relatorio = {
    "origem": str(ORIGEM),
    "fontes_sha256": fontes,
    "limiares_exploratorios": {
        "fracao_minima_area_util": MIN_FRACAO,
        "fracao_maxima_area_util": MAX_FRACAO,
        "bg_interno": ">=1 celula com ocupacao>0.95 em support e query",
        "grade": [32, 32],
        "kernel": 4,
    },
    "resumos": resumos,
    "mascaras": linhas,
    "treinamento_iniciado": False,
}
(AUDITORIA_SAM / "relatorio.json").write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nAuditoria salva em:", AUDITORIA_SAM)
print("Máscaras originais preservadas. Nenhum treino iniciado.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

INSPECAO_SAM = ROOT / (
    "reproducao/inspecao_regioesSAM_20261021T105545994828Z"
)
FONTE_INSPECAO = INSPECAO_SAM / "inspecionar_sam.py"
assert (INSPECAO_SAM / "concluido.json").is_file()
assert FONTE_INSPECAO.is_file()

cfg_inspecao = json.loads(
    (INSPECAO_SAM / "config.json").read_text(encoding="utf-8")
)
fonte = FONTE_INSPECAO.read_text(encoding="utf-8")
assert hashlib.sha256(FONTE_INSPECAO.read_bytes()).hexdigest() == (
    cfg_inspecao["script_sha256"]
)

# Reutiliza exatamente as funções da inspeção, sem executar seu main.
arvore = ast.parse(fonte)
funcoes = {}
for nome in ("sha", "salvar_json", "area_util", "grade"):
    nos = [
        n for n in arvore.body
        if isinstance(n, ast.FunctionDef) and n.name == nome
    ]
    assert len(nos) == 1, nome
    funcoes[nome] = ast.get_source_segment(fonte, nos[0])

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DADOS_SAM_EFT = ROOT / "reproducao" / f"dados_SAM_EFT512_{stamp}"
DADOS_SAM_EFT.mkdir(parents=True, exist_ok=False)
SCRIPT_GERAR_SAM = DADOS_SAM_EFT / "gerar_mascaras.py"

cabecalho = r'''
from pathlib import Path
import fcntl
import hashlib
import inspect
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
sys.path.insert(0, str(ROOT))

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

OUT = Path(__file__).resolve().parent
INSPECAO = ROOT / "reproducao/inspecao_regioesSAM_20261021T105545994828Z"
REFERENCIA = ROOT / "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
'''

corpo = r'''
def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def bg_interno(mask, util):
    t = torch.from_numpy(
        np.ascontiguousarray(util & ~mask)
    ).float()[None, None]
    g = F.interpolate(t, size=(32, 32), mode="nearest")
    return int((F.avg_pool2d(g, 4, 4) > 0.95).sum())

def empilhar(lista):
    return (
        np.stack(lista).astype(bool)
        if lista else np.zeros((0, 512, 512), dtype=bool)
    )

def main():
    assert torch.cuda.is_available()
    torch.set_num_threads(1)
    torch.manual_seed(1234)
    np.random.seed(1234)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False

    cfg_inspecao = ler(INSPECAO / "config.json")
    checkpoint = Path(cfg_inspecao["checkpoint"])
    assert sha(checkpoint) == cfg_inspecao["checkpoint_sha256"]

    fonte_amg = Path(inspect.getfile(SamAutomaticMaskGenerator))
    assert sha(fonte_amg) == cfg_inspecao["gerador_fonte_sha256"], (
        "Implementacao do gerador SAM mudou desde a inspecao."
    )

    plano_path = REFERENCIA / "planos.json"
    assert sha(plano_path) == (
        "880b04375c4a25fa0efd326afc666011c47fe460c5180ad91653fe0cb4be024b"
    )
    plano = ler(plano_path)
    cfg_ref = ler(REFERENCIA / "config.json")
    dados_rgb = Path(cfg_ref["dados"])

    por_indice = {}
    for r in plano["treino"]:
        assert r["split"] == "treino"
        if r["indice"] in por_indice:
            anterior = por_indice[r["indice"]]
            for chave in ("imagem_sha256", "cache_sha256", "cache"):
                assert anterior[chave] == r[chave]
        else:
            por_indice[r["indice"]] = r

    val_ids = {r["indice"] for r in plano["validacao"]}
    assert len(por_indice) == 1304
    assert len(val_ids) == 145
    assert not set(por_indice) & val_ids

    config = {
        "experimento": "candidatas SAM para EFT; somente geracao",
        "modelo": "vit_h",
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha(checkpoint),
        "parametros_AMG": cfg_inspecao["gerador"],
        "entrada_rgb": "cache512 conferido por SHA256",
        "inspecao_origem": str(INSPECAO),
        "inspecao_config_sha256": sha(INSPECAO / "config.json"),
        "referencia": str(REFERENCIA),
        "plano_referencia_sha256": sha(plano_path),
        "script_sha256": sha(__file__),
        "fonte_AMG_sha256": sha(fonte_amg),
        "n_imagens_treino": 1304,
        "validacao_processada": False,
        "mascaras_manuais_utilizadas": False,
        "DINO_utilizado": False,
        "filtros": {
            "area_interna_minima_pixels": 100,
            "fracao_interna_minima": 0.95,
            "cobertura_area_util_minima": 0.005,
            "cobertura_area_util_maxima": 0.80,
            "grade": [32, 32],
            "kernel": 4,
            "background_interno": (
                ">=1 celula com ocupacao>0.95 em support e query"
            ),
            "duplicatas": "somente mascaras recortadas exatamente iguais",
            "sobreposicoes": "preservadas",
        },
        "torch": str(torch.__version__),
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    }

    config_path = OUT / "config.json"
    if config_path.exists():
        assert ler(config_path) == config, (
            "Configuracao, codigo ou ambiente mudou; retomada interrompida."
        )
    else:
        salvar_json(config_path, config)

    (OUT / "cache").mkdir(exist_ok=True)

    # Modelo carregado apenas se houver imagens pendentes.
    gerador = None
    registros = []
    processadas_agora = 0
    inicio = time.perf_counter()

    for pos, indice in enumerate(sorted(por_indice), 1):
        origem = por_indice[indice]
        destino = OUT / "cache" / f"{indice:04d}.npz"
        meta = destino.with_suffix(".json")

        if destino.exists() and meta.exists():
            registro = ler(meta)
            assert registro["indice"] == indice
            assert registro["imagem_sha256"] == origem["imagem_sha256"]
            assert registro["cache_rgb_sha256"] == origem["cache_sha256"]
            assert sha(destino) == registro["cache_sha256"]
            registros.append(registro)
            if pos % 50 == 0:
                print(f"Verificados/reutilizados: {pos}/1304", flush=True)
            continue

        cache_rgb = dados_rgb / origem["cache"]
        assert sha(cache_rgb) == origem["cache_sha256"]
        with np.load(cache_rgb, allow_pickle=False) as d:
            rgb = d["rgb"].copy()

        assert rgb.shape == (512, 512, 3)
        assert rgb.dtype == np.uint8
        util = area_util(rgb)
        assert util.any(), f"Area util vazia: {indice}"

        if gerador is None:
            print("Carregando SAM-H...", flush=True)
            sam = sam_model_registry["vit_h"](checkpoint=str(checkpoint))
            sam = sam.to("cuda").eval().requires_grad_(False)
            gerador = SamAutomaticMaskGenerator(
                sam, **config["parametros_AMG"]
            )

        print(f"Gerando {pos}/1304 | indice={indice}", flush=True)
        t0 = time.perf_counter()
        with torch.inference_mode():
            anotacoes = gerador.generate(rgb)
        torch.cuda.synchronize()

        anotacoes.sort(key=lambda a: (
            -float(a["predicted_iou"]),
            -float(a["stability_score"]),
            -int(a["area"]),
        ))

        brutas, aprovadas, auditoria = [], [], []
        vistas = set()

        for j, a in enumerate(anotacoes):
            original = np.asarray(a["segmentation"], dtype=bool)
            assert original.shape == (512, 512)
            mask = original & util
            area = int(mask.sum())
            fracao_interna = area / max(int(original.sum()), 1)
            cobertura = area / int(util.sum())
            gs, gq = grade(mask), grade(np.fliplr(mask))

            motivos = []
            if area < 100:
                motivos.append("area_interna<100")
            if fracao_interna < 0.95:
                motivos.append("fracao_interna<0.95")
            if not all(
                g["pixels_fg"] > 0 and g["celulas_bg"] > 0
                for g in (gs, gq)
            ):
                motivos.append("grade_EFT")

            # Mesma deduplicacao da inspecao original.
            assinatura = hashlib.sha256(
                np.packbits(mask).tobytes()
            ).hexdigest()
            if not motivos:
                if assinatura in vistas:
                    motivos.append("duplicata_exata")
                else:
                    vistas.add(assinatura)

            if cobertura < 0.005:
                motivos.append("cobertura_util<0.5%")
            if cobertura > 0.80:
                motivos.append("cobertura_util>80%")

            bg_s = bg_interno(mask, util)
            bg_q = bg_interno(np.fliplr(mask), np.fliplr(util))
            if bg_s == 0 or bg_q == 0:
                motivos.append("sem_BG_interno")

            candidato = None
            if not motivos:
                candidato = len(aprovadas)
                aprovadas.append(mask)

            brutas.append(original)
            auditoria.append({
                "id_AMG": j,
                "candidato": candidato,
                "aprovada": not motivos,
                "motivos": motivos,
                "area_original": int(original.sum()),
                "area_interna": area,
                "fracao_interna": fracao_interna,
                "cobertura_util": cobertura,
                "grade_support": gs,
                "grade_query": gq,
                "bg_interno_support": bg_s,
                "bg_interno_query": bg_q,
                "predicted_iou": float(a["predicted_iou"]),
                "stability_score": float(a["stability_score"]),
            })

        # Mantém máscaras AMG para permitir outras auditorias sem regerar.
        tmp = destino.with_suffix(".tmp")
        with tmp.open("wb") as f:
            np.savez_compressed(
                f,
                util=util,
                mascaras_amg=empilhar(brutas),
                candidatas=empilhar(aprovadas),
            )
        os.replace(tmp, destino)

        registro = {
            "indice": indice,
            "nome": origem["nome"],
            "imagem": origem["imagem"],
            "imagem_sha256": origem["imagem_sha256"],
            "split": "treino",
            "cache_rgb": str(cache_rgb),
            "cache_rgb_sha256": origem["cache_sha256"],
            "cache": str(destino.relative_to(OUT)),
            "cache_sha256": sha(destino),
            "n_AMG": len(brutas),
            "n_candidatas": len(aprovadas),
            "segundos": time.perf_counter() - t0,
            "mascaras": auditoria,
        }
        salvar_json(meta, registro)
        registros.append(registro)
        processadas_agora += 1

        media = (time.perf_counter() - inicio) / processadas_agora
        restante_h = media * (1304 - pos) / 3600
        print(
            f"Salvo {pos}/1304 | AMG={len(brutas)} | "
            f"candidatas={len(aprovadas)} | "
            f"restante aproximado={restante_h:.2f}h",
            flush=True,
        )

        salvar_json(OUT / "progresso.json", {
            "imagens_conferidas": pos,
            "total": 1304,
            "ultimo_indice": indice,
        })

    assert len(registros) == 1304
    elegiveis = [r for r in registros if r["n_candidatas"] > 0]
    sem = [r for r in registros if r["n_candidatas"] == 0]
    contagens = np.array([r["n_candidatas"] for r in registros])
    areas = [
        100 * a["cobertura_util"]
        for r in registros for a in r["mascaras"] if a["aprovada"]
    ]

    salvar_json(OUT / "indice.json", [
        {k: v for k, v in r.items() if k != "mascaras"}
        for r in registros
    ])
    salvar_json(OUT / "sem_candidato.json", [
        {k: v for k, v in r.items() if k != "mascaras"}
        for r in sem
    ])

    resumo = {
        "imagens_processadas": 1304,
        "imagens_elegiveis": len(elegiveis),
        "imagens_sem_candidata": len(sem),
        "cobertura_imagens_pct": 100 * len(elegiveis) / 1304,
        "total_candidatas": int(contagens.sum()),
        "candidatas_por_imagem": {
            "minimo": int(contagens.min()),
            "mediana": float(np.median(contagens)),
            "media": float(contagens.mean()),
            "maximo": int(contagens.max()),
        },
        "area_candidatas_pct_da_area_util": (
            {
                "p10": float(np.percentile(areas, 10)),
                "mediana": float(np.median(areas)),
                "p90": float(np.percentile(areas, 90)),
            } if areas else None
        ),
        "indices_sem_candidata": [r["indice"] for r in sem],
        "validacao_processada": False,
        "episodios_montados": False,
        "treinamento_iniciado": False,
    }
    salvar_json(OUT / "concluido.json", resumo)
    print("\nGERACAO CONCLUIDA:", flush=True)
    print(json.dumps(resumo, indent=2, ensure_ascii=False), flush=True)

if __name__ == "__main__":
    with (OUT / "execucao.lock").open("a") as trava:
        fcntl.flock(trava, fcntl.LOCK_EX | fcntl.LOCK_NB)
        main()
'''

codigo = (
    cabecalho
    + "\n\n"
    + "\n\n".join(funcoes.values())
    + "\n\n"
    + corpo
)
compile(codigo, str(SCRIPT_GERAR_SAM), "exec")
SCRIPT_GERAR_SAM.write_text(codigo, encoding="utf-8")

LOG_GERAR_SAM = DADOS_SAM_EFT / f"console_{stamp}.log"
with LOG_GERAR_SAM.open("x", encoding="utf-8") as log:
    processoGERAR_SAM = subprocess.Popen(
        [sys.executable, "-u", str(SCRIPT_GERAR_SAM)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID:", processoGERAR_SAM.pid)
print("DADOS — guarde este caminho:", DADOS_SAM_EFT)
print("LOG:", LOG_GERAR_SAM)
print("Somente geração de máscaras. Nenhum treino será iniciado.")

In [ ]:
estado = processoGERAR_SAM.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)
print("\n".join(
    LOG_GERAR_SAM.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-30:]
))